In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# CRPS & original MASE

### Daily

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

QUANTILE_LEVELS = np.arange(0.1, 1.0, 0.1)

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

DATA_DIR = Path(
    r"BForTFin_log_return/arima_split_datasets/output_daily"
)

RESULTS_DIR = Path(
    r"BForTFin_results/standardised/output_daily/timestral_standardized_train_test"
)

def samples_to_quantile_tensor(
    samples,
    quantile_levels=QUANTILE_LEVELS
):
    samples = np.asarray(
        samples,
        dtype=float
    )

    if samples.ndim != 3:
        raise ValueError(
            f"Expected (N, M, H), got {samples.shape}"
        )

    quantiles = np.quantile(
        samples,
        quantile_levels,
        axis=1
    )

    quantiles = np.moveaxis(
        quantiles,
        0,
        -1
    )

    return quantiles


def quantile_crps(
    y,
    quantile_forecasts,
    quantile_levels=QUANTILE_LEVELS
):
    y = np.asarray(y, dtype=float)
    qf = np.asarray(
        quantile_forecasts,
        dtype=float
    )

    levels = np.asarray(
        quantile_levels,
        dtype=float
    )

    error = y[..., None] - qf

    pinball = np.maximum(
        levels * error,
        (levels - 1.0) * error
    )

    return 2.0 * np.mean(
        pinball,
        axis=-1
    )

def calculate_mase_from_point_forecast(
    X,
    Y_true,
    Y_pred
):
    naive_forecast = (
        X[:, -1][:, None]
    )

    model_mae_by_horizon = np.mean(
        np.abs(
            Y_true - Y_pred
        ),
        axis=0
    )

    naive_mae_by_horizon = np.mean(
        np.abs(
            Y_true - naive_forecast
        ),
        axis=0
    )

    mase_by_horizon = np.full(
        Y_true.shape[1],
        np.nan,
        dtype=float
    )

    valid_horizons = (
        naive_mae_by_horizon > 0
    )

    mase_by_horizon[
        valid_horizons
    ] = (
        model_mae_by_horizon[
            valid_horizons
        ]
        /
        naive_mae_by_horizon[
            valid_horizons
        ]
    )

    mean_mase = np.nanmean(
        mase_by_horizon
    )

    return (
        mean_mase,
        mase_by_horizon,
        model_mae_by_horizon,
        naive_mae_by_horizon
    )

In [ ]:
summary_rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = DATA_DIR / folder_name
    model_dir = (
        RESULTS_DIR /
        folder_name
    )

    print("\n" + "=" * 80)
    print(asset_class)
    print("=" * 80)

    for original_path in sorted(
        input_dir.glob("test_*.csv")
    ):

        dataset_name = (
            original_path.stem
        )

        # Exclude Tether
        if (
            asset_class == "Crypto"
            and "tether" in dataset_name.lower()
        ):
            print(
                f"Skipping Tether: "
                f"{dataset_name}"
            )
            continue

        print(
            f"\nProcessing: "
            f"{dataset_name}"
        )

        # ====================================================
        # ORIGINAL DATA
        # ====================================================

        df = pd.read_csv(
            original_path
        )

        X = df[
            [f"X_{i}" for i in range(250)]
        ].to_numpy(dtype=float)

        Y_true = df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        N = len(df)

        
        print(
            f"N={N}"
        )

        # ====================================================
        # SAVED QUANTILES
        # ====================================================

        quantiles_path = (
            model_dir /
            f"{dataset_name}"
            f"_timestral-209m_standardized_"
            f"zero-shot_quantiles.npz"
        )

        if not quantiles_path.exists():
            print(
                "Missing quantiles:",
                quantiles_path.name
            )
            continue

        data = np.load(
            quantiles_path
        )

        raw_quantiles = data[
            "quantiles"
        ]

        quantile_levels = data[
            "quantile_levels"
        ]

        print(
            dataset_name,
            "raw quantiles:",
            raw_quantiles.shape
        )

        # Expected layout:
        # (N, 9, 20)

        # if raw_quantiles.shape != (
        #     N,
        #     9,
        #     20
        # ):
        #     raise ValueError(
        #         f"{dataset_name}: unexpected "
        #         f"quantile shape "
        #         f"{raw_quantiles.shape}"
        #     )

        if not np.allclose(
            quantile_levels,
            QUANTILE_LEVELS
        ):
            raise ValueError(
                f"{dataset_name}: unexpected "
                f"quantile levels "
                f"{quantile_levels}"
            )

        # --------------------------
        # Convert to common shape
        #
        # (N, 9, 20)
        # ->
        # (N, 20, 9)
        # --------------------------
        # quantiles = raw_quantiles.transpose(
        #     0,
        #     2,
        #     1
        # )

        quantiles = raw_quantiles

        if quantiles.shape != (
            N,
            20,
            9
        ):
            raise ValueError(
                f"{dataset_name}: unexpected "
                f"evaluation quantile shape "
                f"{quantiles.shape}"
            )

        # ====================================================
        # qCRPS
        # ====================================================

        crps_test = quantile_crps(
            Y_true,
            quantiles,
            QUANTILE_LEVELS
        )

        crps_by_horizon = np.mean(
            crps_test,
            axis=0
        )

        mean_qcrps = np.mean(
            crps_by_horizon
        )

        # ====================================================
        # POINT FORECAST CSV
        # ====================================================

        prediction_path = (
            model_dir /
            f"{dataset_name}"
            f"_timestral-209m_standardized_"
            f"zero-shot.csv"
        )

        if not prediction_path.exists():
            print(
                "Missing prediction:",
                prediction_path.name
            )
            continue

        prediction_df = pd.read_csv(
            prediction_path
        )

        Y_pred = prediction_df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        if len(Y_pred) != N:
            raise ValueError(
                f"{dataset_name}: "
                f"prediction N={len(Y_pred)} "
                f"but original N={N}"
            )


        # ====================================================
        # MASE
        # ====================================================

        (
            mean_mase,
            mase_by_horizon,
            model_mae_by_horizon,
            naive_mae_by_horizon
        ) = (
            calculate_mase_from_point_forecast(
                X,
                Y_true,
                Y_pred
            )
        )

        print(
            f"qCRPS={mean_qcrps:.8f} | "
            f"MASE={mean_mase:.8f}"
        )

        # ====================================================
        # NEW FILE PREFIX
        # ====================================================

        prefix = (
            f"{dataset_name}_timestral-209m_standardized_zero-shot"
        )

        # ====================================================
        # SAVE TEST-ONLY QUANTILES
        # ====================================================

        np.savez_compressed(
            model_dir / f"{dataset_name}_timestral-209m_standardized_zero-shot_quantiles.npz",

            quantiles=quantiles,
            quantile_levels=QUANTILE_LEVELS
        )

        # ====================================================
        # SAVE TEST-ONLY CRPS
        # ====================================================

        np.savez_compressed(
            model_dir / f"{dataset_name}_timestral-209m_standardized_zero-shot_crps.npz",

            crps=crps_test
        )

        # ====================================================
        # SAVE qCRPS BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                21
            ),
            "mean_qcrps":
                crps_by_horizon
        }).to_csv(
            model_dir /
            f"{prefix}"
            f"_crps_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SAVE MASE BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                21
            ),
            "model_mae":
                model_mae_by_horizon,
            "naive_mae":
                naive_mae_by_horizon,
            "mase":
                mase_by_horizon
        }).to_csv(
            model_dir /
            f"{prefix}"
            f"_mase_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SUMMARY
        # ====================================================

        summary_rows.append({
            "asset_class":
                asset_class,

            "dataset":
                dataset_name.replace("_no_preprocess_lookback-250_horizon-20_slide-1", "").replace("test_", ""),

            "model":
                "Timestral",

            "mean_qcrps":
                mean_qcrps,

            "mean_mase":
                mean_mase,

            "n_test_windows":
                N
        })

timestral_daily_summary = (
    pd.DataFrame(
        summary_rows
    )
)

timestral_daily_summary.to_csv(
    RESULTS_DIR /
    "timestral_train_test_"
    "summary_all_datasets.csv",
    index=False
)

display(
    timestral_daily_summary
)

### Weekly

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

QUANTILE_LEVELS = np.arange(0.1, 1.0, 0.1)

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

DATA_DIR = Path(
    r"BForTFin_log_return/arima_split_datasets/output_weekly"
)

RESULTS_DIR = Path(
    r"BForTFin_results/standardised/output_weekly/timestral_standardized_train_test"
)

In [ ]:
summary_rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = DATA_DIR / folder_name
    model_dir = (
        RESULTS_DIR /
        folder_name
    )

    print("\n" + "=" * 80)
    print(asset_class)
    print("=" * 80)

    for original_path in sorted(
        input_dir.glob("test_*.csv")
    ):

        dataset_name = (
            original_path.stem
        )

        # Exclude Tether
        if (
            asset_class == "Crypto"
            and "tether" in dataset_name.lower()
        ):
            print(
                f"Skipping Tether: "
                f"{dataset_name}"
            )
            continue

        print(
            f"\nProcessing: "
            f"{dataset_name}"
        )

        # ====================================================
        # ORIGINAL DATA
        # ====================================================

        df = pd.read_csv(
            original_path
        )

        X = df[
            [f"X_{i}" for i in range(250)]
        ].to_numpy(dtype=float)

        Y_true = df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        N = len(df)

        
        print(
            f"N={N}"
        )

        # ====================================================
        # SAVED QUANTILES
        # ====================================================

        quantiles_path = (
            model_dir /
            f"{dataset_name}"
            f"_timestral-209m_standardized_"
            f"zero-shot_quantiles.npz"
        )

        if not quantiles_path.exists():
            print(
                "Missing quantiles:",
                quantiles_path.name
            )
            continue

        data = np.load(
            quantiles_path
        )

        raw_quantiles = data[
            "quantiles"
        ]

        quantile_levels = data[
            "quantile_levels"
        ]

        print(
            dataset_name,
            "raw quantiles:",
            raw_quantiles.shape
        )

        # Expected layout:
        # (N, 9, 20)

        # if raw_quantiles.shape != (
        #     N,
        #     9,
        #     20
        # ):
        #     raise ValueError(
        #         f"{dataset_name}: unexpected "
        #         f"quantile shape "
        #         f"{raw_quantiles.shape}"
        #     )

        if not np.allclose(
            quantile_levels,
            QUANTILE_LEVELS
        ):
            raise ValueError(
                f"{dataset_name}: unexpected "
                f"quantile levels "
                f"{quantile_levels}"
            )

        # --------------------------
        # Convert to common shape
        #
        # (N, 9, 20)
        # ->
        # (N, 20, 9)
        # --------------------------
        # quantiles = raw_quantiles.transpose(
        #     0,
        #     2,
        #     1
        # )

        quantiles = raw_quantiles

        if quantiles.shape != (
            N,
            20,
            9
        ):
            raise ValueError(
                f"{dataset_name}: unexpected "
                f"evaluation quantile shape "
                f"{quantiles.shape}"
            )

        # ====================================================
        # qCRPS
        # ====================================================

        crps_test = quantile_crps(
            Y_true,
            quantiles,
            QUANTILE_LEVELS
        )

        crps_by_horizon = np.mean(
            crps_test,
            axis=0
        )

        mean_qcrps = np.mean(
            crps_by_horizon
        )

        # ====================================================
        # POINT FORECAST CSV
        # ====================================================

        prediction_path = (
            model_dir /
            f"{dataset_name}"
            f"_timestral-209m_standardized_"
            f"zero-shot.csv"
        )

        if not prediction_path.exists():
            print(
                "Missing prediction:",
                prediction_path.name
            )
            continue

        prediction_df = pd.read_csv(
            prediction_path
        )

        Y_pred = prediction_df[
            [f"Y_{i}" for i in range(20)]
        ].to_numpy(dtype=float)

        if len(Y_pred) != N:
            raise ValueError(
                f"{dataset_name}: "
                f"prediction N={len(Y_pred)} "
                f"but original N={N}"
            )


        # ====================================================
        # MASE
        # ====================================================

        (
            mean_mase,
            mase_by_horizon,
            model_mae_by_horizon,
            naive_mae_by_horizon
        ) = (
            calculate_mase_from_point_forecast(
                X,
                Y_true,
                Y_pred
            )
        )

        print(
            f"qCRPS={mean_qcrps:.8f} | "
            f"MASE={mean_mase:.8f}"
        )

        # ====================================================
        # NEW FILE PREFIX
        # ====================================================

        prefix = (
            f"{dataset_name}_timestral-209m_standardized_zero-shot"
        )

        # ====================================================
        # SAVE TEST-ONLY QUANTILES
        # ====================================================

        np.savez_compressed(
            model_dir / f"{dataset_name}_timestral-209m_standardized_zero-shot_quantiles.npz",

            quantiles=quantiles,
            quantile_levels=QUANTILE_LEVELS
        )

        # ====================================================
        # SAVE TEST-ONLY CRPS
        # ====================================================

        np.savez_compressed(
            model_dir / f"{dataset_name}_timestral-209m_standardized_zero-shot_crps.npz",

            crps=crps_test
        )

        # ====================================================
        # SAVE qCRPS BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                21
            ),
            "mean_qcrps":
                crps_by_horizon
        }).to_csv(
            model_dir /
            f"{prefix}"
            f"_crps_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SAVE MASE BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                21
            ),
            "model_mae":
                model_mae_by_horizon,
            "naive_mae":
                naive_mae_by_horizon,
            "mase":
                mase_by_horizon
        }).to_csv(
            model_dir /
            f"{prefix}"
            f"_mase_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SUMMARY
        # ====================================================

        summary_rows.append({
            "asset_class":
                asset_class,

            "dataset":
                dataset_name.replace("_no_preprocess_lookback-250_horizon-20_slide-1", "").replace("test_", ""),

            "model":
                "Timestral",

            "mean_qcrps":
                mean_qcrps,

            "mean_mase":
                mean_mase,

            "n_test_windows":
                N
        })

timestral_weekly_summary = (
    pd.DataFrame(
        summary_rows
    )
)

timestral_weekly_summary.to_csv(
    RESULTS_DIR /
    "timestral_train_test_"
    "summary_all_datasets.csv",
    index=False
)

display(
    timestral_weekly_summary
)

## Aggregation

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

FREQ = ["daily", "weekly"]

for freq in FREQ:

    BASE_DIR = Path(rf"BForTFin_results/standardised/output_{freq}")
    
    CRPS_FILES = {
        "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test" / "garch_1_1_train_test_summary_all_datasets.csv",
        "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test" / "garch-t-1-1-simulated_train_test_summary_all_datasets.csv",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test" / "lightgbm_train_test_summary_all_datasets.csv",
        "Sundial": BASE_DIR / "sundial_standardized_train_test" / "sundial_train_test_summary_all_datasets.csv",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test" / "lag-llama_train_test_summary_all_datasets.csv",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test" / "chronos_train_test_summary_all_datasets.csv",
        "Toto": BASE_DIR / "toto_standardized_train_test" / "toto_train_test_summary_all_datasets.csv",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test" / "timesfm_train_test_summary_all_datasets.csv",
        "Timestral": BASE_DIR / "timestral_standardized_train_test" / "timestral_train_test_summary_all_datasets.csv",
        "Moirai": BASE_DIR / "moirai_standardized_train_test" / "moirai_train_test_summary_all_datasets.csv",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test" / "moirai-moe_train_test_summary_all_datasets.csv",
    }
    
    MASE_FILES = {
        "ARIMA": BASE_DIR / "arima_standardized" / "arima_mase_summary_all_datasets.csv",
        "LightGBM" : BASE_DIR / "lightgbm_standardized_train_test" / "lightgbm_train_test_summary_all_datasets.csv",
        "LSTM" : BASE_DIR / "lstm_standardized_train_test" / "lstm_train_test_summary_all_datasets.csv",
        "Sundial": BASE_DIR / "sundial_standardized_train_test" / "sundial_train_test_summary_all_datasets.csv",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test" / "lag-llama_train_test_summary_all_datasets.csv",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test" / "chronos_train_test_summary_all_datasets.csv",
        "Toto": BASE_DIR / "toto_standardized_train_test" / "toto_train_test_summary_all_datasets.csv",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test" / "timesfm_train_test_summary_all_datasets.csv",
        "Timestral": BASE_DIR / "timestral_standardized_train_test" / "timestral_train_test_summary_all_datasets.csv",
        "Moirai": BASE_DIR / "moirai_standardized_train_test" / "moirai_train_test_summary_all_datasets.csv",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test" / "moirai-moe_train_test_summary_all_datasets.csv",
    }
    
    ASSET_ORDER = [
        "FX",
        "EM FX",
        "MSCI",
        "Rates",
        "REIT",
        "Equities",
        "ETF",
        "Crypto",
    ]
    
    
    def make_asset_class_table(
        model_files,
        metric_col
    ):
        rows = []
    
        for model_name, path in model_files.items():
    
            df = pd.read_csv(path).copy()
    
            # make sure Tether stays excluded
            df = df[
                ~df["dataset"].str.contains(
                    "Tether",
                    case=False,
                    na=False
                )
            ]
    
            asset_means = (
                df
                .groupby("asset_class")[metric_col]
                .mean()
            )
    
            row = {
                "Model": model_name
            }
    
            for asset in ASSET_ORDER:
                row[asset] = asset_means.get(
                    asset,
                    np.nan
                )
    
            rows.append(row)
    
        table = pd.DataFrame(rows)
    
        return table
    
    crps_table = make_asset_class_table(
        CRPS_FILES,
        "mean_qcrps"
    )
    
    mase_table = make_asset_class_table(
        MASE_FILES,
        "mean_mase"
    )
    
    display(crps_table)
    display(mase_table)
    
    def highlight_min(s):
        is_min = s == s.min()
        return [
            "font-weight: bold" if v else ""
            for v in is_min
        ]
    
    
    crps_styled = (
        crps_table
        .style
        .format(precision=6)
        .apply(
            highlight_min,
            subset=ASSET_ORDER
        )
    )
    
    mase_styled = (
        mase_table
        .style
        .format(precision=6)
        .apply(
            highlight_min,
            subset=ASSET_ORDER
        )
    )
    
    display(crps_styled)
    display(mase_styled)
    
    crps_table.to_csv(
        BASE_DIR /
        "final_mode_comparison_qcrps_final20_with_timestral.csv",
        index=False
    )
    
    mase_table.to_csv(
        BASE_DIR /
        "final_mode_comparison_mase_final20_with_timestral.csv",
        index=False
    )
    
    # Combine all CRPS results
    
    crps_frames = []
    
    for model_name, path in CRPS_FILES.items():
    
        df = pd.read_csv(path).copy()
    
        df["model"] = model_name
    
        df = df[
            ~df["dataset"].str.contains(
                "Tether",
                case=False,
                na=False
            )
        ]
    
        crps_frames.append(
            df[
                [
                    "asset_class",
                    "dataset",
                    "model",
                    "mean_qcrps"
                ]
            ]
        )
    
    all_crps = pd.concat(
        crps_frames,
        ignore_index=True
    )
    
    # Combine all MASE results
    mase_frames = []
    
    for model_name, path in MASE_FILES.items():
    
        df = pd.read_csv(path).copy()
    
        df["model"] = model_name
    
        df = df[
            ~df["dataset"].str.contains(
                "Tether",
                case=False,
                na=False
            )
        ]
    
        mase_frames.append(
            df[
                [
                    "asset_class",
                    "dataset",
                    "model",
                    "mean_mase"
                ]
            ]
        )
    
    all_mase = pd.concat(
        mase_frames,
        ignore_index=True
    )
    
    # Calculate winners
    rows = []
    
    datasets = sorted(
        set(all_crps["dataset"])
        & set(all_mase["dataset"])
    )
    
    for dataset in datasets:
    
        crps_group = all_crps[
            all_crps["dataset"] == dataset
        ].copy()
    
        mase_group = all_mase[
            all_mase["dataset"] == dataset
        ].copy()
    
        # -----------------------------
        # qCRPS
        # -----------------------------
    
        best_crps = (
            crps_group["mean_qcrps"].min()
        )
    
        best_model_crps = (
            crps_group.loc[
                crps_group[
                    "mean_qcrps"
                ].idxmin(),
                "model"
            ]
        )
    
        crps_within_1pct = (
            crps_group["mean_qcrps"]
            <= best_crps * 1.01
        )
    
        num_crps_within_1pct = (
            crps_within_1pct.sum()
        )
    
        # -----------------------------
        # MASE
        # -----------------------------
    
        best_mase = (
            mase_group["mean_mase"].min()
        )
    
        best_model_mase = (
            mase_group.loc[
                mase_group[
                    "mean_mase"
                ].idxmin(),
                "model"
            ]
        )
    
        mase_within_1pct = (
            mase_group["mean_mase"]
            <= best_mase * 1.01
        )
    
        num_mase_within_1pct = (
            mase_within_1pct.sum()
        )
    
        asset_class = (
            crps_group[
                "asset_class"
            ].iloc[0]
        )
    
        rows.append({
            "Class": asset_class,
            "Dataset": dataset,
    
            "Best qCRPS":
                best_crps,
    
            "Best Model (qCRPS)":
                best_model_crps,
    
            "Models within 1% (qCRPS)":
                num_crps_within_1pct,
    
            "Best MASE":
                best_mase,
    
            "Best Model (MASE)":
                best_model_mase,
    
            "Models within 1% (MASE)":
                num_mase_within_1pct,
        })
    
    
    best_table = pd.DataFrame(rows)
    
    # Sort according to benchmark table
    CLASS_ORDER = [
        "FX",
        "EM FX",
        "Rates",
        "REIT",
        "ETF",
        "MSCI",
        "Equities",
        "Crypto",
    ]
    
    best_table["Class"] = pd.Categorical(
        best_table["Class"],
        categories=CLASS_ORDER,
        ordered=True
    )
    
    best_table = (
        best_table
        .sort_values(
            ["Class", "Dataset"]
        )
        .reset_index(drop=True)
    )
    
    display(best_table)
    
    best_table.to_csv(
        BASE_DIR /
        "best_crps_mase_by_dataset_final20_with_timestral.csv",
        index=False
    )
    
    all_crps.to_csv(
        BASE_DIR /
        "all_models_crps_by_dataset_with_timestral.csv",
        index=False
    )
    
    all_mase.to_csv(
        BASE_DIR /
        "all_models_mase_by_dataset_with_timestral.csv",
        index=False
    )

# QSS

### Daily

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"BForTFin_results/standardised/output_daily"
)

DATA_DIR = Path(
    r"BForTFin_log_return/arima_split_datasets/output_daily"
)


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
    "Timestral": BASE_DIR / "timestral_standardized_train_test",
}


MASE_MODEL_DIRS = {
    "ARIMA": BASE_DIR / "arima_standardized",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "LSTM": BASE_DIR / "lstm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
    "Timestral": BASE_DIR / "timestral_standardized_train_test",
}


# Quantile-file suffix used by each model.
MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "Student-t GARCH": "_garch-t-1-1-simulated",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Timestral": "_timestral-209m_standardized_zero-shot",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
    "ARIMA": "_arima",
    "LSTM": "_lstm",
}


# ============================================================
# PINBALL LOSS
# ============================================================

def pinball_loss(y, q, tau=0.1):
    """
    Pinball loss rho_tau(y - q).

    y, q must have identical shapes.
    """

    y = np.asarray(y, dtype=float)
    q = np.asarray(q, dtype=float)

    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error,
    )


# ============================================================
# DATA HELPERS
# ============================================================

def load_test_xy(test_path):
    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def reference_q10(X):
    """
    Rolling empirical 10% quantile using ONLY
    the historical context X.
    """

    return np.quantile(
        X,
        TAU,
        axis=1,
    )


# ============================================================
# QSS
# ============================================================

def calculate_qss(
    X,
    Y,
    model_quantiles,
    quantile_levels,
):
    """
    Returns:
        overall model QL
        overall reference QL
        overall QSS
        QSS by horizon
        model QL by horizon
        reference QL by horizon
    """

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    Q = np.asarray(model_quantiles, dtype=float)

    quantile_levels = np.asarray(
        quantile_levels,
        dtype=float,
    )

    # --------------------------------------------------------
    # Locate tau = 0.1
    # --------------------------------------------------------

    q10_idx = np.argmin(
        np.abs(quantile_levels - TAU)
    )

    if not np.isclose(
        quantile_levels[q10_idx],
        TAU,
    ):
        raise ValueError(
            f"No tau={TAU} quantile found."
        )

    model_q10 = Q[:, :, q10_idx]

    if model_q10.shape != Y.shape:
        raise ValueError(
            f"Model q10 shape {model_q10.shape} "
            f"does not match Y shape {Y.shape}"
        )

    # --------------------------------------------------------
    # Model pinball losses
    # --------------------------------------------------------

    model_loss = pinball_loss(
        Y,
        model_q10,
        tau=TAU,
    )

    # --------------------------------------------------------
    # Reference q10:
    #
    # one empirical 10th percentile per forecast origin
    # calculated from X_0,...,X_249 only
    # --------------------------------------------------------

    q_ref = reference_q10(X)

    # Same reference forecast for every future horizon
    q_ref = q_ref[:, None]

    reference_loss = pinball_loss(
        Y,
        q_ref,
        tau=TAU,
    )

    # --------------------------------------------------------
    # Horizon-specific quantities
    # --------------------------------------------------------

    model_ql_by_horizon = np.mean(
        model_loss,
        axis=0,
    )

    ref_ql_by_horizon = np.mean(
        reference_loss,
        axis=0,
    )

    qss_by_horizon = (
        1.0
        - model_ql_by_horizon
        / ref_ql_by_horizon
    )

    # --------------------------------------------------------
    # Overall quantities
    #
    # IMPORTANT:
    # calculate ratio of aggregate losses,
    # NOT mean(qss_by_horizon)
    # --------------------------------------------------------

    model_ql = float(
        np.mean(model_loss)
    )

    ref_ql = float(
        np.mean(reference_loss)
    )

    qss = float(
        1.0
        - model_ql / ref_ql
    )

    return {
        "model_ql": model_ql,
        "reference_ql": ref_ql,
        "qss": qss,
        "model_ql_by_horizon": model_ql_by_horizon,
        "reference_ql_by_horizon": ref_ql_by_horizon,
        "qss_by_horizon": qss_by_horizon,
    }


# ============================================================
# FIND FILES
# ============================================================

def find_test_file(asset_class, dataset_stem):
    """
    Find the original held-out test CSV corresponding
    to the dataset.
    """

    asset_dir = DATA_DIR / asset_class

    matches = list(
        asset_dir.glob(
            f"{dataset_stem}.csv"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely identify test file for "
        f"{dataset_stem} in {asset_class}. "
        f"Found {len(matches)} matches."
    )


def dataset_stem_from_quantile_file(
    quantile_path,
    model_suffix,
):
    """
    Example:

    test_Bitcoin Historical Data_no_preprocess_...
        _lightgbm_quantiles.npz

    becomes

    test_Bitcoin Historical Data_no_preprocess_...
    """

    suffix = (
        f"{model_suffix}_quantiles.npz"
    )

    name = quantile_path.name

    if not name.endswith(suffix):
        raise ValueError(
            f"Unexpected quantile filename: {name}"
        )

    return name[:-len(suffix)]

def clean_garch_dataset_name(dataset_name):
    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace("_no_preprocess_lookback-250_horizon-20_slide-1", "")
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):
    """
    Locate GARCH(1,1) quantile file using the
    original GARCH filename convention.
    """

    asset_dir = model_root / asset_class

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    folder_name = asset_class

    prefix = (
        f"{folder_name.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    # Fallback in case there are naming variations
    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No GARCH quantile file found for "
            f"{dataset_name} in {asset_class}"
        )

    raise RuntimeError(
        f"Multiple GARCH quantile files found for "
        f"{dataset_name}: {matches}"
    )


# ============================================================
# CALCULATE QSS FOR ALL MODELS
# ============================================================

qss_summary_rows = []
qss_horizon_rows = []


for model_name, model_root in MODEL_DIRS.items():

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    model_suffix = MODEL_SUFFIXES[model_name]

    for asset_class, asset_name in ASSET_ORDER.items():

        model_asset_dir = BASE_DIR / model_root / asset_class

        test_asset_dir = DATA_DIR / asset_class

        test_files = sorted(
            test_asset_dir.glob("test_*.csv")
        )

        for test_path in test_files:

            try:
    
                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )
    
                # --------------------------------------------
                # Test X/Y
                # --------------------------------------------
    
                X, Y = load_test_xy(
                    test_path
                )
    
                # --------------------------------------------
                # Model quantiles
                # --------------------------------------------
    
                if model_name == "GARCH(1,1)":
                    garch_path = find_garch_quantile_file(
                        MODEL_DIRS["GARCH(1,1)"],
                        asset_class,
                        dataset_name,
                    )
    
                    quantile_path = garch_path
                else:
                    matches = list(
                        model_asset_dir.glob(
                            f"test_{dataset_name}{model_suffix}_quantiles.npz"
                        )
                    )
                
                    if len(matches) == 1:
                        quantile_path = matches[0]
                    else:
                        raise FileNotFoundError(
                            f"Could not uniquely identify test quantiles for "
                            f"{dataset_name} in {asset_class}. "
                            f"Found {len(matches)} matches."
                        ) 
                    
                with np.load(
                    quantile_path
                ) as q_file:
    
                    quantiles = np.asarray(
                        q_file["quantiles"],
                        dtype=float,
                    )
        
                    quantile_levels = np.asarray(
                        q_file["quantile_levels"],
                        dtype=float,
                    )
    
                # --------------------------------------------
                # QSS
                # --------------------------------------------
    
                result = calculate_qss(
                    X=X,
                    Y=Y,
                    model_quantiles=quantiles,
                    quantile_levels=quantile_levels,
                )
    
                qss_summary_rows.append({
                    "model": model_name,
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "model_ql_0.1": (
                        result["model_ql"]
                    ),
                    "reference_ql_0.1": (
                        result["reference_ql"]
                    ),
                    "qss_0.1": (
                        result["qss"]
                    ),
                })
    
                for h in range(HORIZON):
    
                    qss_horizon_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "horizon": h + 1,
                        "model_ql_0.1": (
                            result[
                                "model_ql_by_horizon"
                            ][h]
                        ),
                        "reference_ql_0.1": (
                            result[
                                "reference_ql_by_horizon"
                            ][h]
                        ),
                        "qss_0.1": (
                            result[
                                "qss_by_horizon"
                            ][h]
                        ),
                    })
    
                print(
                    f"  {dataset_name}: "
                    f"QSS={result['qss']:.6f}"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# SAVE QSS
# ============================================================

qss_summary_df = pd.DataFrame(
    qss_summary_rows
)

qss_horizon_df = pd.DataFrame(
    qss_horizon_rows
)

qss_summary_df.to_csv(
    BASE_DIR / "qss_summary_all_models_with_timestral.csv",
    index=False,
)

qss_horizon_df.to_csv(
    BASE_DIR / "qss_by_horizon_all_models_with_timestral.csv",
    index=False,
)


print()
print("Saved QSS outputs.")

### Weekly

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"BForTFin_results/standardised/output_weekly"
)

DATA_DIR = Path(
    r"BForTFin_log_return/arima_split_datasets/output_weekly"
)


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
    "Timestral": BASE_DIR / "timestral_standardized_train_test",
}


MASE_MODEL_DIRS = {
    "ARIMA": BASE_DIR / "arima_standardized",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "LSTM": BASE_DIR / "lstm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
    "Timestral": BASE_DIR / "timestral_standardized_train_test",
}


# Quantile-file suffix used by each model.
MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "Student-t GARCH": "_garch-t-1-1-simulated",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Timestral": "_timestral-209m_standardized_zero-shot",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
    "ARIMA": "_arima",
    "LSTM": "_lstm",
}


# ============================================================
# PINBALL LOSS
# ============================================================

def pinball_loss(y, q, tau=0.1):
    """
    Pinball loss rho_tau(y - q).

    y, q must have identical shapes.
    """

    y = np.asarray(y, dtype=float)
    q = np.asarray(q, dtype=float)

    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error,
    )


# ============================================================
# DATA HELPERS
# ============================================================

def load_test_xy(test_path):
    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def reference_q10(X):
    """
    Rolling empirical 10% quantile using ONLY
    the historical context X.
    """

    return np.quantile(
        X,
        TAU,
        axis=1,
    )


# ============================================================
# QSS
# ============================================================

def calculate_qss(
    X,
    Y,
    model_quantiles,
    quantile_levels,
):
    """
    Returns:
        overall model QL
        overall reference QL
        overall QSS
        QSS by horizon
        model QL by horizon
        reference QL by horizon
    """

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    Q = np.asarray(model_quantiles, dtype=float)

    quantile_levels = np.asarray(
        quantile_levels,
        dtype=float,
    )

    # --------------------------------------------------------
    # Locate tau = 0.1
    # --------------------------------------------------------

    q10_idx = np.argmin(
        np.abs(quantile_levels - TAU)
    )

    if not np.isclose(
        quantile_levels[q10_idx],
        TAU,
    ):
        raise ValueError(
            f"No tau={TAU} quantile found."
        )

    model_q10 = Q[:, :, q10_idx]

    if model_q10.shape != Y.shape:
        raise ValueError(
            f"Model q10 shape {model_q10.shape} "
            f"does not match Y shape {Y.shape}"
        )

    # --------------------------------------------------------
    # Model pinball losses
    # --------------------------------------------------------

    model_loss = pinball_loss(
        Y,
        model_q10,
        tau=TAU,
    )

    # --------------------------------------------------------
    # Reference q10:
    #
    # one empirical 10th percentile per forecast origin
    # calculated from X_0,...,X_249 only
    # --------------------------------------------------------

    q_ref = reference_q10(X)

    # Same reference forecast for every future horizon
    q_ref = q_ref[:, None]

    reference_loss = pinball_loss(
        Y,
        q_ref,
        tau=TAU,
    )

    # --------------------------------------------------------
    # Horizon-specific quantities
    # --------------------------------------------------------

    model_ql_by_horizon = np.mean(
        model_loss,
        axis=0,
    )

    ref_ql_by_horizon = np.mean(
        reference_loss,
        axis=0,
    )

    qss_by_horizon = (
        1.0
        - model_ql_by_horizon
        / ref_ql_by_horizon
    )

    # --------------------------------------------------------
    # Overall quantities
    #
    # IMPORTANT:
    # calculate ratio of aggregate losses,
    # NOT mean(qss_by_horizon)
    # --------------------------------------------------------

    model_ql = float(
        np.mean(model_loss)
    )

    ref_ql = float(
        np.mean(reference_loss)
    )

    qss = float(
        1.0
        - model_ql / ref_ql
    )

    return {
        "model_ql": model_ql,
        "reference_ql": ref_ql,
        "qss": qss,
        "model_ql_by_horizon": model_ql_by_horizon,
        "reference_ql_by_horizon": ref_ql_by_horizon,
        "qss_by_horizon": qss_by_horizon,
    }


# ============================================================
# FIND FILES
# ============================================================

def find_test_file(asset_class, dataset_stem):
    """
    Find the original held-out test CSV corresponding
    to the dataset.
    """

    asset_dir = DATA_DIR / asset_class

    matches = list(
        asset_dir.glob(
            f"{dataset_stem}.csv"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely identify test file for "
        f"{dataset_stem} in {asset_class}. "
        f"Found {len(matches)} matches."
    )


def dataset_stem_from_quantile_file(
    quantile_path,
    model_suffix,
):
    """
    Example:

    test_Bitcoin Historical Data_no_preprocess_...
        _lightgbm_quantiles.npz

    becomes

    test_Bitcoin Historical Data_no_preprocess_...
    """

    suffix = (
        f"{model_suffix}_quantiles.npz"
    )

    name = quantile_path.name

    if not name.endswith(suffix):
        raise ValueError(
            f"Unexpected quantile filename: {name}"
        )

    return name[:-len(suffix)]

def clean_garch_dataset_name(dataset_name):
    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace("_no_preprocess_lookback-250_horizon-20_slide-1", "")
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):
    """
    Locate GARCH(1,1) quantile file using the
    original GARCH filename convention.
    """

    asset_dir = model_root / asset_class

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    folder_name = asset_class

    prefix = (
        f"{folder_name.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    # Fallback in case there are naming variations
    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No GARCH quantile file found for "
            f"{dataset_name} in {asset_class}"
        )

    raise RuntimeError(
        f"Multiple GARCH quantile files found for "
        f"{dataset_name}: {matches}"
    )


# ============================================================
# CALCULATE QSS FOR ALL MODELS
# ============================================================

qss_summary_rows = []
qss_horizon_rows = []


for model_name, model_root in MODEL_DIRS.items():

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    model_suffix = MODEL_SUFFIXES[model_name]

    for asset_class, asset_name in ASSET_ORDER.items():

        model_asset_dir = BASE_DIR / model_root / asset_class

        test_asset_dir = DATA_DIR / asset_class

        test_files = sorted(
            test_asset_dir.glob("test_*.csv")
        )

        for test_path in test_files:

            try:
    
                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )
    
                # --------------------------------------------
                # Test X/Y
                # --------------------------------------------
    
                X, Y = load_test_xy(
                    test_path
                )
    
                # --------------------------------------------
                # Model quantiles
                # --------------------------------------------
    
                if model_name == "GARCH(1,1)":
                    garch_path = find_garch_quantile_file(
                        MODEL_DIRS["GARCH(1,1)"],
                        asset_class,
                        dataset_name,
                    )
    
                    quantile_path = garch_path
                else:
                    matches = list(
                        model_asset_dir.glob(
                            f"test_{dataset_name}{model_suffix}_quantiles.npz"
                        )
                    )
                
                    if len(matches) == 1:
                        quantile_path = matches[0]
                    else:
                        raise FileNotFoundError(
                            f"Could not uniquely identify test quantiles for "
                            f"{dataset_name} in {asset_class}. "
                            f"Found {len(matches)} matches."
                        ) 
                    
                with np.load(
                    quantile_path
                ) as q_file:
    
                    quantiles = np.asarray(
                        q_file["quantiles"],
                        dtype=float,
                    )
        
                    quantile_levels = np.asarray(
                        q_file["quantile_levels"],
                        dtype=float,
                    )
    
                # --------------------------------------------
                # QSS
                # --------------------------------------------
    
                result = calculate_qss(
                    X=X,
                    Y=Y,
                    model_quantiles=quantiles,
                    quantile_levels=quantile_levels,
                )
    
                qss_summary_rows.append({
                    "model": model_name,
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "model_ql_0.1": (
                        result["model_ql"]
                    ),
                    "reference_ql_0.1": (
                        result["reference_ql"]
                    ),
                    "qss_0.1": (
                        result["qss"]
                    ),
                })
    
                for h in range(HORIZON):
    
                    qss_horizon_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "horizon": h + 1,
                        "model_ql_0.1": (
                            result[
                                "model_ql_by_horizon"
                            ][h]
                        ),
                        "reference_ql_0.1": (
                            result[
                                "reference_ql_by_horizon"
                            ][h]
                        ),
                        "qss_0.1": (
                            result[
                                "qss_by_horizon"
                            ][h]
                        ),
                    })
    
                print(
                    f"  {dataset_name}: "
                    f"QSS={result['qss']:.6f}"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# SAVE QSS
# ============================================================

qss_summary_df = pd.DataFrame(
    qss_summary_rows
)

qss_horizon_df = pd.DataFrame(
    qss_horizon_rows
)

qss_summary_df.to_csv(
    BASE_DIR / "qss_summary_all_models_with_timestral.csv",
    index=False,
)

qss_horizon_df.to_csv(
    BASE_DIR / "qss_by_horizon_all_models_with_timestral.csv",
    index=False,
)


print()
print("Saved QSS outputs.")

# RMSE

### Daily

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

Y_COLS = [
    f"Y_{h}"
    for h in range(HORIZON)
]

# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

FREQ = ["daily", "weekly"]

for freq in FREQ:

    BASE_DIR = Path(
        rf"BForTFin_results/standardised/output_{freq}"
    )
    
    DATA_DIR = Path(
        rf"BForTFin_log_return/arima_split_datasets/output_{freq}"
    )
    
    
    
    MODEL_DIRS = {
        "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
        "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
        "Sundial": BASE_DIR / "sundial_standardized_train_test",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
        "Toto": BASE_DIR / "toto_standardized_train_test",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
        "Moirai": BASE_DIR / "moirai_standardized_train_test",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
        "Timestral": BASE_DIR / "timestral_standardized_train_test",
    }
    
    
    POINT_MODEL_DIRS = {
        "ARIMA": BASE_DIR / "arima_standardized",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
        "LSTM": BASE_DIR / "lstm_standardized_train_test",
        "Sundial": BASE_DIR / "sundial_standardized_train_test",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
        "Toto": BASE_DIR / "toto_standardized_train_test",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
        "Moirai": BASE_DIR / "moirai_standardized_train_test",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
        "Timestral": BASE_DIR / "timestral_standardized_train_test",
    }
    
    
    # Quantile-file suffix used by each model.
    MODEL_SUFFIXES = {
        "GARCH(1,1)": "_garch_1_1",
        "Student-t GARCH": "_garch-t-1-1-simulated",
        "LightGBM": "_lightgbm",
        "Sundial": "_sundial_zero-shot",
        "Lag-Llama": "_lag-llama_zero-shot",
        "Chronos-T5": "_chronos-t5-small_zero-shot",
        "Toto": "_toto_zero-shot",
        "TimesFM": "_timesfm-2.5-200m_zero-shot",
        "Timestral": "_timestral-209m_standardized_zero-shot",
        "Moirai": "_moirai-2.0-r-small_zero-shot",
        "Moirai-MoE": "_moirai-moe-1.0-r-base_zero-shot",
        "ARIMA": "_arima",
        "LSTM": "_lstm",
    }
    
    
    rmse_summary_rows = []
    rmse_horizon_rows = []
    
    
    for model_name, model_root in POINT_MODEL_DIRS.items():
    
        print()
        print(f"Calculating {freq} RMSE: {model_name}")
        model_suffix = MODEL_SUFFIXES[model_name]
    
        for asset_class, asset_name in ASSET_ORDER.items():
    
            asset_dir = model_root / asset_class
    
            if not asset_dir.exists():
                print(f"  Missing: {asset_dir}")
                continue
    
            point_files = sorted(
                asset_dir.glob(f"test_*{model_suffix}.csv")
            )
    
            print(
                f"  {asset_class}: "
                f"{len(point_files)} point files"
            )
    
            for point_path in point_files:
    
                try:
    
                    # ----------------------------------------
                    # Extract / match dataset name here
                    # ----------------------------------------
                    dataset_name = point_path.stem.replace(model_suffix, "")
    
                    test_path = find_test_file(
                        asset_class,
                        dataset_name,
                    )
    
                    # ----------------------------------------
                    # Load truth + forecast
                    # ----------------------------------------
    
                    test_df = pd.read_csv(test_path)
                    pred_df = pd.read_csv(point_path)
    
                    y_true = test_df[
                        Y_COLS
                    ].to_numpy(dtype=float)
    
                    y_pred = pred_df[
                        Y_COLS
                    ].to_numpy(dtype=float)
    
                    # ----------------------------------------
                    # Sanity checks
                    # ----------------------------------------
    
                    if y_true.shape != y_pred.shape:
                        raise ValueError(
                            f"Shape mismatch: "
                            f"true={y_true.shape}, "
                            f"pred={y_pred.shape}"
                        )
    
                    # ----------------------------------------
                    # RMSE
                    # ----------------------------------------
    
                    squared_error = (
                        y_true - y_pred
                    ) ** 2
    
                    rmse_by_horizon = np.sqrt(
                        np.mean(
                            squared_error,
                            axis=0,
                        )
                    )
    
                    overall_rmse = float(
                        np.mean(rmse_by_horizon)
                    )
    
                    # ----------------------------------------
                    # Save summary
                    # ----------------------------------------
    
                    rmse_summary_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name.removeprefix("test_"),
                        "rmse": overall_rmse,
                    })
    
                    for h in range(HORIZON):
    
                        rmse_horizon_rows.append({
                            "model": model_name,
                            "asset_class": asset_name,
                            "dataset": dataset_name.removeprefix("test_"),
                            "horizon": h + 1,
                            "rmse": float(
                                rmse_by_horizon[h]
                            ),
                        })
    
                    print(
                        f"    {dataset_name}: "
                        f"RMSE={overall_rmse:.6f}"
                    )
    
                except Exception as exc:
    
                    print(
                        f"    ERROR {point_path.name}: "
                        f"{exc}"
                    )
    
    
    # ==========================================
    # SAVE
    # ==========================================
    
    rmse_summary_df = pd.DataFrame(
        rmse_summary_rows
    )
    
    rmse_horizon_df = pd.DataFrame(
        rmse_horizon_rows
    )
    
    rmse_summary_df.to_csv(
        BASE_DIR / "rmse_summary_all_models_with_timestral.csv",
        index=False,
    )
    
    rmse_horizon_df.to_csv(
        BASE_DIR / "rmse_by_horizon_all_models_with_timestral.csv",
        index=False,
    )
    
    print()
    print("Saved RMSE outputs.")

## Aggregation

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

FREQ = ["daily", "weekly"]

for freq in FREQ:

    BASE_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_results"
        rf"\standardised\output_{freq}"
    )
    
    
    # DAILY_RMSE = pd.read_csv(
    #     BASE_DIR / "rmse_summary_all_models_with_timestral.csv"
    # )
    
    # DAILY_QSS = pd.read_csv(
    #     BASE_DIR / "qss_summary_all_models_with_timestral.csv"
    # )
    
    ALL_QSS = pd.read_csv(BASE_DIR / "qss_summary_all_models_with_timestral.csv").copy()
    
    ALL_RMSE = pd.read_csv(BASE_DIR / "rmse_summary_all_models_with_timestral.csv").copy()
    
    ASSET_ORDER = [
        "FX",
        "EM FX",
        "MSCI",
        "Rates",
        "REIT",
        "Equities",
        "ETF",
        "Crypto",
    ]
    
    MODEL_ORDER_RMSE = [
        "ARIMA",
        "LightGBM",
        "LSTM",
        "Sundial",
        "Lag-Llama",
        "Chronos-T5",
        "Toto",
        "TimesFM",
        "Timestral",
        "Moirai",
        "Moirai-MoE",
    ]
    
    rmse_summary = (
        ALL_RMSE
        .groupby(
            ["model", "asset_class"],
            as_index=False
        )["rmse"]
        .mean()
    )
    
    rmse_table = (
        rmse_summary
        .pivot(
            index="model",
            columns="asset_class",
            values="rmse"
        )
        .reindex(
            index=MODEL_ORDER_RMSE,
            columns=ASSET_ORDER
        )
    )
    
    rmse_table
    
    def bold_min_per_column(s):
        min_val = s.min()
    
        return [
            "font-weight: bold"
            if np.isclose(v, min_val)
            else ""
            for v in s
        ]
    
    
    styled_rmse = (
        rmse_table
        .style
        .format("{:.6f}")
        .apply(
            bold_min_per_column,
            axis=0
        )
    )
    
    display(styled_rmse)
    
    MODEL_ORDER_QSS = [
        "GARCH(1,1)",
        "Student-t GARCH",
        "LightGBM",
        "Sundial",
        "Lag-Llama",
        "Chronos-T5",
        "Toto",
        "TimesFM",
        "Timestral",
        "Moirai",
        "Moirai-MoE",
    ]
    
    qss_summary = (
        ALL_QSS
        .groupby(
            ["model", "asset_class"],
            as_index=False
        )["qss_0.1"]
        .mean()
    )
    
    qss_table = (
        qss_summary
        .pivot(
            index="model",
            columns="asset_class",
            values="qss_0.1"
        )
        .reindex(
            index=MODEL_ORDER_QSS,
            columns=ASSET_ORDER
        )
    )
    
    qss_table
    
    def bold_max_per_column(s):
        max_val = s.max()
    
        return [
            "font-weight: bold"
            if np.isclose(v, max_val)
            else ""
            for v in s
        ]
    
    
    styled_qss = (
        qss_table
        .style
        .format("{:.6f}")
        .apply(
            bold_max_per_column,
            axis=0
        )
    )
    
    display(styled_qss)
    
    print(f"{freq} RMSE:")
    display(styled_rmse)
    
    print(f"{freq} QSS:")
    display(styled_qss)
    
    qss_table.to_csv(
        BASE_DIR /
        "final_mode_comparison_qss_final20_with_timestral.csv",
        index=True
    )
    
    rmse_table.to_csv(
        BASE_DIR /
        "final_mode_comparison_rmse_final20_with_timestral.csv",
        index=True
    )
    
    # Calculate winners
    rows = []
    
    datasets = sorted(
        set(ALL_QSS["dataset"])
        & set(ALL_RMSE["dataset"])
    )
    
    for dataset in datasets:
    
        qss_group = ALL_QSS[
            ALL_QSS["dataset"] == dataset
        ].copy()
    
        rmse_group = ALL_RMSE[
            ALL_RMSE["dataset"] == dataset
        ].copy()
    
        # -----------------------------
        # QSS
        # -----------------------------
    
        best_qss = (
            qss_group["qss_0.1"].max()
        )
    
        best_model_qss = (
            qss_group.loc[
                qss_group[
                    "qss_0.1"
                ].idxmax(),
                "model"
            ]
        )
    
        qss_within_1pct = (
            best_qss - qss_group["qss_0.1"]
            <= 0.01 * abs(best_qss)
        )
    
        num_qss_within_1pct = (
            qss_within_1pct.sum()
        )
    
        # -----------------------------
        # RMSE
        # -----------------------------
    
        best_rmse = (
            rmse_group["rmse"].min()
        )
    
        best_model_rmse = (
            rmse_group.loc[
                rmse_group[
                    "rmse"
                ].idxmin(),
                "model"
            ]
        )
    
        rmse_within_1pct = (
            rmse_group["rmse"]
            <= best_rmse * 1.01
        )
    
        num_rmse_within_1pct = (
            rmse_within_1pct.sum()
        )
    
        asset_class = (
            qss_group[
                "asset_class"
            ].iloc[0]
        )
    
        rows.append({
            "Class": asset_class,
            "Dataset": dataset.replace(
                "_no_preprocess_lookback-250_horizon-20_slide-1",
                ""
            ),
            "Best QSS": best_qss,
            "Best Model (QSS)": best_model_qss,
            "Models within 1% (QSS)": num_qss_within_1pct,
            "Best RMSE": best_rmse,
            "Best Model (RMSE)": best_model_rmse,
            "Models within 1% (RMSE)": num_rmse_within_1pct,
        })
    
    
    best_table = pd.DataFrame(rows)
    display(best_table)
    
    # Sort according to benchmark table
    CLASS_ORDER = [
        "FX",
        "EM FX",
        "Rates",
        "REIT",
        "ETF",
        "MSCI",
        "Equities",
        "Crypto",
    ]
    
    best_table["Class"] = pd.Categorical(
        best_table["Class"],
        categories=CLASS_ORDER,
        ordered=True
    )
    
    best_table = (
        best_table
        .sort_values(
            ["Class", "Dataset"]
        )
        .reset_index(drop=True)
    )
    
    display(best_table)
    
    best_table.to_csv(
        BASE_DIR /
        "best_qss_rmse_by_dataset_final20_with_timestral.csv",
        index=False
    )

# Conventional MASE

### MAE

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

FREQ = ["daily", "weekly"]

for freq in FREQ:

    BASE_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_results"
        rf"\standardised\output_{freq}"
    )
    
    DATA_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_log_return"
        rf"\arima_split_datasets\output_{freq}"
    )
    
    MODEL_DIRS = {
        "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
        "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
        "Sundial": BASE_DIR / "sundial_standardized_train_test",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
        "Toto": BASE_DIR / "toto_standardized_train_test",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
        "Moirai": BASE_DIR / "moirai_standardized_train_test",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
        "Timestral": BASE_DIR / "timestral_standardized_train_test",
    }
    
    
    MASE_MODEL_DIRS = {
        "ARIMA": BASE_DIR / "arima_standardized",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
        "LSTM": BASE_DIR / "lstm_standardized_train_test",
        "Sundial": BASE_DIR / "sundial_standardized_train_test",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
        "Toto": BASE_DIR / "toto_standardized_train_test",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
        "Moirai": BASE_DIR / "moirai_standardized_train_test",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
        "Timestral": BASE_DIR / "timestral_standardized_train_test",
    }
    
    
    # Quantile-file suffix used by each model.
    MODEL_SUFFIXES = {
        "GARCH(1,1)": "_garch_1_1",
        "Student-t GARCH": "_garch-t-1-1-simulated",
        "LightGBM": "_lightgbm",
        "Sundial": "_sundial",
        "Lag-Llama": "_lag-llama",
        "Chronos-T5": "_chronos-t5-small",
        "Toto": "_toto",
        "TimesFM": "_timesfm-2.5-200m",
        "Timestral": "_timestral-209m_standardized_zero-shot",
        "Moirai": "_moirai-2.0-r-small",
        "Moirai-MoE": "_moirai-moe-1.0-r-base",
        "ARIMA": "_arima",
        "LSTM": "_lstm",
    }
    
    # ============================================================
    # COLLECT EXISTING MAE
    # ============================================================
    
    mae_summary_rows = []
    mae_horizon_rows = []
    
    
    for model_name, model_root in MASE_MODEL_DIRS.items():
    
        print()
        print(f"Collecting MAE: {model_name}")
        model_suffix = MODEL_SUFFIXES[model_name]
    
        for asset_class, asset_name in ASSET_ORDER.items():
    
            asset_dir = model_root / asset_class
    
            if not asset_dir.exists():
                print(
                    f"  Missing: {asset_dir}"
                )
                continue
    
            mase_files = sorted(
                asset_dir.glob(
                    "*_mase_by_horizon.csv"
                )
            )
    
            for mase_path in mase_files:
    
                try:
    
                    df = pd.read_csv(
                        mase_path
                    )
    
                    required = {
                        "horizon",
                        "model_mae",
                    }
    
                    missing = (
                        required
                        - set(df.columns)
                    )
    
                    if missing:
                        raise ValueError(
                            f"Missing columns: {missing}"
                        )
    
                    if len(df) != HORIZON:
                        raise ValueError(
                            f"Expected {HORIZON} horizons, "
                            f"found {len(df)}."
                        )
    
                    # Remove model-specific suffix
                    # while keeping dataset identifier.
                    dataset_name = (
                        mase_path.name
                        .replace(
                            "_mase_by_horizon.csv",
                            "",
                        )
                        .replace(model_suffix, "")
                        .removeprefix("test_")
                    )
    
                    overall_mae = float(
                        df["model_mae"].mean()
                    )
    
                    mae_summary_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "mae": overall_mae,
                    })
    
                    for _, row in df.iterrows():
    
                        mae_horizon_rows.append({
                            "model": model_name,
                            "asset_class": asset_name,
                            "dataset": dataset_name,
                            "horizon": int(
                                row["horizon"]
                            ),
                            "mae": float(
                                row["model_mae"]
                            ),
                        })
    
                except Exception as exc:
    
                    print(
                        f"  ERROR {mase_path.name}: "
                        f"{exc}"
                    )
    
    
    mae_summary_df = pd.DataFrame(
        mae_summary_rows
    )
    
    mae_horizon_df = pd.DataFrame(
        mae_horizon_rows
    )
    
    
    mae_summary_df.to_csv(
        BASE_DIR / "mae_summary_all_models_with_timestral.csv",
        index=False,
    )
    
    mae_horizon_df.to_csv(
        BASE_DIR / "mae_by_horizon_all_models_with_timestral.csv",
        index=False,
    )
    
    print("Saved MAE outputs.")

In [ ]:
import pandas as pd
from pathlib import Path

FREQ = ["daily", "weekly"]

for freq in FREQ:

    BASE_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_results"
        rf"\standardised\output_{freq}"
    )
    
    
    # ============================================================
    # LOAD EXISTING MODEL MAE
    # ============================================================
    
    mae_df = pd.read_csv(
        BASE_DIR / "mae_summary_all_models_with_timestral.csv"
    )
    
    
    # ============================================================
    # LOAD DATASET-SPECIFIC MASE SCALES
    # ============================================================
    
    baseline_df = pd.read_csv(
        BASE_DIR / f"point_baselines_{freq}.csv"
    )
    
    mase_scales = (
        baseline_df[
            [
                "asset_class",
                "dataset",
                "mase_scale",
            ]
        ]
        .drop_duplicates()
    )
    
    
    # ============================================================
    # MERGE
    # ============================================================
    
    out = mae_df.merge(
        mase_scales,
        on=[
            "asset_class",
            "dataset",
        ],
        how="left",
        validate="many_to_one",
    )
    
    
    if out["mase_scale"].isna().any():
        missing = out[
            out["mase_scale"].isna()
        ][
            [
                "asset_class",
                "dataset",
            ]
        ].drop_duplicates()
    
        raise ValueError(
            "Missing MASE scales for:\n"
            + missing.to_string(index=False)
        )
    
    
    # ============================================================
    # CONVENTIONAL MASE
    # ============================================================
    
    out["mase"] = (
        out["mae"]
        / out["mase_scale"]
    )
    
    
    # ============================================================
    # SAVE
    # ============================================================
    
    output_path = (
        BASE_DIR
        / "mase_conventional_summary_all_models_with_timestral.csv"
    )
    
    out.to_csv(
        output_path,
        index=False,
    )
    
    print(out.head())
    print()
    print(f"Rows: {len(out)}")
    print(f"Saved to: {output_path}")
    
    mae_h = pd.read_csv(
        BASE_DIR / "mae_by_horizon_all_models_with_timestral.csv"
    )
    
    out_h = mae_h.merge(
        mase_scales,
        on=[
            "asset_class",
            "dataset",
        ],
        how="left",
        validate="many_to_one",
    )
    
    out_h["mase"] = (
        out_h["mae"]
        / out_h["mase_scale"]
    )
    
    out_h.to_csv(
        BASE_DIR / "mase_conventional_by_horizon_all_models_with_timestral.csv",
        index=False,
    )
    
    print(out_h.head())
    print()
    print(f"Rows: {len(out_h)}")

### Rolling-context

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

LOOKBACK = 250
HORIZON = 20

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ============================================================
# MODEL DIRECTORIES
# ============================================================

POINT_MODEL_DIRS = {
    "ARIMA": "arima_standardized",
    "LightGBM": "lightgbm_standardized_train_test",
    "LSTM": "lstm_standardized_train_test",
    "Sundial": "sundial_standardized_train_test",
    "Lag-Llama": "lag-llama_standardized_train_test",
    "Chronos-T5": "chronos_standardized_train_test",
    "Toto": "toto_standardized_train_test",
    "TimesFM": "timesfm_standardized_train_test",
    "Timestral": "timestral_standardized_train_test",
    "Moirai": "moirai_standardized_train_test",
    "Moirai-MoE": "moirai_moe_standardized_train_test",
}


# ============================================================
# EXACT POINT-FORECAST SUFFIXES
# ============================================================

MODEL_SUFFIXES = {
    "ARIMA": "_arima",
    "LightGBM": "_lightgbm",
    "LSTM": "_lstm",
    "Sundial": "_sundial_zero-shot",
    "Lag-Llama": "_lag-llama_zero-shot",
    "Chronos-T5": "_chronos-t5-small_zero-shot",
    "Toto": "_toto_zero-shot",
    "TimesFM": "_timesfm-2.5-200m_zero-shot",
    "Timestral": "_timestral-209m_standardized_zero-shot",
    "Moirai": "_moirai-2.0-r-small_zero-shot",
    "Moirai-MoE": "_moirai-moe-1.0-r-base_zero-shot",
}


X_COLS = [
    f"X_{i}"
    for i in range(LOOKBACK)
]

Y_COLS = [
    f"Y_{h}"
    for h in range(HORIZON)
]


# ============================================================
# HELPER: LOAD ORIGINAL TEST DATA
# ============================================================

def load_test_xy(test_path):

    df = pd.read_csv(test_path)

    missing = [
        c for c in X_COLS + Y_COLS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {test_path.name}: "
            f"{missing}"
        )

    X = df[X_COLS].to_numpy(dtype=float)
    Y = df[Y_COLS].to_numpy(dtype=float)

    if X.shape[1] != LOOKBACK:
        raise ValueError(
            f"Unexpected X shape: {X.shape}"
        )

    if Y.shape[1] != HORIZON:
        raise ValueError(
            f"Unexpected Y shape: {Y.shape}"
        )

    if not np.all(np.isfinite(X)):
        raise ValueError(
            f"Non-finite X values in {test_path.name}"
        )

    if not np.all(np.isfinite(Y)):
        raise ValueError(
            f"Non-finite Y values in {test_path.name}"
        )

    return X, Y


# ============================================================
# HELPER: ROLLING 250-CONTEXT MASE SCALE
# ============================================================

def calculate_context_scales(X):

    """
    X shape:
        (n_origins, 250)

    Returns:
        One MASE scaling factor per forecast origin.

    scale_i =
        mean(abs(diff(X_i)))

    Shape:
        (n_origins,)
    """

    X = np.asarray(X, dtype=float)

    if X.ndim != 2 or X.shape[1] != LOOKBACK:
        raise ValueError(
            f"Unexpected context shape: {X.shape}"
        )

    differences = np.abs(
        np.diff(X, axis=1)
    )

    scales = np.mean(
        differences,
        axis=1
    )

    if not np.all(np.isfinite(scales)):
        raise ValueError(
            "Non-finite context scales detected."
        )

    if np.any(scales <= 0):
        raise ValueError(
            "Zero or negative context MASE scale."
        )

    return scales


# ============================================================
# HELPER: LOAD SAVED POINT FORECASTS
# ============================================================

def load_point_forecasts(prediction_path):

    df = pd.read_csv(prediction_path)

    missing = [
        c for c in Y_COLS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing prediction columns in "
            f"{prediction_path.name}: {missing}"
        )

    predictions = df[
        Y_COLS
    ].to_numpy(dtype=float)

    if not np.all(np.isfinite(predictions)):
        raise ValueError(
            f"Non-finite predictions in "
            f"{prediction_path.name}"
        )

    return predictions


# ============================================================
# HELPER: EVALUATE ONE MODEL ON ONE DATASET
# ============================================================

def evaluate_point_context_mase(
    Y,
    predictions,
    context_scales,
):

    Y = np.asarray(Y, dtype=float)

    predictions = np.asarray(
        predictions,
        dtype=float
    )

    context_scales = np.asarray(
        context_scales,
        dtype=float
    )

    if Y.shape != predictions.shape:
        raise ValueError(
            f"Prediction shape mismatch: "
            f"Y={Y.shape}, "
            f"predictions={predictions.shape}"
        )

    if context_scales.shape != (Y.shape[0],):
        raise ValueError(
            "Context scales do not match "
            "the number of forecast origins."
        )

    # --------------------------------------------------------
    # ERRORS
    # --------------------------------------------------------

    errors = Y - predictions

    absolute_errors = np.abs(errors)

    squared_errors = errors ** 2


    # --------------------------------------------------------
    # RMSE
    #
    # 1. RMSE separately for each horizon
    # 2. Average horizon RMSE values
    # --------------------------------------------------------

    rmse_by_horizon = np.sqrt(
        np.mean(
            squared_errors,
            axis=0
        )
    )

    overall_rmse = float(
        np.mean(rmse_by_horizon)
    )


    # --------------------------------------------------------
    # MAE
    # --------------------------------------------------------

    mae_by_horizon = np.mean(
        absolute_errors,
        axis=0
    )

    overall_mae = float(
        np.mean(mae_by_horizon)
    )


    # --------------------------------------------------------
    # ROLLING 250-CONTEXT MASE
    #
    # Divide BEFORE averaging.
    # --------------------------------------------------------

    scaled_errors = (
        absolute_errors
        / context_scales[:, None]
    )

    mase_by_horizon = np.mean(
        scaled_errors,
        axis=0
    )

    overall_mase = float(
        np.mean(mase_by_horizon)
    )


    return {
        "rmse": overall_rmse,
        "mae": overall_mae,
        "mase_250": overall_mase,
        "rmse_by_horizon": rmse_by_horizon,
        "mae_by_horizon": mae_by_horizon,
        "mase_250_by_horizon": mase_by_horizon,
    }


# ============================================================
# PROCESS DAILY AND WEEKLY
# ============================================================

all_summary_rows = []
all_horizon_rows = []
all_failures = []

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 75)
    print(f"PROCESSING {frequency.upper()}")
    print("=" * 75)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    if not DATA_DIR.is_dir():
        raise FileNotFoundError(DATA_DIR)

    if not BASE_DIR.is_dir():
        raise FileNotFoundError(BASE_DIR)

    frequency_summary = []
    frequency_horizon = []
    frequency_failures = []

    # --------------------------------------------------------
    # LOOP THROUGH ASSET CLASSES
    # --------------------------------------------------------

    for folder, asset_class in ASSET_ORDER.items():

        test_asset_dir = DATA_DIR / folder

        if not test_asset_dir.is_dir():
            raise FileNotFoundError(test_asset_dir)

        test_files = sorted(
            test_asset_dir.glob("test_*.csv")
        )

        print()
        print(
            f"{asset_class}: "
            f"{len(test_files)} test datasets"
        )

        for test_path in test_files:

            dataset_stem = test_path.stem

            dataset_name = (
                dataset_stem.removeprefix("test_")
            )

            try:

                # ------------------------------------------------
                # ORIGINAL TEST INPUTS AND TARGETS
                # ------------------------------------------------

                X_test, Y_test = load_test_xy(
                    test_path
                )

                n_origins = len(Y_test)

                if n_origins == 0:
                    raise ValueError(
                        "Empty test dataset."
                    )

                # ------------------------------------------------
                # ONE SCALE PER ORIGIN
                # ------------------------------------------------

                context_scales = calculate_context_scales(
                    X_test
                )

            except Exception as exc:

                # All models depend on these shared inputs.
                # Record every affected model.

                for model_name in POINT_MODEL_DIRS:

                    frequency_failures.append({
                        "frequency": frequency,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "model": model_name,
                        "error": str(exc),
                    })

                print(
                    f"  ERROR loading {dataset_name}: {exc}"
                )

                continue


            # ----------------------------------------------------
            # LOOP THROUGH MODELS
            # ----------------------------------------------------

            for model_name, model_dir in POINT_MODEL_DIRS.items():

                model_asset_dir = (
                    BASE_DIR
                    / model_dir
                    / folder
                )

                model_suffix = MODEL_SUFFIXES[
                    model_name
                ]

                prediction_path = (
                    model_asset_dir
                    / f"{dataset_stem}{model_suffix}.csv"
                )

                try:

                    if not prediction_path.is_file():
                        raise FileNotFoundError(
                            prediction_path
                        )

                    # --------------------------------------------
                    # LOAD SAVED POINT FORECASTS
                    # --------------------------------------------

                    Y_pred = load_point_forecasts(
                        prediction_path
                    )

                    # --------------------------------------------
                    # CHECK ALIGNMENT SHAPE
                    # --------------------------------------------

                    if Y_pred.shape != Y_test.shape:
                        raise ValueError(
                            f"Shape mismatch: "
                            f"pred={Y_pred.shape}, "
                            f"actual={Y_test.shape}"
                        )

                    # --------------------------------------------
                    # EVALUATE
                    # --------------------------------------------

                    result = evaluate_point_context_mase(
                        Y_test,
                        Y_pred,
                        context_scales
                    )

                    # --------------------------------------------
                    # SUMMARY ROW
                    # --------------------------------------------

                    frequency_summary.append({
                        "frequency": frequency,
                        "model": model_name,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "rmse": result["rmse"],
                        "mae": result["mae"],
                        "mase_250": result["mase_250"],
                        "n_origins": n_origins,
                        "mean_context_scale": float(
                            np.mean(context_scales)
                        ),
                        "min_context_scale": float(
                            np.min(context_scales)
                        ),
                    })

                    # --------------------------------------------
                    # HORIZON ROWS
                    # --------------------------------------------

                    for h in range(HORIZON):

                        frequency_horizon.append({
                            "frequency": frequency,
                            "model": model_name,
                            "asset_class": asset_class,
                            "dataset": dataset_name,
                            "horizon": h + 1,
                            "rmse": float(
                                result["rmse_by_horizon"][h]
                            ),
                            "mae": float(
                                result["mae_by_horizon"][h]
                            ),
                            "mase_250": float(
                                result["mase_250_by_horizon"][h]
                            ),
                        })

                except Exception as exc:

                    frequency_failures.append({
                        "frequency": frequency,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "model": model_name,
                        "error": str(exc),
                    })

                    print(
                        f"  ERROR: {model_name} / "
                        f"{dataset_name}: {exc}"
                    )


    # ========================================================
    # SUMMARY DATAFRAMES
    # ========================================================

    summary_df = pd.DataFrame(
        frequency_summary
    )

    horizon_df = pd.DataFrame(
        frequency_horizon
    )

    failures_df = pd.DataFrame(
        frequency_failures
    )

    # --------------------------------------------------------
    # EXPECTED COVERAGE
    # --------------------------------------------------------

    n_datasets = sum(
        len(list((DATA_DIR / folder).glob("test_*.csv")))
        for folder in ASSET_ORDER
    )

    n_models = len(POINT_MODEL_DIRS)

    expected_rows = n_datasets * n_models

    print()
    print("-" * 75)
    print(f"Frequency: {frequency}")
    print(f"Datasets found: {n_datasets}")
    print(f"Models: {n_models}")
    print(f"Expected model-dataset rows: {expected_rows}")
    print(f"Successful rows: {len(summary_df)}")
    print(f"Failed rows: {len(failures_df)}")
    print("-" * 75)

    if not summary_df.empty:

        print()
        print("Mean MASE by model:")

        print(
            summary_df
            .groupby("model")["mase_250"]
            .agg(["count", "mean", "median"])
            .sort_values("mean")
            .round(6)
            .to_string()
        )


    # ========================================================
    # SAVE
    #
    # Do not overwrite original metric files.
    # ========================================================

    summary_path = (
        BASE_DIR
        / f"point_models_context_mase_{frequency}_with_timestral.csv"
    )

    horizon_path = (
        BASE_DIR
        / f"point_models_context_mase_by_horizon_{frequency}_with_timestral.csv"
    )

    failures_path = (
        BASE_DIR
        / f"point_models_context_mase_failures_{frequency}_with_timestral.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    horizon_df.to_csv(
        horizon_path,
        index=False
    )

    failures_df.to_csv(
        failures_path,
        index=False
    )

    print()
    print(f"Saved: {summary_path}")
    print(f"Saved: {horizon_path}")
    print(f"Saved: {failures_path}")

    all_summary_rows.extend(frequency_summary)
    all_horizon_rows.extend(frequency_horizon)
    all_failures.extend(frequency_failures)

    if len(failures_df) > 0:
        print(
            "\nWARNING: Some model-dataset combinations "
            "failed. Check the failures CSV."
        )

    if len(summary_df) != expected_rows:
        print(
            "\nWARNING: Coverage is incomplete."
        )


# ============================================================
# FINAL COVERAGE CHECK
# ============================================================

print()
print("=" * 75)
print("FINAL SUMMARY")
print("=" * 75)

all_summary_df = pd.DataFrame(
    all_summary_rows
)

if not all_summary_df.empty:

    print(
        all_summary_df
        .groupby(["frequency", "model"])["mase_250"]
        .agg(["count", "mean", "median"])
        .round(6)
        .to_string()
    )

print()
print(f"Total failures: {len(all_failures)}")

if all_failures:
    print(
        "\nReview the saved failure reports."
    )

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

for frequency in ["daily", "weekly"]:

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    baseline_path = (
        BASE_DIR
        / f"point_baselines_context_mase_{frequency}.csv"
    )

    model_path = (
        BASE_DIR
        / f"point_models_context_mase_{frequency}_with_timestral.csv"
    )

    baselines = pd.read_csv(baseline_path)
    models = pd.read_csv(model_path)

    # Standardize method column
    baselines = baselines.rename(
        columns={"baseline": "model"}
    )

    baselines["type"] = "Simple baseline"
    models["type"] = "Forecasting model"

    required = [
        "asset_class",
        "dataset",
        "model",
        "rmse",
        "mae",
        "mase_250",
    ]

    for name, df in [
        ("Baselines", baselines),
        ("Models", models),
    ]:
        missing = [
            col for col in required
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"{name} missing columns: {missing}"
            )

    combined = pd.concat(
        [
            baselines[
                required + ["type"]
            ],
            models[
                required + ["type"]
            ],
        ],
        ignore_index=True,
    )

    # Prevent duplicate dataset-method entries
    keys = [
        "asset_class",
        "dataset",
        "model",
    ]

    if combined.duplicated(keys).any():
        raise ValueError(
            "Duplicate dataset-method entries detected."
        )

    dataset_keys = [
        "asset_class",
        "dataset",
    ]

    # Ensure every method has the same dataset coverage
    counts = combined.groupby("model").size()

    n_datasets = (
        combined[dataset_keys]
        .drop_duplicates()
        .shape[0]
    )

    if not (counts == n_datasets).all():
        raise ValueError(
            "Incomplete model coverage:\n"
            + counts.to_string()
        )

    # --------------------------------------------------
    # RANK MODELS WITHIN EACH DATASET
    # --------------------------------------------------

    combined["mase_rank"] = (
        combined
        .groupby(dataset_keys)["mase_250"]
        .rank(method="average", ascending=True)
    )

    combined["rmse_rank"] = (
        combined
        .groupby(dataset_keys)["rmse"]
        .rank(method="average", ascending=True)
    )

    # --------------------------------------------------
    # COMPARE AGAINST ZERO RETURN
    # --------------------------------------------------

    zero = (
        combined[
            combined["model"] == "Zero"
        ][dataset_keys + ["mase_250", "rmse"]]
        .rename(
            columns={
                "mase_250": "zero_mase",
                "rmse": "zero_rmse",
            }
        )
    )

    combined = combined.merge(
        zero,
        on=dataset_keys,
        how="left",
        validate="many_to_one",
    )

    if combined[
        ["zero_mase", "zero_rmse"]
    ].isna().any().any():
        raise ValueError(
            "Some datasets are missing the zero baseline."
        )

    combined["beats_zero_mase"] = (
        combined["mase_250"]
        <
        combined["zero_mase"]
    )

    combined["beats_zero_rmse"] = (
        combined["rmse"]
        <
        combined["zero_rmse"]
    )

    # --------------------------------------------------
    # WINS
    # --------------------------------------------------

    # Count all methods tied for best score.
    # Wins may therefore sum to more than D.
    best_mase = (
        combined
        .groupby(dataset_keys)["mase_250"]
        .transform("min")
    )

    combined["mase_win"] = (
        combined["mase_250"] == best_mase
    )

    # --------------------------------------------------
    # SUMMARIZE
    # --------------------------------------------------

    summary = (
        combined
        .groupby(["model", "type"])
        .agg(
            datasets=("dataset", "size"),
            mean_mase=("mase_250", "mean"),
            median_mase=("mase_250", "median"),
            mean_mase_rank=("mase_rank", "mean"),
            mase_wins=("mase_win", "sum"),
            beats_zero_mase_pct=(
                "beats_zero_mase", "mean"
            ),
            mean_rmse=("rmse", "mean"),
            mean_rmse_rank=("rmse_rank", "mean"),
            beats_zero_rmse_pct=(
                "beats_zero_rmse", "mean"
            ),
        )
        .reset_index()
    )

    summary[
        "beats_zero_mase_pct"
    ] *= 100

    summary[
        "beats_zero_rmse_pct"
    ] *= 100

    summary = summary.sort_values(
        "mean_mase_rank"
    )

    print()
    print("=" * 80)
    print(f"{frequency.upper()} COMPARISON")
    print("=" * 80)

    print(
        summary.round(4).to_string(index=False)
    )

    combined.to_csv(
        BASE_DIR
        / f"point_all_context_mase_{frequency}_with_timestral.csv",
        index=False,
    )

    summary.to_csv(
        BASE_DIR
        / f"point_all_context_mase_summary_{frequency}_with_timestral.csv",
        index=False,
    )

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

for frequency in ["daily", "weekly"]:

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    path = (
        BASE_DIR
        / f"point_all_context_mase_{frequency}_with_timestral.csv"
    )

    df = pd.read_csv(path)

    keys = ["asset_class", "dataset"]

    # ---------------------------------------------------
    # A. FIXED VS ROLLING LOCATION BASELINES
    # ---------------------------------------------------

    wide = df.pivot(
        index=keys,
        columns="model",
        values="mase_250"
    )

    comparisons = [
        ("Train Mean", "Context Mean"),
        ("Train Median", "Context Median"),
    ]

    location_rows = []

    for fixed, rolling in comparisons:

        difference = (
            wide[rolling] - wide[fixed]
        )

        location_rows.append({
            "comparison": f"{fixed} vs {rolling}",
            "datasets": len(difference),
            "fixed_better_pct": (
                100 * (difference > 0).mean()
            ),
            "rolling_better_pct": (
                100 * (difference < 0).mean()
            ),
            "mean_rolling_minus_fixed": (
                difference.mean()
            ),
            "median_rolling_minus_fixed": (
                difference.median()
            ),
        })

    location_summary = pd.DataFrame(
        location_rows
    )

    print()
    print("=" * 75)
    print(f"{frequency.upper()}: FIXED VS ROLLING")
    print("=" * 75)

    print(
        location_summary
        .round(6)
        .to_string(index=False)
    )

    # ---------------------------------------------------
    # B. RELATIVE IMPROVEMENT OVER ZERO
    # ---------------------------------------------------

    zero = wide["Zero"]

    relative_improvement = (
        wide.sub(zero, axis=0)
        .mul(-1)
        .div(zero, axis=0)
    ) * 100

    improvement_summary = (
        relative_improvement
        .agg(["mean", "median"])
        .T
        .rename(
            columns={
                "mean": "mean_improvement_pct",
                "median": "median_improvement_pct",
            }
        )
    )

    improvement_summary[
        "beats_zero_pct"
    ] = (
        relative_improvement.gt(0).mean() * 100
    )

    improvement_summary = (
        improvement_summary
        .sort_values(
            "median_improvement_pct",
            ascending=False
        )
        .reset_index()
        .rename(columns={"model": "model"})
    )

    print()
    print("=" * 75)
    print(
        f"{frequency.upper()}: "
        "RELATIVE IMPROVEMENT OVER ZERO"
    )
    print("=" * 75)

    print(
        improvement_summary
        .round(4)
        .to_string(index=False)
    )

    # ---------------------------------------------------
    # SAVE
    # ---------------------------------------------------

    location_summary.to_csv(
        BASE_DIR
        / f"fixed_vs_rolling_location_{frequency}_with_timestral.csv",
        index=False
    )

    improvement_summary.to_csv(
        BASE_DIR
        / f"mase_improvement_over_zero_{frequency}_with_timestral.csv",
        index=False
    )

# MSIS

In [ ]:
import numpy as np


def calculate_msis_250(
    Y,
    lower,
    upper,
    context_scales,
    alpha=0.2,
):
    """
    Rolling 250-context Mean Scaled Interval Score.

    Parameters
    ----------
    Y : array, shape (n_origins, 20)
        Actual future returns.

    lower : array, shape (n_origins, 20)
        Predicted lower quantile (q0.1 for 80% intervals).

    upper : array, shape (n_origins, 20)
        Predicted upper quantile (q0.9 for 80% intervals).

    context_scales : array, shape (n_origins,)
        Mean absolute one-step naive differences
        computed from each origin's 250-point context.

    alpha : float
        Miscoverage probability.
        alpha=0.2 corresponds to 80% nominal coverage.

    Returns
    -------
    Dictionary containing:
        Overall MSIS
        Horizon-specific MSIS
        Mean interval score (unscaled)
        Empirical interval coverage
        Mean interval width
    """

    Y = np.asarray(Y, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    context_scales = np.asarray(
        context_scales,
        dtype=float
    )

    if not (
        Y.shape == lower.shape == upper.shape
    ):
        raise ValueError(
            "Y, lower and upper must have matching shapes."
        )

    if context_scales.shape != (Y.shape[0],):
        raise ValueError(
            "Context scale shape mismatch."
        )

    if not (0 < alpha < 1):
        raise ValueError(
            "alpha must be between 0 and 1."
        )

    if not np.all(np.isfinite(Y)):
        raise ValueError("Non-finite targets.")

    if not np.all(np.isfinite(lower)):
        raise ValueError("Non-finite lower bounds.")

    if not np.all(np.isfinite(upper)):
        raise ValueError("Non-finite upper bounds.")

    if np.any(lower > upper):
        raise ValueError(
            "Some prediction intervals have lower > upper."
        )

    if (
        not np.all(np.isfinite(context_scales))
        or np.any(context_scales <= 0)
    ):
        raise ValueError(
            "Invalid context scaling factors."
        )

    # --------------------------------------------------
    # INTERVAL WIDTH
    # --------------------------------------------------

    width = upper - lower

    # --------------------------------------------------
    # LOWER AND UPPER MIS-COVERAGE PENALTIES
    # --------------------------------------------------

    lower_penalty = (
        (2.0 / alpha)
        * np.maximum(lower - Y, 0.0)
    )

    upper_penalty = (
        (2.0 / alpha)
        * np.maximum(Y - upper, 0.0)
    )

    # --------------------------------------------------
    # UNSCALED INTERVAL SCORE
    # --------------------------------------------------

    interval_scores = (
        width
        + lower_penalty
        + upper_penalty
    )

    # --------------------------------------------------
    # ROLLING-CONTEXT SCALED INTERVAL SCORE
    # --------------------------------------------------

    scaled_scores = (
        interval_scores
        / context_scales[:, None]
    )

    # --------------------------------------------------
    # HORIZON-SPECIFIC AGGREGATION
    # --------------------------------------------------

    msis_by_horizon = np.mean(
        scaled_scores,
        axis=0
    )

    # --------------------------------------------------
    # OVERALL MSIS
    # --------------------------------------------------

    msis = float(
        np.mean(msis_by_horizon)
    )

    # --------------------------------------------------
    # ADDITIONAL DIAGNOSTICS
    # --------------------------------------------------

    inside = (
        (Y >= lower)
        & (Y <= upper)
    )

    coverage = float(
        np.mean(inside)
    )

    mean_width = float(
        np.mean(width)
    )

    mean_interval_score = float(
        np.mean(interval_scores)
    )

    return {
        "msis_250": msis,
        "msis_250_by_horizon": msis_by_horizon,
        "mean_interval_score": mean_interval_score,
        "coverage": coverage,
        "mean_width": mean_width,
    }

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

LOOKBACK = 250
HORIZON = 20
ALPHA = 0.20

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_DIRECTORIES = {
    "GARCH(1,1)": "garch_standardized_train_test",
    "LightGBM": "lightgbm_standardized_train_test",
    "Sundial": "sundial_standardized_train_test",
    "Lag-Llama": "lag-llama_standardized_train_test",
    "Chronos-T5": "chronos_standardized_train_test",
    "Toto": "toto_standardized_train_test",
    "TimesFM": "timesfm_standardized_train_test",
    "Timestral": "timestral_standardized_train_test",
    "Moirai": "moirai_standardized_train_test",
    "Moirai-MoE": "moirai_moe_standardized_train_test",
    "Student-t GARCH": "garch_t_1_1_standardized_train_test",
}

MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Timestral": "_timestral-209m_standardized_zero-shot",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
    "Student-t GARCH": "_garch-t-1-1-simulated",
}


# ============================================================
# 1. LOAD ORIGINAL TEST DATA
# ============================================================

def msis_load_test_xy(test_path):

    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}" for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}" for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    if (
        X.ndim != 2
        or X.shape[1] != LOOKBACK
        or Y.shape != (len(X), HORIZON)
    ):
        raise ValueError(
            f"Unexpected X/Y shapes: {X.shape}, {Y.shape}"
        )

    if not (
        np.all(np.isfinite(X))
        and np.all(np.isfinite(Y))
    ):
        raise ValueError(
            f"Non-finite test data: {test_path}"
        )

    return X, Y


# ============================================================
# 2. FIND QUANTILE FILES
# ============================================================

def msis_clean_garch_name(dataset_name):

    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace(
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            ""
        )
        .replace(" ", "_")
        .replace(".", "")
    )


def msis_find_quantile_file(
    model_name,
    model_root,
    asset_class,
    dataset_name,
):

    model_asset_dir = model_root / asset_class

    if model_name == "GARCH(1,1)":

        clean_name = msis_clean_garch_name(
            dataset_name
        )

        prefix = (
            f"{asset_class.lower()}_"
            f"{clean_name}_"
            f"garch_1_1_train_test"
        )

        expected = (
            model_asset_dir
            / f"{prefix}_quantiles.npz"
        )

        if expected.exists():
            return expected

        matches = list(
            model_asset_dir.glob(
                f"*{clean_name}*"
                f"garch_1_1_train_test*"
                f"quantiles.npz"
            )
        )

    else:

        suffix = MODEL_SUFFIXES[model_name]

        matches = list(
            model_asset_dir.glob(
                f"test_{dataset_name}"
                f"{suffix}_quantiles.npz"
            )
        )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"{model_name} / {dataset_name}: "
            f"expected one quantile file, "
            f"found {len(matches)}"
        )

    return matches[0]


# ============================================================
# 3. LOAD QUANTILES
# ============================================================

def msis_load_quantiles(path):

    with np.load(path) as data:

        Q = np.asarray(
            data["quantiles"],
            dtype=float
        )

        levels = np.asarray(
            data["quantile_levels"],
            dtype=float
        )

    return Q, levels


# ============================================================
# 4. CALCULATE 250-CONTEXT SCALE
# ============================================================

def msis_context_scales(X):

    scales = np.mean(
        np.abs(np.diff(X, axis=1)),
        axis=1
    )

    if (
        not np.all(np.isfinite(scales))
        or np.any(scales <= 0)
    ):
        raise ValueError(
            "Invalid 250-context scaling factor"
        )

    return scales


# ============================================================
# 5. EXTRACT 80% INTERVAL
# ============================================================

def msis_extract_interval(Y, Q, levels):

    if Q.ndim != 3:
        raise ValueError(
            f"Expected 3D quantiles, got {Q.shape}"
        )

    if Q.shape[:2] != Y.shape:
        raise ValueError(
            f"Forecast/target mismatch: "
            f"{Q.shape} vs {Y.shape}"
        )

    if Q.shape[2] != len(levels):
        raise ValueError(
            "Quantile-level dimension mismatch"
        )

    if not np.all(np.diff(levels) > 0):
        raise ValueError(
            "Quantile levels are not increasing"
        )

    if not np.all(np.isfinite(Q)):
        raise ValueError(
            "Non-finite predicted quantiles"
        )

    idx10 = np.flatnonzero(
        np.isclose(levels, 0.1)
    )

    idx90 = np.flatnonzero(
        np.isclose(levels, 0.9)
    )

    if len(idx10) != 1 or len(idx90) != 1:
        raise ValueError(
            "Missing or duplicated q0.1/q0.9"
        )

    lower = Q[:, :, idx10[0]]
    upper = Q[:, :, idx90[0]]

    if np.any(lower > upper):
        raise ValueError(
            "Crossed 80% interval bounds"
        )

    return lower, upper


# ============================================================
# 6. CALCULATE MSIS
# ============================================================

def calculate_msis_250(
    Y,
    lower,
    upper,
    scales,
    alpha=0.2,
):

    width = upper - lower

    lower_penalty = (
        (2.0 / alpha)
        * np.maximum(lower - Y, 0)
    )

    upper_penalty = (
        (2.0 / alpha)
        * np.maximum(Y - upper, 0)
    )

    interval_score = (
        width
        + lower_penalty
        + upper_penalty
    )

    scaled_score = (
        interval_score / scales[:, None]
    )

    msis_by_horizon = np.mean(
        scaled_score,
        axis=0
    )

    msis = float(
        np.mean(msis_by_horizon)
    )

    coverage = float(
        np.mean(
            (Y >= lower) & (Y <= upper)
        )
    )

    mean_width = float(
        np.mean(width)
    )

    return {
        "msis_250": msis,
        "msis_250_by_horizon": msis_by_horizon,
        "mean_interval_score": float(
            np.mean(interval_score)
        ),
        "coverage_80": coverage,
        "mean_width": mean_width,
    }


# ============================================================
# 7. RUN DAILY AND WEEKLY
# ============================================================

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 75)
    print(f"{frequency.upper()} MSIS")
    print("=" * 75)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    dataset_rows = []
    horizon_rows = []
    failures = []

    for model_name, model_folder in MODEL_DIRECTORIES.items():

        model_root = BASE_DIR / model_folder

        print(f"\nModel: {model_name}")

        for folder, asset_class in ASSET_ORDER.items():

            test_files = sorted(
                (DATA_DIR / folder).glob("test_*.csv")
            )

            for test_path in test_files:

                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )

                try:

                    X, Y = msis_load_test_xy(
                        test_path
                    )

                    quantile_path = (
                        msis_find_quantile_file(
                            model_name,
                            model_root,
                            folder,
                            dataset_name,
                        )
                    )

                    Q, levels = msis_load_quantiles(
                        quantile_path
                    )

                    lower, upper = (
                        msis_extract_interval(
                            Y, Q, levels
                        )
                    )

                    scales = msis_context_scales(X)

                    result = calculate_msis_250(
                        Y,
                        lower,
                        upper,
                        scales,
                        alpha=ALPHA,
                    )

                    dataset_rows.append({
                        "model": model_name,
                        "asset_class": asset_class,
                        "asset_group": folder,
                        "dataset": dataset_name,
                        "msis_250": result["msis_250"],
                        "mean_interval_score": result[
                            "mean_interval_score"
                        ],
                        "coverage_80": result[
                            "coverage_80"
                        ],
                        "mean_width": result[
                            "mean_width"
                        ],
                        "n_origins": len(Y),
                    })

                    for h in range(HORIZON):

                        horizon_rows.append({
                            "model": model_name,
                            "asset_class": asset_class,
                            "dataset": dataset_name,
                            "horizon": h + 1,
                            "msis_250": float(
                                result[
                                    "msis_250_by_horizon"
                                ][h]
                            ),
                        })

                except Exception as exc:

                    failures.append({
                        "model": model_name,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "error": str(exc),
                    })

                    print(
                        f"ERROR: {model_name} / "
                        f"{dataset_name}: {exc}"
                    )

    # ========================================================
    # 8. SAVE DATASET-LEVEL RESULTS
    # ========================================================

    dataset_df = pd.DataFrame(dataset_rows)
    horizon_df = pd.DataFrame(horizon_rows)
    failures_df = pd.DataFrame(failures)

    if dataset_df.empty:
        raise RuntimeError(
            f"No successful MSIS results for {frequency}"
        )

    dataset_path = (
        BASE_DIR
        / f"msis_250_by_dataset_{frequency}_with_timestral.csv"
    )

    horizon_path = (
        BASE_DIR
        / f"msis_250_by_horizon_{frequency}_with_timestral.csv"
    )

    dataset_df.to_csv(
        dataset_path,
        index=False
    )

    horizon_df.to_csv(
        horizon_path,
        index=False
    )

    # ========================================================
    # 9. MODEL-LEVEL SUMMARY
    # ========================================================

    dataset_df["msis_rank"] = (
        dataset_df
        .groupby(["asset_group", "dataset"])[
            "msis_250"
        ]
        .rank(
            method="average",
            ascending=True
        )
    )

    summary_df = (
        dataset_df
        .groupby("model")
        .agg(
            datasets=("dataset", "count"),
            mean_msis=("msis_250", "mean"),
            median_msis=("msis_250", "median"),
            mean_rank=("msis_rank", "mean"),
            mean_coverage=("coverage_80", "mean"),
            mean_width=("mean_width", "mean"),
        )
        .reset_index()
        .sort_values("mean_rank")
    )

    summary_path = (
        BASE_DIR
        / f"msis_250_summary_{frequency}_with_timestral.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    # ========================================================
    # 10. REPORT
    # ========================================================

    expected_datasets = sum(
        len(list(
            (DATA_DIR / folder).glob("test_*.csv")
        ))
        for folder in ASSET_ORDER
    )

    expected_rows = (
        expected_datasets * len(MODEL_DIRECTORIES)
    )

    print()
    print(f"Expected rows: {expected_rows}")
    print(f"Successful rows: {len(dataset_df)}")
    print(f"Failures: {len(failures_df)}")

    print()
    print(
        summary_df.round(5).to_string(
            index=False
        )
    )

    print()
    print(f"Saved: {dataset_path}")
    print(f"Saved: {horizon_path}")
    print(f"Saved: {summary_path}")

    if not failures_df.empty:

        failure_path = (
            BASE_DIR
            / f"msis_250_failures_{frequency}_with_timestral.csv"
        )

        failures_df.to_csv(
            failure_path,
            index=False
        )

        print(f"Failures saved: {failure_path}")

        raise RuntimeError(
            f"{frequency}: MSIS evaluation incomplete. "
            "Review the failure report."
        )

### Deconstruction

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# MSIS DECOMPOSITION + ROLLING EMPIRICAL BASELINE
#
# Reuses functions/configuration from the successful MSIS cell:
#
#   msis_load_test_xy()
#   msis_find_quantile_file()
#   msis_load_quantiles()
#   msis_extract_interval()
#   msis_context_scales()
#   MODEL_DIRECTORIES
#   ASSET_ORDER
#   PROJECT_DIR
# ============================================================

ALPHA = 0.20
HORIZON = 20


# ============================================================
# 1. DECOMPOSE MSIS
# ============================================================

def decompose_msis_250(
    Y,
    lower,
    upper,
    scales,
    alpha=0.20,
):

    Y = np.asarray(Y, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    scales = np.asarray(scales, dtype=float)

    if not (
        Y.shape == lower.shape == upper.shape
    ):
        raise ValueError(
            "Target and interval shapes do not match."
        )

    if scales.shape != (Y.shape[0],):
        raise ValueError(
            "Scaling factors do not match origins."
        )

    if not (
        np.all(np.isfinite(Y))
        and np.all(np.isfinite(lower))
        and np.all(np.isfinite(upper))
        and np.all(np.isfinite(scales))
    ):
        raise ValueError(
            "Non-finite values detected."
        )

    if np.any(scales <= 0):
        raise ValueError(
            "Invalid context scale."
        )

    if np.any(lower > upper):
        raise ValueError(
            "Crossed interval bounds."
        )

    # --------------------------------------------------------
    # UNSCALED COMPONENTS
    # --------------------------------------------------------

    width = upper - lower

    lower_penalty = (
        2.0 / alpha
    ) * np.maximum(
        lower - Y,
        0.0
    )

    upper_penalty = (
        2.0 / alpha
    ) * np.maximum(
        Y - upper,
        0.0
    )

    # --------------------------------------------------------
    # SCALE EACH ORIGIN BEFORE AVERAGING
    # --------------------------------------------------------

    scaled_width = (
        width / scales[:, None]
    )

    scaled_lower_penalty = (
        lower_penalty / scales[:, None]
    )

    scaled_upper_penalty = (
        upper_penalty / scales[:, None]
    )

    scaled_total = (
        scaled_width
        + scaled_lower_penalty
        + scaled_upper_penalty
    )

    # --------------------------------------------------------
    # AGGREGATE
    # --------------------------------------------------------

    mean_width_component = float(
        np.mean(scaled_width)
    )

    mean_lower_component = float(
        np.mean(scaled_lower_penalty)
    )

    mean_upper_component = float(
        np.mean(scaled_upper_penalty)
    )

    total_msis = float(
        np.mean(scaled_total)
    )

    # --------------------------------------------------------
    # DIAGNOSTICS
    # --------------------------------------------------------

    below = Y < lower
    above = Y > upper

    coverage = float(
        np.mean(
            (Y >= lower) & (Y <= upper)
        )
    )

    return {
        "msis_250": total_msis,

        "scaled_width": mean_width_component,

        "lower_penalty": mean_lower_component,

        "upper_penalty": mean_upper_component,

        "coverage_80": coverage,

        "below_rate": float(np.mean(below)),

        "above_rate": float(np.mean(above)),

        "mean_width": float(np.mean(width)),

        "n_origins": len(Y),
    }


# ============================================================
# 2. ROLLING EMPIRICAL INTERVAL
# ============================================================

def empirical_interval_forecast(X):

    X = np.asarray(X, dtype=float)

    if X.ndim != 2 or X.shape[1] != 250:
        raise ValueError(
            f"Unexpected context shape: {X.shape}"
        )

    # One lower/upper quantile per origin
    q10 = np.quantile(
        X,
        0.10,
        axis=1
    )

    q90 = np.quantile(
        X,
        0.90,
        axis=1
    )

    # Repeat across all 20 horizons
    lower = np.repeat(
        q10[:, None],
        HORIZON,
        axis=1
    )

    upper = np.repeat(
        q90[:, None],
        HORIZON,
        axis=1
    )

    return lower, upper


# ============================================================
# 3. PROCESS DAILY AND WEEKLY
# ============================================================

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 80)
    print(f"{frequency.upper()} MSIS DECOMPOSITION")
    print("=" * 80)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    rows = []
    failures = []

    for folder, asset_class in ASSET_ORDER.items():

        test_files = sorted(
            (DATA_DIR / folder).glob("test_*.csv")
        )

        for test_path in test_files:

            dataset_name = (
                test_path.stem
                .removeprefix("test_")
            )

            try:

                X, Y = msis_load_test_xy(
                    test_path
                )

                scales = msis_context_scales(X)

                # --------------------------------------------
                # EMPIRICAL DISTRIBUTION BASELINE
                # --------------------------------------------

                empirical_lower, empirical_upper = (
                    empirical_interval_forecast(X)
                )

                result = decompose_msis_250(
                    Y,
                    empirical_lower,
                    empirical_upper,
                    scales,
                )

                rows.append({
                    "frequency": frequency,
                    "model": "Empirical Distribution",
                    "asset_class": asset_class,
                    "asset_group": folder,
                    "dataset": dataset_name,
                    **result,
                })

                # --------------------------------------------
                # ALL EXISTING MODELS
                # --------------------------------------------

                for model_name, model_folder in (
                    MODEL_DIRECTORIES.items()
                ):

                    model_root = (
                        BASE_DIR / model_folder
                    )

                    quantile_path = (
                        msis_find_quantile_file(
                            model_name,
                            model_root,
                            folder,
                            dataset_name,
                        )
                    )

                    Q, levels = msis_load_quantiles(
                        quantile_path
                    )

                    lower, upper = (
                        msis_extract_interval(
                            Y,
                            Q,
                            levels,
                        )
                    )

                    result = decompose_msis_250(
                        Y,
                        lower,
                        upper,
                        scales,
                    )

                    rows.append({
                        "frequency": frequency,
                        "model": model_name,
                        "asset_class": asset_class,
                        "asset_group": folder,
                        "dataset": dataset_name,
                        **result,
                    })

            except Exception as exc:

                failures.append({
                    "asset_class": asset_class,
                    "dataset": dataset_name,
                    "error": str(exc),
                })

                print(
                    f"ERROR: {dataset_name}: {exc}"
                )

    # ========================================================
    # 4. RESULTS DATAFRAME
    # ========================================================

    df = pd.DataFrame(rows)

    if failures:
        print(pd.DataFrame(failures).to_string(index=False))
        raise RuntimeError(
            f"{frequency}: decomposition incomplete. "
            "No outputs were saved."
        )

    expected_datasets = sum(
        len(list(
            (DATA_DIR / folder).glob("test_*.csv")
        ))
        for folder in ASSET_ORDER
    )

    expected_models = (
        len(MODEL_DIRECTORIES) + 1
    )

    expected_rows = (
        expected_datasets * expected_models
    )

    if len(df) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} rows, got {len(df)}"
        )

    if df.duplicated(
        ["asset_group", "dataset", "model"]
    ).any():
        raise RuntimeError(
            "Duplicate dataset-model rows detected."
        )

    # ========================================================
    # 5. VERIFY DECOMPOSITION
    # ========================================================

    reconstructed = (
        df["scaled_width"]
        + df["lower_penalty"]
        + df["upper_penalty"]
    )

    if not np.allclose(
        df["msis_250"],
        reconstructed,
        rtol=1e-10,
        atol=1e-10,
    ):
        raise RuntimeError(
            "MSIS decomposition does not sum correctly."
        )

    # ========================================================
    # 6. COMPARE AGAINST PREVIOUS MSIS RESULTS
    # ========================================================

    previous_path = (
        BASE_DIR
        / f"msis_250_by_dataset_{frequency}_with_timestral.csv"
    )

    if previous_path.exists():

        previous = pd.read_csv(previous_path)

        check = df.merge(
            previous[
                [
                    "model",
                    "asset_group",
                    "dataset",
                    "msis_250",
                ]
            ],
            on=[
                "model",
                "asset_group",
                "dataset",
            ],
            how="inner",
            suffixes=("_new", "_old"),
            validate="one_to_one",
        )

        if len(check) != len(previous):
            raise RuntimeError(
                "Previous MSIS results do not align "
                "with the new decomposition."
            )

        if not np.allclose(
            check["msis_250_new"],
            check["msis_250_old"],
            rtol=1e-8,
            atol=1e-8,
        ):
            raise RuntimeError(
                "New MSIS differs from previous results. "
                "Check forecast loading and alignment."
            )

        print(
            "Previous MSIS results reproduced successfully."
        )

    # ========================================================
    # 7. RANK MODELS WITHIN DATASETS
    # ========================================================

    df["rank"] = (
        df.groupby(
            ["asset_group", "dataset"]
        )["msis_250"]
        .rank(
            method="average",
            ascending=True
        )
    )

    # ========================================================
    # 8. MODEL SUMMARY
    # ========================================================

    summary = (
        df.groupby("model")
        .agg(
            datasets=("dataset", "count"),

            mean_msis=("msis_250", "mean"),

            median_msis=("msis_250", "median"),

            mean_rank=("rank", "mean"),

            scaled_width=("scaled_width", "mean"),

            lower_penalty=("lower_penalty", "mean"),

            upper_penalty=("upper_penalty", "mean"),

            mean_coverage=("coverage_80", "mean"),

            below_rate=("below_rate", "mean"),

            above_rate=("above_rate", "mean"),

            mean_width=("mean_width", "mean"),
        )
        .reset_index()
        .sort_values("mean_rank")
    )

    # ========================================================
    # 9. RELATIVE IMPROVEMENT OVER EMPIRICAL
    # ========================================================

    empirical = (
        df[
            df["model"] == "Empirical Distribution"
        ][
            [
                "asset_group",
                "dataset",
                "msis_250",
            ]
        ]
        .rename(
            columns={
                "msis_250": "empirical_msis"
            }
        )
    )

    df = df.merge(
        empirical,
        on=["asset_group", "dataset"],
        how="left",
        validate="many_to_one",
    )

    df["improvement_over_empirical_pct"] = (
        100.0
        * (
            1.0
            - df["msis_250"]
            / df["empirical_msis"]
        )
    )

    improvement_summary = (
        df.groupby("model")
        .agg(
            mean_improvement_pct=(
                "improvement_over_empirical_pct",
                "mean",
            ),

            median_improvement_pct=(
                "improvement_over_empirical_pct",
                "median",
            ),

            beats_empirical_pct=(
                "improvement_over_empirical_pct",
                lambda x: 100.0 * (x > 0).mean(),
            ),
        )
        .reset_index()
    )

    summary = summary.merge(
        improvement_summary,
        on="model",
        validate="one_to_one",
    )

    summary = summary.sort_values(
        "mean_rank"
    )

    # ========================================================
    # 10. SAVE RESULTS
    # ========================================================

    dataset_path = (
        BASE_DIR
        / f"msis_250_decomposition_{frequency}_with_timestral.csv"
    )

    summary_path = (
        BASE_DIR
        / f"msis_250_with_empirical_{frequency}_with_timestral.csv"
    )

    df.to_csv(
        dataset_path,
        index=False
    )

    summary.to_csv(
        summary_path,
        index=False
    )

    # ========================================================
    # 11. PRINT RESULTS
    # ========================================================

    print()
    print(f"Datasets: {expected_datasets}")
    print(f"Methods: {expected_models}")
    print(f"Dataset-method rows: {len(df)}")

    display_cols = [
        "model",
        "mean_msis",
        "mean_rank",
        "scaled_width",
        "lower_penalty",
        "upper_penalty",
        "mean_coverage",
        "beats_empirical_pct",
    ]

    print()
    print(
        summary[display_cols]
        .round(5)
        .to_string(index=False)
    )

    print()
    print(f"Saved: {dataset_path}")
    print(f"Saved: {summary_path}")

    # ========================================================
    # 12. MOIRAI VERSUS MOIRAI-MOE
    # ========================================================

    print()
    print("MOIRAI VS MOIRAI-MOE")

    comparison = summary[
        summary["model"].isin(
            ["Moirai", "Moirai-MoE"]
        )
    ][
        [
            "model",
            "mean_msis",
            "scaled_width",
            "lower_penalty",
            "upper_penalty",
            "mean_coverage",
            "below_rate",
            "above_rate",
        ]
    ]

    print(
        comparison
        .round(6)
        .to_string(index=False)
    )

# Statistical Significance

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 20,
})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def significance_analysis(csv_path, metric_col, output_prefix, freq):
    # ------------------------------------------------------------
    # 1. Load data
    # ------------------------------------------------------------
    df = pd.read_csv(csv_path)

    print(f"\n=== {output_prefix.upper()} ===")
    print(f"Datasets: {df['dataset'].nunique()}")
    print(f"Models: {df['model'].nunique()}")
    print(df['model'].unique())

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index='dataset',
        columns='model',
        values=metric_col
    )

    # Friedman requires every dataset to contain every model.
    complete = pivot.dropna().rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    print(f"Complete datasets used: {len(complete)}")

    if len(complete) != len(pivot):
        print(
            f"WARNING: dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank models within every dataset
    #    1 = lowest metric = best rank
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=True,
        method='average'
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[complete[col].values for col in complete.columns]
    )

    print(
        f"\nFriedman chi-square = {stat:.4f}, "
        f"p = {p_value:.4e}"
    )

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc test
    # ------------------------------------------------------------
    if p_value < 0.05:
        p_matrix = sp.posthoc_nemenyi_friedman(complete)

        print("\nNemenyi pairwise p-values:")
        print(p_matrix)

        # Save pairwise p-values
        p_matrix.to_csv(
            f"{output_prefix}_nemenyi_pvalues_with_timestral.csv"
        )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)
            
        if output_prefix == "mase":
            title = "Persistence MAE"
        else:
            title = "CRPS"
            
        plt.title(
            f"{freq.capitalize()} {title} Critical Difference Diagram\n"
            f"Friedman p = {p_value:.2e}"
        )

        plt.tight_layout()
        plt.savefig(
            f"{freq}_{output_prefix}_critical_difference_with_timestral.png",
            dpi=300,
            bbox_inches='tight'
        )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nFriedman test was not significant; "
            "Nemenyi post-hoc comparisons are not interpreted."
        )

    # ------------------------------------------------------------
    # 7. Save ranks
    # ------------------------------------------------------------
    avg_rank_df = (
        avg_ranks
        .rename('average_rank')
        .reset_index()
    )

    avg_rank_df.to_csv(
        f"{freq}_{output_prefix}_average_ranks_with_timestral.csv",
        index=False
    )

    return complete, ranks, avg_ranks, stat, p_value, p_matrix

for freq in ["daily", "weekly"]:
    # ================================================================
    # MASE
    # ================================================================
    mase_results = significance_analysis(
        csv_path=f"BForTFin_results/standardised/output_{freq}/all_models_mase_by_dataset_with_timestral.csv",
        metric_col="mean_mase",
        output_prefix="mase",
        freq=freq,
    )
    
    
    # ================================================================
    # CRPS
    # ================================================================
    crps_results = significance_analysis(
        csv_path=f"BForTFin_results/standardised/output_{freq}/all_models_crps_by_dataset_with_timestral.csv",
        metric_col="mean_qcrps",
        output_prefix="crps",
        freq=freq,
    )

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def overall_significance_analysis(
    csv_path,
    metric_col,
    metric_name,
    freq,
    higher_is_better=False,
    output_prefix=None
):
    df = pd.read_csv(csv_path)

    print("\n" + "=" * 80)
    print(f"OVERALL {metric_name.upper()} SIGNIFICANCE ANALYSIS")
    print("=" * 80)

    print("Datasets:", df["dataset"].nunique())
    print("Models:", df["model"].nunique())
    print("Models:", list(df["model"].unique()))

    # ------------------------------------------------------------
    # 1. Check duplicates
    # ------------------------------------------------------------
    duplicates = df.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nDuplicate dataset/model rows found:")
        print(
            df.loc[
                duplicates,
                ["dataset", "model", metric_col]
            ]
        )
        raise ValueError("Duplicate dataset/model entries detected.")

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index="dataset",
        columns="model",
        values=metric_col
    )

    print("\nOriginal matrix shape:", pivot.shape)

    # Friedman needs complete blocks
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    print(
        f"Complete datasets used: {len(complete)} / {len(pivot)}"
    )

    if len(complete) != len(pivot):
        print(
            f"Dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank within each dataset
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=not higher_is_better,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test:")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc
    # ------------------------------------------------------------
    if p_value < 0.05:

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-values:")
        print(p_matrix)

        if output_prefix is not None:
            p_matrix.to_csv(
                f"{freq}_{output_prefix}_nemenyi_pvalues_with_timestral.csv"
            )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        # thicker lines + larger text
        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)

        for text in ax.texts:
            text.set_fontsize(17)

        ax.set_title(
            f"{freq.capitalize()} {metric_name} Critical Difference Diagram\n"
            f"Friedman p = {p_value:.2e}",
            fontsize=18
        )

        plt.tight_layout()

        if output_prefix is not None:
            plt.savefig(
                f"{freq}_{output_prefix}_critical_difference_with_timestral.png",
                dpi=300,
                bbox_inches="tight"
            )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nNo significant overall difference detected."
        )

    # ------------------------------------------------------------
    # 7. Save average ranks
    # ------------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value
    rank_df["n_datasets"] = len(complete)

    if output_prefix is not None:
        rank_df.to_csv(
            f"{freq}_{output_prefix}_average_ranks_with_timestral.csv",
            index=False
        )

    return {
        "pivot": complete,
        "ranks": ranks,
        "average_ranks": avg_ranks,
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "nemenyi": p_matrix
    }
    
for freq in ["daily", "weekly"]:
    DIR = Path(rf"BForTFin_results/standardised/output_{freq}")
    
    rmse_results = overall_significance_analysis(
        csv_path=DIR / "rmse_summary_all_models_with_timestral.csv",
        metric_col="rmse",
        metric_name="RMSE",
        freq=freq,
        higher_is_better=False,
        output_prefix="rmse_overall"
    )
    
    qss_results = overall_significance_analysis(
        csv_path=DIR / "qss_summary_all_models_with_timestral.csv",
        metric_col="qss_0.1",
        metric_name="QSS",
        freq=freq,
        higher_is_better=True,
        output_prefix="qss_overall"
    )

In [ ]:
from pathlib import Path
import pandas as pd
import re


# ============================================================
# CONFIG
# ============================================================

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_NAMES = {
    "arima_standardized": "ARIMA",
    "lightgbm_standardized_train_test": "LightGBM",
    "lstm_standardized_train_test": "LSTM",
    "sundial_standardized_train_test": "Sundial",
    "lag-llama_standardized_train_test": "Lag-Llama",
    "chronos_standardized_train_test": "Chronos-T5", 
    "toto_standardized_train_test": "Toto", 
    "timesfm_standardized_train_test": "TimesFM",
    "timestral_standardized_train_test": "Timestral",
    "moirai_standardized_train_test": "Moirai",
    "moirai_moe_standardized_train_test": "Moirai-MoE",
    "garch_standardized_train_test": "GARCH(1,1)",
    "garch_t_1_1_standardized_train_test": "Student-t GARCH",
}


ANCHOR = "_no_preprocess_lookback-250_horizon-20_slide-1_"


# ============================================================
# HELPERS
# ============================================================

def extract_dataset_name(filename):
    """
    Example:
    test_USD_EUR Historical Data_no_preprocess_lookback-250_horizon-20_slide-1_...
    
    -> USD_EUR Historical Data
    """
    stem = Path(filename).stem

    if not stem.startswith("test_"):
        #raise ValueError(f"Unexpected filename: {filename}")
        anchor = "_garch_t_1_1_simulated_train_test_"
        before_anchor = stem.split(anchor, 1)[0]
        res = before_anchor
    else:
        before_anchor = stem.split(ANCHOR, 1)[0]
        res = before_anchor[len("test_"):]

    return res


def read_metric_by_horizon(file_path, metric):
    """
    Reads a *_mase_by_horizon.csv or *_crps_by_horizon.csv
    and returns:

        horizon | value

    Tries to handle slightly different column names automatically.
    """
    df = pd.read_csv(file_path)

    # Remove accidental saved dataframe-index columns
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

    # --------------------------------------------------------
    # Find horizon column
    # --------------------------------------------------------
    horizon_candidates = [
        c for c in df.columns
        if str(c).lower() in {
            "horizon",
            "h",
            "forecast_horizon",
            "forecast_step",
            "step"
        }
    ]

    if horizon_candidates:
        horizon_col = horizon_candidates[0]
        horizons = df[horizon_col].to_numpy()

    else:
        # If there is no explicit horizon column, assume rows
        # correspond to h = 0, ..., 19
        horizons = range(len(df))

    # --------------------------------------------------------
    # Find metric column
    # --------------------------------------------------------
    metric_lower = metric.lower()

    metric_candidates = [
        c for c in df.columns
        if metric_lower in str(c).lower()
        and str(c).lower() not in {
            "horizon",
            "forecast_horizon"
        }
    ]

    if len(metric_candidates) == 1:
        value_col = metric_candidates[0]

    elif len(metric_candidates) > 1:
        raise ValueError(
            f"Multiple possible {metric} columns in {file_path}:\n"
            f"{metric_candidates}"
        )

    else:
        # Fallback: find numeric columns that aren't horizon columns
        numeric_cols = df.select_dtypes(include="number").columns.tolist()

        if horizon_candidates:
            numeric_cols = [
                c for c in numeric_cols
                if c != horizon_candidates[0]
            ]

        if len(numeric_cols) != 1:
            raise ValueError(
                f"Could not uniquely identify metric column in:\n"
                f"{file_path}\n"
                f"Columns: {list(df.columns)}"
            )

        value_col = numeric_cols[0]

    out = pd.DataFrame({
        "horizon": list(horizons),
        metric: df[value_col].to_numpy()
    })

    return out


def dataset_match_key(name):
    """
    Creates a temporary key ONLY for matching dataset names.

    Examples:
        'AUD_USD Historical Data' -> 'audusd'
        'aud_usd'                 -> 'audusd'
        'AUD-USD'                 -> 'audusd'
    """
    name = str(name).lower().strip()

    # Remove common suffix appearing in the normal filenames
    name = name.replace("historical data", "")

    # Ignore spaces, underscores, hyphens, punctuation, etc.
    name = re.sub(r"[^a-z0-9]", "", name)

    return name

def build_canonical_dataset_lookup(metric):

    lookup = {}

    suffix = f"_{metric}_by_horizon.csv"

    for model_dir, model in MODEL_NAMES.items():

        # Don't use GARCH filenames to establish canonical names
        if model.lower() == "garch(1,1)" or model.lower() == "student-t garch":
            continue

        for asset_folder, asset_class in ASSET_FOLDERS.items():

            folder = RESULTS_DIR / model_dir / asset_folder

            if not folder.exists():
                continue

            for file_path in folder.glob(f"*{suffix}"):

                try:
                    dataset = extract_dataset_name(file_path.name)

                    key = dataset_match_key(dataset)

                    if key in lookup and lookup[key] != dataset:
                        print(
                            f"WARNING: duplicate matching key '{key}': "
                            f"'{lookup[key]}' vs '{dataset}'"
                        )
                    else:
                        lookup[key] = dataset

                except Exception:
                    pass

    return lookup

# ============================================================
# COLLECT ONE METRIC
# ============================================================

def collect_metric(metric):
    records = []

    canonical_lookup = build_canonical_dataset_lookup(metric)

    print(
        f"Found {len(canonical_lookup)} canonical dataset names "
        f"for {metric.upper()}"
    )

    print(canonical_lookup)

    suffix = f"_{metric}_by_horizon.csv"

   

    print(f"\nCollecting {metric.upper()}")
    print("=" * 70)

    for model_dir, model in MODEL_NAMES.items():

        #model = MODEL_NAMES[model_dir.name]

        # Only keep GARCH(1,1) files from the GARCH directory
        is_garch_dir = model == "GARCH(1,1)"

        for asset_folder, asset_class in ASSET_FOLDERS.items():

            folder = RESULTS_DIR / model_dir / asset_folder

            if not folder.exists():
                continue

            files = list(folder.glob(f"*{suffix}"))

            for file_path in files:

                # Skip GARCH variants other than (1,1)
                if is_garch_dir and "_garch_1_1_" not in file_path.name:
                    continue
            
                try:
                    if is_garch_dir:

                        stem = file_path.stem.lower()
                    
                        # Example:
                        # g7_aud_usd_garch_1_1_train_test_crps_by_horizon
                        #
                        # -> g7_aud_usd
                        raw_dataset = stem.split("_garch_1_1_")[0]
                    
                        # Remove asset prefix:
                        # g7_aud_usd -> aud_usd
                        prefix = asset_folder.lower() + "_"
                    
                        if raw_dataset.startswith(prefix):
                            raw_dataset = raw_dataset[len(prefix):]
                    
                        # Turn into comparison key
                        key = dataset_match_key(raw_dataset)
                    
                        # Retrieve canonical dataset name
                        if key not in canonical_lookup:
                            print(
                                f"WARNING: Could not match GARCH dataset "
                                f"'{raw_dataset}' ({file_path.name})"
                            )
                            continue
                    
                        dataset = canonical_lookup[key]
                    
                    else:
                        dataset = extract_dataset_name(file_path.name)
            
                    horizon_df = read_metric_by_horizon(
                        file_path,
                        metric
                    )
            
                    for _, row in horizon_df.iterrows():
                        records.append({
                            "dataset": dataset,
                            "asset_class": asset_class,
                            "asset_folder": asset_folder,
                            "model": model,
                            "horizon": int(row["horizon"] - 1),
                            metric: row[metric],
                        })
            
                except Exception as e:
                    print(f"\nERROR reading:")
                    print(file_path)
                    print(e)
                

    result = pd.DataFrame(records)

    if result.empty:
        raise RuntimeError(
            f"No {metric} horizon files were found."
        )

    result = result.sort_values(
        ["asset_class", "dataset", "model", "horizon"]
    ).reset_index(drop=True)

    return result

for freq in ["daily", "weekly"]:
    RESULTS_DIR = Path(rf"BForTFin_results/standardised/output_{freq}")
    # ============================================================
    # BUILD BOTH MASTER SUMMARIES
    # ============================================================
    
    mase = collect_metric("mase")
    crps = collect_metric("crps")
    
    
    # ============================================================
    # SANITY CHECKS
    # ============================================================
    
    print(f"\n{freq.capitalize()} MASE summary")
    print(mase.shape)
    print("Datasets:", mase["dataset"].nunique())
    print("Models:", mase["model"].nunique())
    print("Horizons:", sorted(mase["horizon"].unique()))
    
    print(f"\n{freq.capitalize()} CRPS summary")
    print(crps.shape)
    print("Datasets:", crps["dataset"].nunique())
    print("Models:", crps["model"].nunique())
    print("Horizons:", sorted(crps["horizon"].unique()))
    
    garch_crps = crps[
        crps["model"] == "GARCH(1,1)"
    ]
    
    print(f"\n{freq.capitalize()} GARCH CHECK")
    print("-" * 50)
    print("GARCH datasets:", garch_crps["dataset"].nunique())
    print("GARCH rows:", len(garch_crps))
    print("GARCH horizons:", sorted(garch_crps["horizon"].unique()))
    
    all_datasets = set(
        crps.loc[
            crps["model"] != "GARCH(1,1)",
            "dataset"
        ].unique()
    )
    
    garch_datasets = set(
        crps.loc[
            crps["model"] == "GARCH(1,1)",
            "dataset"
        ].unique()
    )
    
    missing_garch = sorted(all_datasets - garch_datasets)
    
    print(
        f"\n {freq.capitalize()} Datasets without GARCH: {len(missing_garch)}"
    )
    
    for ds in missing_garch:
        print("  ", ds)
    
    
    # ============================================================
    # SAVE MASTER FILES
    # ============================================================
    
    mase.to_csv(
        RESULTS_DIR / "all_models_mase_by_dataset_and_horizon_with_timestral.csv",
        index=False
    )
    
    crps.to_csv(
        RESULTS_DIR / "all_models_crps_by_dataset_and_horizon_with_timestral.csv",
        index=False
    )
    
    
    # ============================================================
    # SAVE JUST h = 0, 4, 19
    # ============================================================
    
    SELECTED_HORIZONS = [0, 4, 19]
    
    mase_selected = mase[
        mase["horizon"].isin(SELECTED_HORIZONS)
    ].copy()
    
    crps_selected = crps[
        crps["horizon"].isin(SELECTED_HORIZONS)
    ].copy()
    
    mase_selected.to_csv(
        RESULTS_DIR / "all_models_mase_h0_h4_h19_with_timestral.csv",
        index=False
    )
    
    crps_selected.to_csv(
        RESULTS_DIR / "all_models_crps_h0_h4_h19_with_timestral.csv",
        index=False
    )
    
    print("\nDone ❤️")

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIG
# ============================================================

HORIZONS = [0, 4, 19]
ALPHA = 0.05


# ============================================================
# ONE HORIZON / ONE METRIC
# ============================================================

def analyse_horizon(
    df,
    metric,
    horizon,
    output_dir
):
    print("\n" + "=" * 80)
    print(f"{freq.capitalize()} {metric.upper()} — HORIZON h={horizon}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Select this horizon
    # --------------------------------------------------------
    df_h = df[df["horizon"] == horizon].copy()

    print("Rows:", len(df_h))
    print("Datasets:", df_h["dataset"].nunique())
    print("Models:", df_h["model"].nunique())

    # --------------------------------------------------------
    # 2. Check for duplicate dataset/model entries
    # --------------------------------------------------------
    duplicates = df_h.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nWARNING: duplicate dataset/model rows found:")
        print(
            df_h.loc[
                duplicates,
                ["dataset", "model", metric]
            ].sort_values(["dataset", "model"])
        )

        raise ValueError(
            f"Duplicate dataset/model entries found for "
            f"{metric}, h={horizon}."
        )

    # --------------------------------------------------------
    # 3. Create complete-block matrix
    #
    # rows    = datasets
    # columns = models
    # values  = metric
    # --------------------------------------------------------
    pivot = df_h.pivot(
        index="dataset",
        columns="model",
        values=metric
    )

    print("\nOriginal pivot shape:", pivot.shape)

    # Show missing values before dropping anything
    missing_counts = pivot.isna().sum()

    if missing_counts.sum() > 0:
        print("\nMissing values by model:")
        print(
            missing_counts[
                missing_counts > 0
            ].sort_values(ascending=False)
        )

    # Friedman needs a complete block:
    # every included dataset must have every included model.
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    dropped = len(pivot) - len(complete)

    print(
        f"\nComplete datasets used: {len(complete)} / {len(pivot)}"
    )

    if dropped > 0:
        print(
            f"Dropped {dropped} datasets because at least "
            f"one model was missing."
        )

    if len(complete) < 2:
        raise ValueError(
            "Not enough complete datasets for significance testing."
        )

    # --------------------------------------------------------
    # 4. Rank models separately within each dataset
    #
    # Lower MASE / CRPS = better
    # therefore rank 1 = best
    # --------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=True,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # --------------------------------------------------------
    # 5. Friedman omnibus test
    # --------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # --------------------------------------------------------
    # 6. Save average ranks
    # --------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["horizon"] = horizon
    rank_df["n_datasets"] = len(complete)
    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value

    rank_path = (
        output_dir /
        f"{metric}_h{horizon}_average_ranks_with_timestral.csv"
    )

    rank_df.to_csv(rank_path, index=False)

    # --------------------------------------------------------
    # 7. Nemenyi post-hoc
    # --------------------------------------------------------
    p_matrix = None

    if p_value < ALPHA:

        print(
            "\nSignificant overall difference detected. "
            "Running Nemenyi post-hoc test..."
        )

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-value matrix:")
        print(p_matrix)

        p_matrix.to_csv(
            output_dir /
            f"{metric}_h{horizon}_nemenyi_pvalues_with_timestral.csv"
        )

        # ----------------------------------------------------
        # 8. Critical-difference diagram
        # ----------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)

        # h=0 is forecast step 1, etc.
        forecast_step = horizon + 1

        if metric == "mase":
            metric_name = "Persistence MAE"
        else:
            metric_name = "CRPS"
            
        ax.set_title(
            f"{freq.capitalize()} {metric_name}"
            f"(h={horizon+1})\n"
            f"Friedman p = {p_value:.2e}"
        )

        plt.tight_layout()

        figure_path = (
            output_dir /
            f"{metric}_h{horizon}_critical_difference_with_timestral.png"
        )

        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:
        print(
            "\nNo significant overall difference at "
            f"alpha={ALPHA}; Nemenyi results are not interpreted."
        )

    return {
        "metric": metric,
        "horizon": horizon,
        "n_datasets": len(complete),
        "n_models": complete.shape[1],
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "average_ranks": avg_ranks,
        "nemenyi": p_matrix,
    }

for freq in ["daily", "weekly"]:
    RESULTS_DIR = Path(
        r"."
        rf"\BForTFin_results\standardised\output_{freq}"
    )

    MASE_FILE = RESULTS_DIR / "all_models_mase_by_dataset_and_horizon_with_timestral.csv"
    CRPS_FILE = RESULTS_DIR / "all_models_crps_by_dataset_and_horizon_with_timestral.csv"
    
    # ============================================================
    # RUN ALL THREE HORIZONS
    # ============================================================
    
    mase = pd.read_csv(MASE_FILE)
    crps = pd.read_csv(CRPS_FILE)
    
    OUTPUT_DIR = RESULTS_DIR / "statistical_significance"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    all_results = []
    
    
    for metric, df in [
        ("mase", mase),
        ("crps", crps),
    ]:
    
        for h in HORIZONS:
    
            result = analyse_horizon(
                df=df,
                metric=metric,
                horizon=h,
                output_dir=OUTPUT_DIR
            )
    
            all_results.append({
                "metric": metric,
                "horizon": h,
                "forecast_step": h + 1,
                "n_datasets": result["n_datasets"],
                "n_models": result["n_models"],
                "friedman_statistic": result["friedman_statistic"],
                "friedman_pvalue": result["friedman_pvalue"],
            })
    
    
    # ============================================================
    # SAVE OVERALL FRIEDMAN SUMMARY
    # ============================================================
    
    friedman_summary = pd.DataFrame(all_results)
    
    friedman_summary.to_csv(
        OUTPUT_DIR / "friedman_horizon_summary_with_timestral.csv",
        index=False
    )
    
    print(f"\n\nFINAL {freq.upper()} FRIEDMAN SUMMARY")
    print("=" * 80)
    print(friedman_summary)

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIG
# ============================================================

HORIZONS = [1, 5, 20]
ALPHA = 0.05


# ============================================================
# ONE HORIZON / ONE METRIC
# ============================================================

def analyse_horizon(
    df,
    metric,
    horizon,
    output_dir
):
    print("\n" + "=" * 80)
    print(f"{freq.capitalize()} {metric.upper()} — HORIZON h={horizon}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Select this horizon
    # --------------------------------------------------------
    df_h = df[df["horizon"] == horizon].copy()

    print("Rows:", len(df_h))
    print("Datasets:", df_h["dataset"].nunique())
    print("Models:", df_h["model"].nunique())

    # --------------------------------------------------------
    # 2. Check for duplicate dataset/model entries
    # --------------------------------------------------------
    duplicates = df_h.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nWARNING: duplicate dataset/model rows found:")
        print(
            df_h.loc[
                duplicates,
                ["dataset", "model", metric]
            ].sort_values(["dataset", "model"])
        )

        raise ValueError(
            f"Duplicate dataset/model entries found for "
            f"{metric}, h={horizon-1}."
        )

    # --------------------------------------------------------
    # 3. Create complete-block matrix
    #
    # rows    = datasets
    # columns = models
    # values  = metric
    # --------------------------------------------------------
    pivot = df_h.pivot(
        index="dataset",
        columns="model",
        values=metric
    )

    print("\nOriginal pivot shape:", pivot.shape)

    # Show missing values before dropping anything
    missing_counts = pivot.isna().sum()

    if missing_counts.sum() > 0:
        print("\nMissing values by model:")
        print(
            missing_counts[
                missing_counts > 0
            ].sort_values(ascending=False)
        )

    # Friedman needs a complete block:
    # every included dataset must have every included model.
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    dropped = len(pivot) - len(complete)

    print(
        f"\nComplete datasets used: {len(complete)} / {len(pivot)}"
    )

    if dropped > 0:
        print(
            f"Dropped {dropped} datasets because at least "
            f"one model was missing."
        )

    if len(complete) < 2:
        raise ValueError(
            "Not enough complete datasets for significance testing."
        )

    # --------------------------------------------------------
    # 4. Rank models separately within each dataset
    #
    # Lower RMSE = better
    # Higher QSS = better
    # --------------------------------------------------------
    if metric.lower() == "qss_0.1":
        ascending = False   # higher is better
    else:
        ascending = True    # lower is better
    
    ranks = complete.rank(
        axis=1,
        ascending=ascending,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # --------------------------------------------------------
    # 5. Friedman omnibus test
    # --------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # --------------------------------------------------------
    # 6. Save average ranks
    # --------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["horizon"] = horizon-1
    rank_df["n_datasets"] = len(complete)
    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value

    rank_path = (
        output_dir /
        f"{metric}_h{horizon-1}_average_ranks_with_timestral.csv"
    )

    rank_df.to_csv(rank_path, index=False)

    # --------------------------------------------------------
    # 7. Nemenyi post-hoc
    # --------------------------------------------------------
    p_matrix = None

    if p_value < ALPHA:

        print(
            "\nSignificant overall difference detected. "
            "Running Nemenyi post-hoc test..."
        )

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-value matrix:")
        print(p_matrix)

        p_matrix.to_csv(
            output_dir /
            f"{metric}_h{horizon-1}_nemenyi_pvalues_with_timestral.csv"
        )

        # ----------------------------------------------------
        # 8. Critical-difference diagram
        # ----------------------------------------------------
        if metric == "qss_0.1":
            metric_name = metric.replace("_0.1", "")
        else:
            metric_name = metric
        
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)

        # h=0 is forecast step 1, etc.
        forecast_step = horizon 
        
        ax.set_title(
            f"{freq.capitalize()} {metric_name.upper()}"
            f"(h={horizon})\n"
            f"Friedman p = {p_value:.2e}"
        )

        plt.tight_layout()

        figure_path = (
            output_dir /
            f"{metric_name}_h{horizon-1}_critical_difference_with_timestral.png"
        )

        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:
        print(
            "\nNo significant overall difference at "
            f"alpha={ALPHA}; Nemenyi results are not interpreted."
        )

    return {
        "metric": metric_name,
        "horizon": horizon-1,
        "n_datasets": len(complete),
        "n_models": complete.shape[1],
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "average_ranks": avg_ranks,
        "nemenyi": p_matrix,
    }

for freq in ["daily", "weekly"]:
    RESULTS_DIR = Path(
        r"."
        rf"\BForTFin_results\standardised\output_{freq}"
    )
    
    RMSE_FILE = RESULTS_DIR / "rmse_by_horizon_all_models_with_timestral.csv"
    QSS_FILE = RESULTS_DIR / "qss_by_horizon_all_models_with_timestral.csv"
    # ============================================================
    # RUN ALL THREE HORIZONS
    # ============================================================
    
    rmse = pd.read_csv(RMSE_FILE)
    qss = pd.read_csv(QSS_FILE)
    
    OUTPUT_DIR = RESULTS_DIR / "statistical_significance"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    all_results = []
    
    
    for metric, df in [
        ("rmse", rmse),
        ("qss_0.1", qss),
    ]:
    
        for h in HORIZONS:
    
            result = analyse_horizon(
                df=df,
                metric=metric,
                horizon=h,
                output_dir=OUTPUT_DIR
            )
    
            all_results.append({
                "metric": metric.replace("_0.1",""),
                "horizon": h-1,
                "forecast_step": h,
                "n_datasets": result["n_datasets"],
                "n_models": result["n_models"],
                "friedman_statistic": result["friedman_statistic"],
                "friedman_pvalue": result["friedman_pvalue"],
            })
    
    
    # ============================================================
    # SAVE OVERALL FRIEDMAN SUMMARY
    # ============================================================
    
    friedman_summary = pd.DataFrame(all_results)
    
    friedman_summary.to_csv(
        OUTPUT_DIR / "friedman_qss_horizon_summary_with_timestral.csv",
        index=False
    )
    
    print(f"\n\nFINAL {freq.upper()} FRIEDMAN SUMMARY")
    print("=" * 80)
    print(friedman_summary)

### MSIS and MASE

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "BForTFin_results"
    / "standardised"
)

ALPHA = 0.05

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
})


# ============================================================
# FRIEDMAN-NEMENYI ANALYSIS
# ============================================================

def run_friedman_nemenyi(
    csv_path,
    metric_col,
    metric_name,
    frequency,
    output_dir,
):

    print("\n" + "=" * 90)
    print(
        f"{metric_name.upper()} | "
        f"{frequency.upper()}"
    )
    print("=" * 90)

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    df = pd.read_csv(csv_path)

    required_cols = [
        "asset_class",
        "dataset",
        "model",
        metric_col,
    ]

    missing_cols = [
        c for c in required_cols
        if c not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing columns: {missing_cols}"
        )

    df = df[required_cols].copy()

    df["model"] = df["model"].replace({
        "Chronos-T5": "Chronos",
        "GARCH(1,1)": "Gaussian GARCH",
    })

    df[metric_col] = pd.to_numeric(
        df[metric_col],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 1. CHECK DUPLICATES
    # --------------------------------------------------------

    dataset_cols = [
        "asset_class",
        "dataset",
    ]

    duplicate_mask = df.duplicated(
        subset=dataset_cols + ["model"],
        keep=False
    )

    if duplicate_mask.any():

        print(
            df.loc[
                duplicate_mask
            ].to_string(index=False)
        )

        raise ValueError(
            "Duplicate dataset-model combinations."
        )

    # --------------------------------------------------------
    # 2. CONSTRUCT DATASET × MODEL MATRIX
    # --------------------------------------------------------

    pivot = df.pivot(
        index=dataset_cols,
        columns="model",
        values=metric_col
    )

    pivot = pivot.reindex(
        sorted(pivot.columns),
        axis=1
    )

    print(
        f"Original datasets: {len(pivot)}"
    )

    print(
        f"Models: {len(pivot.columns)}"
    )

    print(
        "\nModels included:"
    )

    print(
        list(pivot.columns)
    )

    # --------------------------------------------------------
    # 3. CHECK COMPLETENESS
    # --------------------------------------------------------

    missing = pivot.isna()

    incomplete = missing.any(axis=1)

    missing_report = (
        missing.loc[incomplete]
        .stack()
        .reset_index(name="missing")
    )

    missing_report = missing_report[
        missing_report["missing"]
    ].drop(columns="missing")

    missing_report.to_csv(
        output_dir / "missing_coverage_with_timestral.csv",
        index=False
    )

    complete = pivot.dropna(
        axis=0,
        how="any"
    )

    print(
        f"\nComplete datasets: "
        f"{len(complete)} / {len(pivot)}"
    )

    if len(complete) != len(pivot):

        print(
            "\nWARNING: Incomplete datasets "
            "were excluded."
        )

        print(
            missing_report.to_string(
                index=False
            )
        )

    if len(complete) < 3:
        raise ValueError(
            "Insufficient complete datasets."
        )

    if complete.shape[1] < 3:
        raise ValueError(
            "Friedman comparison requires "
            "at least three methods."
        )

    if not np.isfinite(
        complete.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Non-finite scores detected."
        )

    # Save exact evaluation matrix

    complete.to_csv(
        output_dir / "complete_score_matrix.csv"
    )

    # --------------------------------------------------------
    # 4. RANK MODELS WITHIN EACH DATASET
    # --------------------------------------------------------

    # Both MASE and MSIS:
    # Lower score = better performance.

    ranks = complete.rank(
        axis=1,
        ascending=True,
        method="average"
    )

    ranks.to_csv(
        output_dir / "dataset_ranks_with_timestral.csv"
    )

    avg_ranks = (
        ranks.mean(axis=0)
        .sort_values()
    )

    print("\nAverage ranks:")
    print(avg_ranks)

    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df.to_csv(
        output_dir / "average_ranks_with_timestral.csv",
        index=False
    )

    # --------------------------------------------------------
    # 5. FRIEDMAN TEST
    # --------------------------------------------------------

    statistic, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    n_datasets = len(complete)
    n_models = len(complete.columns)

    # Kendall's W:
    # Effect size for the Friedman ranking test.

    kendalls_w = (
        statistic
        / (
            n_datasets
            * (n_models - 1)
        )
    )

    print("\nFriedman test:")
    print(f"Statistic: {statistic:.6f}")
    print(f"p-value: {p_value:.6e}")
    print(f"Kendall's W: {kendalls_w:.6f}")

    summary = pd.DataFrame([{
        "metric": metric_name,
        "frequency": frequency,
        "n_datasets": n_datasets,
        "n_models": n_models,
        "friedman_statistic": statistic,
        "friedman_pvalue": p_value,
        "kendalls_w": kendalls_w,
        "significant": p_value < ALPHA,
    }])

    summary.to_csv(
        output_dir / "friedman_summary_with_timestral.csv",
        index=False
    )

    # --------------------------------------------------------
    # 6. NEMENYI POST-HOC
    # --------------------------------------------------------

    if p_value < ALPHA:

        p_matrix = (
            sp.posthoc_nemenyi_friedman(
                complete
            )
        )

        print("\nNemenyi p-values:")
        print(p_matrix.round(5))

        p_matrix.to_csv(
            output_dir / "nemenyi_pvalues_with_timestral.csv"
        )

        # Significant pairwise comparisons

        significant_pairs = []

        for i, model_a in enumerate(
            p_matrix.index
        ):

            for model_b in p_matrix.columns[i + 1:]:

                p = p_matrix.loc[
                    model_a,
                    model_b
                ]

                if p < ALPHA:

                    rank_a = avg_ranks[model_a]
                    rank_b = avg_ranks[model_b]

                    better = (
                        model_a
                        if rank_a < rank_b
                        else model_b
                    )

                    significant_pairs.append({
                        "model_a": model_a,
                        "model_b": model_b,
                        "p_value": p,
                        "better_average_rank": better,
                    })

        significant_df = pd.DataFrame(
            significant_pairs,
            columns=[
                "model_a",
                "model_b",
                "p_value",
                "better_average_rank",
            ]
        )

        significant_df.to_csv(
            output_dir / "significant_pairs_with_timestral.csv",
            index=False
        )

        print(
            "\nSignificant pairwise comparisons:",
            len(significant_df)
        )

        # ----------------------------------------------------
        # 7. CRITICAL-DIFFERENCE DIAGRAM
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(15, 7)
        )

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)

        for label in ax.texts:
            label.set_fontsize(17)

        ax.set_title(
            f"{metric_name} "
            f"{frequency.capitalize()}\n"
            f"Friedman p = {p_value:.2e}",
            fontsize=20
        )

        plt.tight_layout()

        plt.savefig(
            output_dir
            / f"{frequency}_{metric_col}_critical_difference_with_timestral.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:

        p_matrix = None

        print(
            "\nFriedman test not significant. "
            "Nemenyi interpretation skipped."
        )

    return {
        "complete": complete,
        "ranks": ranks,
        "average_ranks": avg_ranks,
        "friedman_statistic": statistic,
        "friedman_pvalue": p_value,
        "kendalls_w": kendalls_w,
        "nemenyi": p_matrix,
    }


# ============================================================
# RUN ALL FOUR ANALYSES
# ============================================================

all_results = {}

for frequency in ["daily", "weekly"]:

    base_dir = (
        RESULTS_DIR
        / f"output_{frequency}"
    )

    significance_dir = (
        base_dir
        / "statistical_significance"
    )

    # --------------------------------------------------------
    # ROLLING-CONTEXT MASE
    # --------------------------------------------------------

    mase_path = (
        base_dir
        / f"point_all_context_mase_{frequency}_with_timestral.csv"
    )

    mase_results = run_friedman_nemenyi(
        csv_path=mase_path,
        metric_col="mase_250",
        metric_name="Rolling-context MASE",
        frequency=frequency,
        output_dir=significance_dir / "MASE",
    )

    all_results[
        ("MASE", frequency)
    ] = mase_results

    # --------------------------------------------------------
    # ROLLING-CONTEXT MSIS
    # --------------------------------------------------------
    
    msis_path = (
        base_dir
        / f"msis_250_decomposition_{frequency}_with_timestral.csv"
    )
    
    msis_results = run_friedman_nemenyi(
        csv_path=msis_path,
        metric_col="msis_250",
        metric_name="Rolling-context MSIS",
        frequency=frequency,
        output_dir=significance_dir / "MSIS",
    )

    all_results[
        ("MSIS", frequency)
    ] = msis_results


# ============================================================
# COMBINED SUMMARY
# ============================================================

summary_rows = []

for (metric, frequency), result in all_results.items():

    summary_rows.append({
        "metric": metric,
        "frequency": frequency,
        "n_datasets": len(result["complete"]),
        "n_models": len(
            result["complete"].columns
        ),
        "friedman_statistic":
            result["friedman_statistic"],
        "friedman_pvalue":
            result["friedman_pvalue"],
        "kendalls_w":
            result["kendalls_w"],
    })

combined_summary = pd.DataFrame(
    summary_rows
)

combined_summary.to_csv(
    RESULTS_DIR
    / "friedman_nemenyi_summary_250.csv_with_timestral",
    index=False
)

print("\n" + "=" * 90)
print("OVERALL FRIEDMAN-NEMENYI SUMMARY")
print("=" * 90)

print(
    combined_summary.to_string(
        index=False
    )
)

### MSIS and MASE model only

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "BForTFin_results"
    / "standardised"
)

POINT_MODELS = {
    "ARIMA",
    "LSTM",
    "LightGBM",
    "Sundial",
    "Lag-Llama",
    "Chronos",
    "Toto",
    "TimesFM",
    "Timestral",
    "Moirai",
    "Moirai-MoE",
}

PROB_MODELS = {
    "Gaussian GARCH",
    "Student-t GARCH",
    "LightGBM",
    "Sundial",
    "Lag-Llama",
    "Chronos",
    "Toto",
    "TimesFM",
    "Timestral",
    "Moirai",
    "Moirai-MoE",
}

ALPHA = 0.05

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
})


# ============================================================
# FRIEDMAN-NEMENYI ANALYSIS
# ============================================================

def run_friedman_nemenyi(
    csv_path,
    metric_col,
    metric_name,
    frequency,
    output_dir,
):

    print("\n" + "=" * 90)
    print(
        f"{metric_name.upper()} | "
        f"{frequency.upper()}"
    )
    print("=" * 90)

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    df = pd.read_csv(csv_path)

    required_cols = [
        "asset_class",
        "dataset",
        "model",
        metric_col,
    ]

    missing_cols = [
        c for c in required_cols
        if c not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing columns: {missing_cols}"
        )

    df = df[required_cols].copy()

    df["model"] = df["model"].replace({
        "Chronos-T5": "Chronos",
        "GARCH(1,1)": "Gaussian GARCH",
    })

    df = df[df["model"].isin(POINT_MODELS) | df["model"].isin(PROB_MODELS)]

    df[metric_col] = pd.to_numeric(
        df[metric_col],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 1. CHECK DUPLICATES
    # --------------------------------------------------------

    dataset_cols = [
        "asset_class",
        "dataset",
    ]

    duplicate_mask = df.duplicated(
        subset=dataset_cols + ["model"],
        keep=False
    )

    if duplicate_mask.any():

        print(
            df.loc[
                duplicate_mask
            ].to_string(index=False)
        )

        raise ValueError(
            "Duplicate dataset-model combinations."
        )

    # --------------------------------------------------------
    # 2. CONSTRUCT DATASET × MODEL MATRIX
    # --------------------------------------------------------

    pivot = df.pivot(
        index=dataset_cols,
        columns="model",
        values=metric_col
    )

    pivot = pivot.reindex(
        sorted(pivot.columns),
        axis=1
    )

    print(
        f"Original datasets: {len(pivot)}"
    )

    print(
        f"Models: {len(pivot.columns)}"
    )

    print(
        "\nModels included:"
    )

    print(
        list(pivot.columns)
    )

    # --------------------------------------------------------
    # 3. CHECK COMPLETENESS
    # --------------------------------------------------------

    missing = pivot.isna()

    incomplete = missing.any(axis=1)

    missing_report = (
        missing.loc[incomplete]
        .stack()
        .reset_index(name="missing")
    )

    missing_report = missing_report[
        missing_report["missing"]
    ].drop(columns="missing")

    missing_report.to_csv(
        output_dir / "model_only_missing_coverage_with_timestral.csv",
        index=False
    )

    complete = pivot.dropna(
        axis=0,
        how="any"
    )

    print(
        f"\nComplete datasets: "
        f"{len(complete)} / {len(pivot)}"
    )

    if len(complete) != len(pivot):

        print(
            "\nWARNING: Incomplete datasets "
            "were excluded."
        )

        print(
            missing_report.to_string(
                index=False
            )
        )

    if len(complete) < 3:
        raise ValueError(
            "Insufficient complete datasets."
        )

    if complete.shape[1] < 3:
        raise ValueError(
            "Friedman comparison requires "
            "at least three methods."
        )

    if not np.isfinite(
        complete.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Non-finite scores detected."
        )

    # Save exact evaluation matrix

    complete.to_csv(
        output_dir / "model_only_complete_score_matrix.csv"
    )

    # --------------------------------------------------------
    # 4. RANK MODELS WITHIN EACH DATASET
    # --------------------------------------------------------

    # Both MASE and MSIS:
    # Lower score = better performance.

    ranks = complete.rank(
        axis=1,
        ascending=True,
        method="average"
    )

    ranks.to_csv(
        output_dir / "model_only_dataset_ranks_with_timestral.csv"
    )

    avg_ranks = (
        ranks.mean(axis=0)
        .sort_values()
    )

    print("\nAverage ranks:")
    print(avg_ranks)

    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df.to_csv(
        output_dir / "model_only_average_ranks_with_timestral.csv",
        index=False
    )

    # --------------------------------------------------------
    # 5. FRIEDMAN TEST
    # --------------------------------------------------------

    statistic, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    n_datasets = len(complete)
    n_models = len(complete.columns)

    # Kendall's W:
    # Effect size for the Friedman ranking test.

    kendalls_w = (
        statistic
        / (
            n_datasets
            * (n_models - 1)
        )
    )

    print("\nFriedman test:")
    print(f"Statistic: {statistic:.6f}")
    print(f"p-value: {p_value:.6e}")
    print(f"Kendall's W: {kendalls_w:.6f}")

    summary = pd.DataFrame([{
        "metric": metric_name,
        "frequency": frequency,
        "n_datasets": n_datasets,
        "n_models": n_models,
        "friedman_statistic": statistic,
        "friedman_pvalue": p_value,
        "kendalls_w": kendalls_w,
        "significant": p_value < ALPHA,
    }])

    summary.to_csv(
        output_dir / "model_only_friedman_summary_with_timestral.csv",
        index=False
    )

    # --------------------------------------------------------
    # 6. NEMENYI POST-HOC
    # --------------------------------------------------------

    if p_value < ALPHA:

        p_matrix = (
            sp.posthoc_nemenyi_friedman(
                complete
            )
        )

        print("\nNemenyi p-values:")
        print(p_matrix.round(5))

        p_matrix.to_csv(
            output_dir / "model_only_nemenyi_pvalues_with_timestral.csv"
        )

        # Significant pairwise comparisons

        significant_pairs = []

        for i, model_a in enumerate(
            p_matrix.index
        ):

            for model_b in p_matrix.columns[i + 1:]:

                p = p_matrix.loc[
                    model_a,
                    model_b
                ]

                if p < ALPHA:

                    rank_a = avg_ranks[model_a]
                    rank_b = avg_ranks[model_b]

                    better = (
                        model_a
                        if rank_a < rank_b
                        else model_b
                    )

                    significant_pairs.append({
                        "model_a": model_a,
                        "model_b": model_b,
                        "p_value": p,
                        "better_average_rank": better,
                    })

        significant_df = pd.DataFrame(
            significant_pairs,
            columns=[
                "model_a",
                "model_b",
                "p_value",
                "better_average_rank",
            ]
        )

        significant_df.to_csv(
            output_dir / "model_only_significant_pairs_with_timestral.csv",
            index=False
        )

        print(
            "\nSignificant pairwise comparisons:",
            len(significant_df)
        )

        # ----------------------------------------------------
        # 7. CRITICAL-DIFFERENCE DIAGRAM
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(15, 7)
        )

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)

        for label in ax.texts:
            label.set_fontsize(17)

        ax.set_title(
            f"{metric_name} "
            f"{frequency.capitalize()}\n"
            f"Friedman p = {p_value:.2e}",
            fontsize=20
        )

        plt.tight_layout()

        plt.savefig(
            output_dir
            / f"model_only_{frequency}_{metric_col}_critical_difference_with_timestral.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:

        p_matrix = None

        print(
            "\nFriedman test not significant. "
            "Nemenyi interpretation skipped."
        )

    return {
        "complete": complete,
        "ranks": ranks,
        "average_ranks": avg_ranks,
        "friedman_statistic": statistic,
        "friedman_pvalue": p_value,
        "kendalls_w": kendalls_w,
        "nemenyi": p_matrix,
    }


# ============================================================
# RUN ALL FOUR ANALYSES
# ============================================================

all_results = {}

for frequency in ["daily", "weekly"]:

    base_dir = (
        RESULTS_DIR
        / f"output_{frequency}"
    )

    significance_dir = (
        base_dir
        / "statistical_significance"
    )

    # --------------------------------------------------------
    # ROLLING-CONTEXT MASE
    # --------------------------------------------------------

    mase_path = (
        base_dir
        / f"point_all_context_mase_{frequency}_with_timestral.csv"
    )

    mase_results = run_friedman_nemenyi(
        csv_path=mase_path,
        metric_col="mase_250",
        metric_name="Rolling-context MASE",
        frequency=frequency,
        output_dir=significance_dir / "MASE",
    )

    all_results[
        ("MASE", frequency)
    ] = mase_results

    # --------------------------------------------------------
    # ROLLING-CONTEXT MSIS
    # --------------------------------------------------------
    
    msis_path = (
        base_dir
        / f"msis_250_decomposition_{frequency}_with_timestral.csv"
    )
    
    msis_results = run_friedman_nemenyi(
        csv_path=msis_path,
        metric_col="msis_250",
        metric_name="Rolling-context MSIS",
        frequency=frequency,
        output_dir=significance_dir / "MSIS",
    )

    all_results[
        ("MSIS", frequency)
    ] = msis_results


# ============================================================
# COMBINED SUMMARY
# ============================================================

summary_rows = []

for (metric, frequency), result in all_results.items():

    summary_rows.append({
        "metric": metric,
        "frequency": frequency,
        "n_datasets": len(result["complete"]),
        "n_models": len(
            result["complete"].columns
        ),
        "friedman_statistic":
            result["friedman_statistic"],
        "friedman_pvalue":
            result["friedman_pvalue"],
        "kendalls_w":
            result["kendalls_w"],
    })

combined_summary = pd.DataFrame(
    summary_rows
)

combined_summary.to_csv(
    RESULTS_DIR
    / "model_only_friedman_nemenyi_summary_250.csv_with_timestral",
    index=False
)

print("\n" + "=" * 90)
print("OVERALL FRIEDMAN-NEMENYI SUMMARY")
print("=" * 90)

print(
    combined_summary.to_string(
        index=False
    )
)

### MSIS and MASE horizon specific

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIG
# ============================================================

HORIZONS = [1, 5, 20]
ALPHA = 0.05


# ============================================================
# ONE HORIZON / ONE METRIC
# ============================================================

def analyse_horizon(
    df,
    metric,
    horizon,
    output_dir
):
    print("\n" + "=" * 80)
    print(f"{freq.capitalize()} {metric.upper()} — HORIZON h={horizon}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Select this horizon
    # --------------------------------------------------------
    df_h = df[df["horizon"] == horizon].copy()

    print("Rows:", len(df_h))
    print("Datasets:", df_h["dataset"].nunique())
    print("Models:", df_h["model"].nunique())

    # --------------------------------------------------------
    # 2. Check for duplicate dataset/model entries
    # --------------------------------------------------------
    duplicates = df_h.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nWARNING: duplicate dataset/model rows found:")
        print(
            df_h.loc[
                duplicates,
                ["dataset", "model", metric]
            ].sort_values(["dataset", "model"])
        )

        raise ValueError(
            f"Duplicate dataset/model entries found for "
            f"{metric}, h={horizon-1}."
        )

    # --------------------------------------------------------
    # 3. Create complete-block matrix
    #
    # rows    = datasets
    # columns = models
    # values  = metric
    # --------------------------------------------------------
    pivot = df_h.pivot(
        index="dataset",
        columns="model",
        values=metric
    )

    print("\nOriginal pivot shape:", pivot.shape)

    # Show missing values before dropping anything
    missing_counts = pivot.isna().sum()

    if missing_counts.sum() > 0:
        print("\nMissing values by model:")
        print(
            missing_counts[
                missing_counts > 0
            ].sort_values(ascending=False)
        )

    # Friedman needs a complete block:
    # every included dataset must have every included model.
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    dropped = len(pivot) - len(complete)

    print(
        f"\nComplete datasets used: {len(complete)} / {len(pivot)}"
    )

    if dropped > 0:
        print(
            f"Dropped {dropped} datasets because at least "
            f"one model was missing."
        )

    if len(complete) < 2:
        raise ValueError(
            "Not enough complete datasets for significance testing."
        )

    # --------------------------------------------------------
    # 4. Rank models separately within each dataset
    #
    # Lower RMSE = better
    # Higher QSS = better
    # --------------------------------------------------------
    if metric.lower() == "qss_0.1":
        ascending = False   # higher is better
    else:
        ascending = True    # lower is better
    
    ranks = complete.rank(
        axis=1,
        ascending=ascending,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # --------------------------------------------------------
    # 5. Friedman omnibus test
    # --------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # --------------------------------------------------------
    # 6. Save average ranks
    # --------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["horizon"] = horizon-1
    rank_df["n_datasets"] = len(complete)
    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value

    rank_path = (
        output_dir /
        f"model_only_{metric}_h{horizon-1}_average_ranks_with_timestral.csv"
    )

    rank_df.to_csv(rank_path, index=False)

    # --------------------------------------------------------
    # 7. Nemenyi post-hoc
    # --------------------------------------------------------
    p_matrix = None

    if p_value < ALPHA:

        print(
            "\nSignificant overall difference detected. "
            "Running Nemenyi post-hoc test..."
        )

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-value matrix:")
        print(p_matrix)

        p_matrix.to_csv(
            output_dir /
            f"model_only_{metric}_h{horizon-1}_nemenyi_pvalues_with_timestral.csv"
        )

        # ----------------------------------------------------
        # 8. Critical-difference diagram
        # ----------------------------------------------------
        metric_name = metric.replace("_250", "")
        
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)

        # h=0 is forecast step 1, etc.
        forecast_step = horizon 
        
        ax.set_title(
            f"{freq.capitalize()} {metric_name.upper()}"
            f"(h={horizon})\n"
            f"Friedman p = {p_value:.2e}"
        )

        plt.tight_layout()

        figure_path = (
            output_dir /
            f"model_only_{freq}_{metric_name}_h{horizon-1}_critical_difference_with_timestral.png"
        )

        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:
        print(
            "\nNo significant overall difference at "
            f"alpha={ALPHA}; Nemenyi results are not interpreted."
        )

    return {
        "metric": metric_name,
        "horizon": horizon-1,
        "n_datasets": len(complete),
        "n_models": complete.shape[1],
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "average_ranks": avg_ranks,
        "nemenyi": p_matrix,
    }

for freq in ["daily", "weekly"]:
    RESULTS_DIR = Path(
        r"."
        rf"\BForTFin_results\standardised\output_{freq}"
    )
    
    MSIS_FILE = RESULTS_DIR / f"msis_250_by_horizon_{freq}_with_timestral.csv"
    MASE_FILE = RESULTS_DIR / f"point_models_context_mase_by_horizon_{freq}_with_timestral.csv"
    # ============================================================
    # RUN ALL THREE HORIZONS
    # ============================================================
    
    msis = pd.read_csv(MSIS_FILE)
    mase = pd.read_csv(MASE_FILE)
    
    OUTPUT_DIR = RESULTS_DIR / "statistical_significance"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    all_results = []
    
    
    for metric, df in [
        ("msis_250", msis),
        ("mase_250", mase),
    ]:
    
        for h in HORIZONS:
    
            result = analyse_horizon(
                df=df,
                metric=metric,
                horizon=h,
                output_dir=OUTPUT_DIR
            )
    
            all_results.append({
                "metric": metric.replace("_0.1",""),
                "horizon": h-1,
                "forecast_step": h,
                "n_datasets": result["n_datasets"],
                "n_models": result["n_models"],
                "friedman_statistic": result["friedman_statistic"],
                "friedman_pvalue": result["friedman_pvalue"],
            })
    
    
    # ============================================================
    # SAVE OVERALL FRIEDMAN SUMMARY
    # ============================================================
    
    friedman_summary = pd.DataFrame(all_results)
    
    friedman_summary.to_csv(
        OUTPUT_DIR / "model_only_friedman_msis_mase_horizon_summary_with_timestral.csv",
        index=False
    )
    
    print(f"\n\nFINAL {freq.upper()} FRIEDMAN SUMMARY")
    print("=" * 80)
    print(friedman_summary)

# Other metrics

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability"
    r"\BForTFin_results"
    r"\standardised"
    r"\output_daily"
)

# RANKING_DIR = (
#     BASE_DIR / "student_t_garch_ranking_update"
# )

# QSS_PATH = (
#     RANKING_DIR
#     / "all_models_qss_with_student_t_garch.csv"
# )

# CRPS_PATH = (
#     RANKING_DIR
#     / "all_models_qcrps_with_student_t_garch.csv"
# )

QSS_PATH = (BASE_DIR / "qss_summary_all_models_with_timestral.csv")

CRPS_PATH = (BASE_DIR / "all_models_crps_by_dataset_with_timestral.csv")

EMPIRICAL_PATH = (
    BASE_DIR / "empirical_quantile_crps_daily.csv"
)

MASE_PATH = (
    BASE_DIR / "point_all_context_mase_daily_with_timestral.csv"
)

OUTPUT_DIR = (
    BASE_DIR / "best_model_tables"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

WITHIN_PCT = 0.01

EXPECTED_DATASETS = 72


# ============================================================
# MODEL ELIGIBILITY
# ============================================================

POINT_MODELS = {
    "ARIMA",
    "LSTM",
    "LightGBM",
    "Sundial",
    "Lag-Llama",
    "Chronos-T5",
    "Toto",
    "TimesFM",
    "Timestral",
    "Moirai",
    "Moirai-MoE",
}

PROB_MODELS = {
    "GARCH(1,1)",
    "Student-t GARCH",
    "LightGBM",
    "Sundial",
    "Lag-Llama",
    "Chronos-T5",
    "Toto",
    "TimesFM",
    "Timestral",
    "Moirai",
    "Moirai-MoE",
}


# ============================================================
# NORMALISE MODEL NAMES
# ============================================================

def normalize_model(s):

    s = str(s).strip()

    aliases = {
        "Chronos": "Chronos-T5",
        "Gaussian GARCH": "GARCH(1,1)",
        "GARCH": "GARCH(1,1)",
        "GARCH(1,1)": "GARCH(1,1)",
        "Student-t GARCH": "Student-t GARCH",
        "Student-t GARCH(1,1)": "Student-t GARCH",
        "Zero Return": "Zero",
        "Zero-return": "Zero",
        "Zero": "Zero",
    }

    return aliases.get(s, s)


# ============================================================
# NORMALISE DATASET NAMES
# ============================================================

def normalize_dataset(s):

    s = str(s).strip().lower()

    s = unicodedata.normalize(
        "NFKD", s
    )

    s = "".join(
        c for c in s
        if not unicodedata.combining(c)
    )

    # Remove supervised-dataset suffixes.
    # This is needed for the empirical baseline.

    s = re.sub(
        r"_no_preprocess_lookback[-_]\d+"
        r"_horizon[-_]\d+"
        r"_slide[-_]\d+.*$",
        "",
        s,
    )

    s = s.replace("historical data", "")
    s = s.replace("historical", "")

    s = re.sub(
        r"[^a-z0-9]+",
        "",
        s
    )

    return s


def prepare_keys(df):

    df = df.copy()

    df["asset_class"] = (
        df["asset_class"]
        .astype(str)
        .str.strip()
    )

    df["dataset_key"] = (
        df["dataset"]
        .apply(normalize_dataset)
    )

    return df


# ============================================================
# VALIDATION HELPERS
# ============================================================

KEYS = ["asset_class", "dataset_key"]


def check_unique(df, keys, label):

    duplicated = df.duplicated(
        subset=keys,
        keep=False
    )

    if duplicated.any():

        print(f"\nDUPLICATES IN {label}")

        display(
            df.loc[duplicated]
            .sort_values(keys)
        )

        raise ValueError(
            f"Duplicate keys detected: {label}"
        )


def check_coverage(reference, other, label):

    ref_keys = reference[KEYS].drop_duplicates()

    other_keys = other[KEYS].drop_duplicates()

    missing = ref_keys.merge(
        other_keys,
        on=KEYS,
        how="left",
        indicator=True
    )

    missing = missing[
        missing["_merge"] == "left_only"
    ]

    if len(missing):

        print(f"\nMISSING DATASETS: {label}")

        display(missing)

        raise ValueError(
            f"Incomplete dataset coverage: {label}"
        )


def check_models(df, expected_models, label):

    actual = set(df["model"].unique())

    missing = expected_models - actual
    unexpected = actual - expected_models

    if missing or unexpected:

        raise ValueError(
            f"{label}\n"
            f"Missing models: {sorted(missing)}\n"
            f"Unexpected models: {sorted(unexpected)}"
        )


# ============================================================
# LOAD RESULTS
# ============================================================

qss = prepare_keys(
    pd.read_csv(QSS_PATH)
)

crps = prepare_keys(
    pd.read_csv(CRPS_PATH)
)

empirical = prepare_keys(
    pd.read_csv(EMPIRICAL_PATH)
)

mase = prepare_keys(
    pd.read_csv(MASE_PATH)
)

for df in [qss, crps, empirical, mase]:

    df["model"] = (
        df["model"].apply(normalize_model)
    )


# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

required = {
    "QSS": (
        qss,
        ["model", "asset_class", "dataset", "qss_0.1"]
    ),
    "CRPS": (
        crps,
        ["model", "asset_class", "dataset", "mean_qcrps"]
    ),
    "Empirical CRPS": (
        empirical,
        ["model", "asset_class", "dataset", "mean_qcrps"]
    ),
    "MASE": (
        mase,
        ["model", "asset_class", "dataset", "mase_250"]
    ),
}

for label, (df, columns) in required.items():

    missing = [
        c for c in columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{label}: missing columns {missing}"
        )


# ============================================================
# FILTER ELIGIBLE MODELS
# ============================================================

qss = qss[
    qss["model"].isin(PROB_MODELS)
].copy()

crps = crps[
    crps["model"].isin(PROB_MODELS)
].copy()

mase_models = mase[
    mase["model"].isin(POINT_MODELS)
].copy()

zero = mase[
    mase["model"] == "Zero"
].copy()


# ============================================================
# CHECK MODEL COVERAGE
# ============================================================

check_models(
    qss,
    PROB_MODELS,
    "QSS"
)

check_models(
    crps,
    PROB_MODELS,
    "CRPS"
)

check_models(
    mase_models,
    POINT_MODELS,
    "MASE"
)

if len(zero) == 0:
    raise ValueError(
        "Zero-return baseline not found in MASE file."
    )

if not (
    empirical["model"] == "Empirical Quantiles"
).all():

    raise ValueError(
        "Unexpected empirical baseline model labels."
    )


# ============================================================
# CHECK DUPLICATES
# ============================================================

for label, df in [
    ("QSS", qss),
    ("CRPS", crps),
    ("MASE", mase_models),
]:

    check_unique(
        df,
        KEYS + ["model"],
        label
    )

check_unique(
    empirical,
    KEYS,
    "Empirical baseline"
)

check_unique(
    zero,
    KEYS,
    "Zero baseline"
)


# ============================================================
# USE QSS DATASETS AS REFERENCE
# ============================================================

datasets = (
    qss[
        ["asset_class", "dataset", "dataset_key"]
    ]
    .drop_duplicates(subset=KEYS)
    .copy()
)

if len(datasets) != EXPECTED_DATASETS:

    raise ValueError(
        f"Expected {EXPECTED_DATASETS} datasets, "
        f"found {len(datasets)}"
    )


# ============================================================
# VALIDATE COVERAGE
# ============================================================

for label, df in [
    ("CRPS", crps),
    ("MASE", mase_models),
    ("Empirical", empirical),
    ("Zero", zero),
]:

    check_coverage(
        datasets,
        df,
        label
    )

# Check that every dataset has every eligible model.

for label, df, expected in [
    ("QSS", qss, PROB_MODELS),
    ("CRPS", crps, PROB_MODELS),
    ("MASE", mase_models, POINT_MODELS),
]:

    counts = (
        df.groupby(KEYS)["model"]
        .nunique()
    )

    if (
        len(counts) != EXPECTED_DATASETS
        or not counts.eq(len(expected)).all()
    ):

        print(f"\nINCOMPLETE MODEL COVERAGE: {label}")

        display(counts)

        raise ValueError(
            f"{label}: incomplete model coverage"
        )


# ============================================================
# NUMERIC VALIDATION
# ============================================================

for df, column, label in [
    (qss, "qss_0.1", "QSS"),
    (crps, "mean_qcrps", "CRPS"),
    (empirical, "mean_qcrps", "Empirical CRPS"),
    (mase_models, "mase_250", "MASE"),
    (zero, "mase_250", "Zero MASE"),
]:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    if not np.isfinite(
        df[column].to_numpy()
    ).all():

        raise ValueError(
            f"{label}: non-finite values"
        )


# ============================================================
# BASELINES
# ============================================================

empirical_ref = (
    empirical[
        KEYS + ["mean_qcrps"]
    ]
    .rename(
        columns={
            "mean_qcrps": "empirical_crps"
        }
    )
)

zero_ref = (
    zero[
        KEYS + ["mase_250"]
    ]
    .rename(
        columns={
            "mase_250": "zero_mase"
        }
    )
)

if (
    empirical_ref["empirical_crps"] <= 0
).any():

    raise ValueError(
        "Empirical CRPS must be positive."
    )

if (
    zero_ref["zero_mase"] <= 0
).any():

    raise ValueError(
        "Zero MASE must be positive."
    )


# ============================================================
# ADD CRPSS
# ============================================================

crps = crps.merge(
    empirical_ref,
    on=KEYS,
    how="left",
    validate="many_to_one"
)

if crps["empirical_crps"].isna().any():
    raise ValueError(
        "Missing empirical CRPS after merge."
    )

crps["crpss"] = (
    1.0
    - crps["mean_qcrps"]
    / crps["empirical_crps"]
)


# ============================================================
# ADD MASESS
# ============================================================

# mase_models = mase_models.merge(
#     zero_ref,
#     on=KEYS,
#     how="left",
#     validate="many_to_one"
# )

# if mase_models["zero_mase"].isna().any():
#     raise ValueError(
#         "Missing zero MASE after merge."
#     )

# mase_models["masess"] = (
#     1.0
#     - mase_models["mase_250"]
#     / mase_models["zero_mase"]
# )


mase_models["zero_mase"] = pd.to_numeric(
    mase_models["zero_mase"],
    errors="coerce"
)

# Validate the existing baseline.
if (
    not np.isfinite(mase_models["zero_mase"]).all()
    or (mase_models["zero_mase"] <= 0).any()
):
    raise ValueError(
        "Invalid zero-return MASE baseline."
    )

# Calculate skill relative to zero return.
mase_models["masess"] = (
    1.0
    - mase_models["mase_250"]
    / mase_models["zero_mase"]
)

print("MASESS calculated successfully!")

display(
    mase_models[
        ["model", "mase_250", "zero_mase", "masess"]
    ].head()
)


# ============================================================
# BEST MODEL HELPER
# ============================================================

def best_by_dataset(
    df,
    score_col,
    prefix,
    higher_is_better,
    tolerance_col=None,
):

    """
    Select the best model independently for each dataset.

    Within-1% counts use the raw loss where available.

    For QSS, use an absolute tolerance of
    1% of the best score's magnitude.
    """

    output_rows = []

    for keys, group in df.groupby(
        KEYS,
        sort=False
    ):

        group = group.copy()

        # if higher_is_better:

        #     best_score = group[score_col].max()

        #     winners = group[
        #         np.isclose(
        #             group[score_col],
        #             best_score,
        #             rtol=1e-12,
        #             atol=1e-14
        #         )
        #     ]

        # else:

        #     best_score = group[score_col].min()

        #     winners = group[
        #         np.isclose(
        #             group[score_col],
        #             best_score,
        #             rtol=1e-12,
        #             atol=1e-14
        #         )
        #     ]

        # best_model = sorted(
        #     winners["model"].tolist()
        # )[0]

        if higher_is_better:
            best_idx = group[score_col].idxmax()
        else:
            best_idx = group[score_col].idxmin()
        
        best_score = group.loc[
            best_idx, score_col
        ]
        
        best_model = group.loc[
            best_idx, "model"
        ]

        # ----------------------------------------
        # WITHIN 1%
        # ----------------------------------------

        if tolerance_col is not None:

            # Use corresponding raw loss.
            # Lower is better.

            best_raw = group[
                tolerance_col
            ].min()

            threshold = (
                best_raw * (1 + WITHIN_PCT)
            )

            n_within = int(
                (
                    group[tolerance_col]
                    <= threshold
                ).sum()
            )

        elif higher_is_better:

            # QSS can be negative.
            # Therefore, avoid multiplying the
            # best score directly by 0.99.

            tolerance = (
                WITHIN_PCT * abs(best_score)
            )

            n_within = int(
                (
                    group[score_col]
                    >= best_score - tolerance
                ).sum()
            )

        else:

            threshold = (
                best_score * (1 + WITHIN_PCT)
            )

            n_within = int(
                (
                    group[score_col]
                    <= threshold
                ).sum()
            )

        output_rows.append({
            "asset_class": keys[0],
            "dataset_key": keys[1],
            f"best_{prefix}": best_score,
            f"best_{prefix}_model": best_model,
            f"{prefix}_within_1pct": n_within,
        })

    return pd.DataFrame(output_rows)


# ============================================================
# COMPUTE BEST VALUES
# ============================================================

best_qss = best_by_dataset(
    qss,
    score_col="qss_0.1",
    prefix="qss",
    higher_is_better=True,
)

best_crps = best_by_dataset(
    crps,
    score_col="mean_qcrps",
    prefix="crps",
    higher_is_better=False,
)

# best_crpss = best_by_dataset(
#     crps,
#     score_col="crpss",
#     prefix="crpss",
#     higher_is_better=True,
#     tolerance_col="mean_qcrps",
# )

best_crpss = best_by_dataset(
    crps,
    score_col="crpss",
    prefix="crpss",
    higher_is_better=True,
)

best_mase = best_by_dataset(
    mase_models,
    score_col="mase_250",
    prefix="mase",
    higher_is_better=False,
)

# best_masess = best_by_dataset(
#     mase_models,
#     score_col="masess",
#     prefix="masess",
#     higher_is_better=True,
#     tolerance_col="mase_250",
# )

best_masess = best_by_dataset(
    mase_models,
    score_col="masess",
    prefix="masess",
    higher_is_better=True,
)


# ============================================================
# COMBINE ALL COLUMNS
# ============================================================

result = datasets.copy()

components = [
    best_crps,
    best_qss,
    best_mase,
    best_masess,
    best_crpss,
    empirical_ref,
    zero_ref,
]

for component in components:

    result = result.merge(
        component,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )


# ============================================================
# FINAL COLUMN ORDER
# ============================================================

result = result[
    [
        "asset_class",
        "dataset",

        "best_crps",
        "best_crps_model",
        "crps_within_1pct",

        "best_qss",
        "best_qss_model",
        "qss_within_1pct",

        "best_mase",
        "best_mase_model",
        "mase_within_1pct",

        "best_masess",
        "best_masess_model",
        "masess_within_1pct",

        "best_crpss",
        "best_crpss_model",
        "crpss_within_1pct",

        "empirical_crps",
        "zero_mase",
    ]
]


# ============================================================
# FINAL VALIDATION
# ============================================================

if len(result) != EXPECTED_DATASETS:

    raise ValueError(
        f"Expected {EXPECTED_DATASETS} rows, "
        f"got {len(result)}"
    )

if result.isna().any().any():

    print(
        result.loc[
            result.isna().any(axis=1)
        ]
    )

    raise ValueError(
        "Missing values in final table."
    )


# MASE and MASESS should have identical winners.

if not (
    result["best_mase_model"]
    == result["best_masess_model"]
).all():

    raise ValueError(
        "MASE and MASESS winners differ."
    )


# CRPS and CRPSS should have identical winners.

if not (
    result["best_crps_model"]
    == result["best_crpss_model"]
).all():

    raise ValueError(
        "CRPS and CRPSS winners differ."
    )


# ============================================================
# SAVE
# ============================================================

output_path = (
    OUTPUT_DIR
    / "best_metrics_with_skill_daily_with_timestral.csv"
)

result.to_csv(
    output_path,
    index=False
)

print(f"\nSaved: {output_path}")

print(
    f"Datasets: {len(result)}"
)

display(result)


# ============================================================
# SUMMARY: POSITIVE BEST-MODEL SKILL
# ============================================================

summary_rows = []

for metric in [
    "qss",
    "masess",
    "crpss",
]:

    values = result[f"best_{metric}"]

    summary_rows.append({
        "metric": metric.upper(),
        "n_datasets": len(values),
        "n_positive": int((values > 0).sum()),
        "n_zero": int((values == 0).sum()),
        "n_negative": int((values < 0).sum()),
        "pct_positive": (
            100 * (values > 0).mean()
        ),
        "mean_best_score": values.mean(),
        "median_best_score": values.median(),
    })

summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUTPUT_DIR
    / "best_skill_summary_daily_with_timestral.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print("\nBEST-MODEL SKILL SUMMARY")

display(summary)

In [ ]:
# ============================================================
# EXTEND BEST-MODEL TABLE:
# RMSE, RMSESS, MSIS, MSISS
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# 1. LOAD MSIS RESULTS
# ============================================================

MSIS_PATH = (
    BASE_DIR
    / "msis_250_decomposition_daily_with_timestral.csv"
)

msis = prepare_keys(
    pd.read_csv(MSIS_PATH)
)

msis["model"] = msis["model"].apply(normalize_model)

print("MSIS models:")
print(sorted(msis["model"].unique()))


# ============================================================
# 2. PREPARE RMSE AND RMSESS
# ============================================================

# The existing MASE results already contain:
# rmse, zero_rmse, mase_250, zero_mase

rmse_models = mase_models.copy()

for col in ["rmse", "zero_rmse"]:

    rmse_models[col] = pd.to_numeric(
        rmse_models[col],
        errors="coerce"
    )

    if not np.isfinite(
        rmse_models[col].to_numpy()
    ).all():
        raise ValueError(
            f"Non-finite values in {col}"
        )

if (rmse_models["zero_rmse"] <= 0).any():
    raise ValueError(
        "Zero-return RMSE must be positive."
    )

if (rmse_models["rmse"] < 0).any():
    raise ValueError(
        "Negative RMSE detected."
    )

# RMSE skill score relative to zero return

rmse_models["rmsess"] = (
    1.0
    - rmse_models["rmse"]
    / rmse_models["zero_rmse"]
)

# Dataset-level reference

zero_rmse_ref = (
    rmse_models[
        KEYS + ["zero_rmse"]
    ]
    .drop_duplicates()
)

check_unique(
    zero_rmse_ref,
    KEYS,
    "Zero RMSE reference"
)


# ============================================================
# 3. PREPARE MSIS AND MSISS
# ============================================================

required_msis_cols = [
    "model",
    "asset_class",
    "dataset",
    "msis_250",
    "empirical_msis",
]

missing_cols = [
    col for col in required_msis_cols
    if col not in msis.columns
]

if missing_cols:
    raise ValueError(
        f"MSIS missing columns: {missing_cols}"
    )

# Only probabilistic forecasting models.
# Exclude the empirical baseline from best-model selection.

msis_models = msis[
    msis["model"].isin(PROB_MODELS)
].copy()

check_models(
    msis_models,
    PROB_MODELS,
    "MSIS"
)

check_unique(
    msis_models,
    KEYS + ["model"],
    "MSIS"
)

check_coverage(
    datasets,
    msis_models,
    "MSIS"
)

# Confirm all eligible models are present
# for every dataset.

msis_counts = (
    msis_models
    .groupby(KEYS)["model"]
    .nunique()
)

if (
    len(msis_counts) != EXPECTED_DATASETS
    or not msis_counts.eq(len(PROB_MODELS)).all()
):
    raise ValueError(
        "Incomplete MSIS model coverage."
    )

# Convert scores to numeric

for col in ["msis_250", "empirical_msis"]:

    msis_models[col] = pd.to_numeric(
        msis_models[col],
        errors="coerce"
    )

    if not np.isfinite(
        msis_models[col].to_numpy()
    ).all():
        raise ValueError(
            f"Non-finite MSIS values in {col}"
        )

if (msis_models["empirical_msis"] <= 0).any():
    raise ValueError(
        "Empirical MSIS must be positive."
    )

if (msis_models["msis_250"] < 0).any():
    raise ValueError(
        "Negative MSIS detected."
    )

# Ensure the empirical baseline is consistent
# across models within each dataset.

baseline_counts = (
    msis_models
    .groupby(KEYS)["empirical_msis"]
    .nunique()
)

if not baseline_counts.eq(1).all():
    raise ValueError(
        "Inconsistent empirical MSIS references "
        "within one or more datasets."
    )

# Calculate MSIS skill score

msis_models["msiss"] = (
    1.0
    - msis_models["msis_250"]
    / msis_models["empirical_msis"]
)

# Dataset-level empirical MSIS reference

empirical_msis_ref = (
    msis_models[
        KEYS + ["empirical_msis"]
    ]
    .drop_duplicates()
)

check_unique(
    empirical_msis_ref,
    KEYS,
    "Empirical MSIS reference"
)


# ============================================================
# 4. BEST RMSE AND RMSESS
# ============================================================

best_rmse = best_by_dataset(
    rmse_models,
    score_col="rmse",
    prefix="rmse",
    higher_is_better=False,
)

best_rmsess = best_by_dataset(
    rmse_models,
    score_col="rmsess",
    prefix="rmsess",
    higher_is_better=True,
)


# ============================================================
# 5. BEST MSIS AND MSISS
# ============================================================

best_msis = best_by_dataset(
    msis_models,
    score_col="msis_250",
    prefix="msis",
    higher_is_better=False,
)

best_msiss = best_by_dataset(
    msis_models,
    score_col="msiss",
    prefix="msiss",
    higher_is_better=True,
)


# ============================================================
# 6. COMBINE WITH EXISTING RESULT
# ============================================================

# Begin with the previously generated table.
# Remove any extra columns from rerunning this extension
# to avoid pandas creating _x and _y suffixes.

new_components = [
    best_rmse,
    best_rmsess,
    best_msis,
    best_msiss,
    zero_rmse_ref,
    empirical_msis_ref,
]

new_columns = sorted({
    col
    for component in new_components
    for col in component.columns
    if col not in KEYS
})

# ============================================================
# 6. COMBINE WITH EXISTING RESULT — FIXED
# ============================================================

# Start from the previously generated table.
result_extended = result.copy()

# Reconstruct dataset_key using our existing
# dataset-name normalisation function.
result_extended["dataset_key"] = (
    result_extended["dataset"]
    .apply(normalize_dataset)
)

# Verify dataset identifiers are unique.
check_unique(
    result_extended,
    KEYS,
    "Existing best-model table"
)

# Remove columns that might already exist
# if this extension has been run before.
new_columns = sorted({
    col
    for component in new_components
    for col in component.columns
    if col not in KEYS
})

result_extended = result_extended.drop(
    columns=new_columns,
    errors="ignore"
)

# Merge the additional metrics.
for component in new_components:

    # Check that the component has both join keys.
    missing_keys = [
        key for key in KEYS
        if key not in component.columns
    ]

    if missing_keys:
        raise ValueError(
            f"Component missing join keys: {missing_keys}"
        )

    result_extended = result_extended.merge(
        component,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

print("All additional metrics merged successfully!")


# ============================================================
# 7. VERIFY BEST-MODEL CONSISTENCY
# ============================================================

# Raw loss and its corresponding skill score
# must select the same best model.

model_pairs = [
    ("rmse", "rmsess"),
    ("mase", "masess"),
    ("crps", "crpss"),
    ("msis", "msiss"),
]

for raw_metric, skill_metric in model_pairs:

    raw_col = f"best_{raw_metric}_model"
    skill_col = f"best_{skill_metric}_model"

    if not (
        result_extended[raw_col]
        == result_extended[skill_col]
    ).all():

        raise ValueError(
            f"{raw_metric.upper()} and "
            f"{skill_metric.upper()} "
            "best models do not match."
        )

print(
    "\nAll raw-loss and skill-score "
    "best-model identities match."
)


# ============================================================
# 8. FINAL COLUMN ORDER
# ============================================================

column_order = [
    "asset_class",
    "dataset",

    # RMSE
    "best_rmse",
    "best_rmse_model",
    "rmse_within_1pct",

    # RMSESS
    "best_rmsess",
    "best_rmsess_model",
    "rmsess_within_1pct",

    # MASE
    "best_mase",
    "best_mase_model",
    "mase_within_1pct",

    # MASESS
    "best_masess",
    "best_masess_model",
    "masess_within_1pct",

    # QSS
    "best_qss",
    "best_qss_model",
    "qss_within_1pct",

    # CRPS
    "best_crps",
    "best_crps_model",
    "crps_within_1pct",

    # CRPSS
    "best_crpss",
    "best_crpss_model",
    "crpss_within_1pct",

    # MSIS
    "best_msis",
    "best_msis_model",
    "msis_within_1pct",

    # MSISS
    "best_msiss",
    "best_msiss_model",
    "msiss_within_1pct",

    # Reference scores
    "zero_rmse",
    "zero_mase",
    "empirical_crps",
    "empirical_msis",
]

missing_output_cols = [
    col for col in column_order
    if col not in result_extended.columns
]

if missing_output_cols:
    raise ValueError(
        f"Missing output columns: {missing_output_cols}"
    )

result_extended = result_extended[
    column_order
]


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

if len(result_extended) != EXPECTED_DATASETS:
    raise ValueError(
        f"Expected {EXPECTED_DATASETS} datasets, "
        f"got {len(result_extended)}"
    )

if result_extended.isna().any().any():

    display(
        result_extended.loc[
            result_extended.isna().any(axis=1)
        ]
    )

    raise ValueError(
        "Missing values in extended table."
    )


# ============================================================
# 10. SAVE EXTENDED TABLE
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    OUTPUT_DIR
    / "best_metrics_all_daily_with_timestral.csv"
)

result_extended.to_csv(
    output_path,
    index=False
)

print("\nSaved extended table:")
print(output_path)

display(result_extended)


# ============================================================
# 11. BEST-MODEL SKILL SUMMARY
# ============================================================

skill_metrics = [
    "rmsess",
    "masess",
    "qss",
    "crpss",
    "msiss",
]

summary_rows = []

for metric in skill_metrics:

    values = result_extended[
        f"best_{metric}"
    ]

    summary_rows.append({
        "metric": metric.upper(),
        "n_datasets": len(values),
        "n_positive": int((values > 0).sum()),
        "n_zero": int((values == 0).sum()),
        "n_negative": int((values < 0).sum()),
        "pct_positive": (
            100 * (values > 0).mean()
        ),
        "mean_best_score": float(
            values.mean()
        ),
        "median_best_score": float(
            values.median()
        ),
    })

summary_extended = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUTPUT_DIR
    / "best_skill_summary_all_daily_with_timestral.csv"
)

summary_extended.to_csv(
    summary_path,
    index=False
)

print("\nBEST-MODEL SKILL SUMMARY")

display(summary_extended)

### Weekly

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability"
    r"\BForTFin_results"
    r"\standardised"
    r"\output_weekly"
)

QSS_PATH = (BASE_DIR / "qss_summary_all_models_with_timestral.csv")

CRPS_PATH = (BASE_DIR / "all_models_crps_by_dataset_with_timestral.csv")


EMPIRICAL_PATH = (
    BASE_DIR / "empirical_quantile_crps_weekly.csv"
)

MASE_PATH = (
    BASE_DIR / "point_all_context_mase_weekly_with_timestral.csv"
)

OUTPUT_DIR = (
    BASE_DIR / "best_model_tables"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

WITHIN_PCT = 0.01

EXPECTED_DATASETS = 71


# ============================================================
# MODEL ELIGIBILITY
# ============================================================

POINT_MODELS = {
    "ARIMA",
    "LSTM",
    "LightGBM",
    "Sundial",
    "Lag-Llama",
    "Chronos-T5",
    "Toto",
    "TimesFM",
    "Timestral",
    "Moirai",
    "Moirai-MoE",
}

PROB_MODELS = {
    "GARCH(1,1)",
    "Student-t GARCH",
    "LightGBM",
    "Sundial",
    "Lag-Llama",
    "Chronos-T5",
    "Toto",
    "TimesFM",
    "Timestral",
    "Moirai",
    "Moirai-MoE",
}


# ============================================================
# NORMALISE MODEL NAMES
# ============================================================

def normalize_model(s):

    s = str(s).strip()

    aliases = {
        "Chronos": "Chronos-T5",
        "Gaussian GARCH": "GARCH(1,1)",
        "GARCH": "GARCH(1,1)",
        "GARCH(1,1)": "GARCH(1,1)",
        "Student-t GARCH": "Student-t GARCH",
        "Student-t GARCH(1,1)": "Student-t GARCH",
        "Zero Return": "Zero",
        "Zero-return": "Zero",
        "Zero": "Zero",
    }

    return aliases.get(s, s)


# ============================================================
# NORMALISE DATASET NAMES
# ============================================================

def normalize_dataset(s):

    s = str(s).strip().lower()

    s = unicodedata.normalize(
        "NFKD", s
    )

    s = "".join(
        c for c in s
        if not unicodedata.combining(c)
    )

    # Remove supervised-dataset suffixes.
    # This is needed for the empirical baseline.

    s = re.sub(
        r"_no_preprocess_lookback[-_]\d+"
        r"_horizon[-_]\d+"
        r"_slide[-_]\d+.*$",
        "",
        s,
    )

    s = s.replace("historical data", "")
    s = s.replace("historical", "")

    s = re.sub(
        r"[^a-z0-9]+",
        "",
        s
    )

    return s


def prepare_keys(df):

    df = df.copy()

    df["asset_class"] = (
        df["asset_class"]
        .astype(str)
        .str.strip()
    )

    df["dataset_key"] = (
        df["dataset"]
        .apply(normalize_dataset)
    )

    return df


# ============================================================
# VALIDATION HELPERS
# ============================================================

KEYS = ["asset_class", "dataset_key"]


def check_unique(df, keys, label):

    duplicated = df.duplicated(
        subset=keys,
        keep=False
    )

    if duplicated.any():

        print(f"\nDUPLICATES IN {label}")

        display(
            df.loc[duplicated]
            .sort_values(keys)
        )

        raise ValueError(
            f"Duplicate keys detected: {label}"
        )


def check_coverage(reference, other, label):

    ref_keys = reference[KEYS].drop_duplicates()

    other_keys = other[KEYS].drop_duplicates()

    missing = ref_keys.merge(
        other_keys,
        on=KEYS,
        how="left",
        indicator=True
    )

    missing = missing[
        missing["_merge"] == "left_only"
    ]

    if len(missing):

        print(f"\nMISSING DATASETS: {label}")

        display(missing)

        raise ValueError(
            f"Incomplete dataset coverage: {label}"
        )


def check_models(df, expected_models, label):

    actual = set(df["model"].unique())

    missing = expected_models - actual
    unexpected = actual - expected_models

    if missing or unexpected:

        raise ValueError(
            f"{label}\n"
            f"Missing models: {sorted(missing)}\n"
            f"Unexpected models: {sorted(unexpected)}"
        )


# ============================================================
# LOAD RESULTS
# ============================================================

qss = prepare_keys(
    pd.read_csv(QSS_PATH)
)

crps = prepare_keys(
    pd.read_csv(CRPS_PATH)
)

empirical = prepare_keys(
    pd.read_csv(EMPIRICAL_PATH)
)

mase = prepare_keys(
    pd.read_csv(MASE_PATH)
)

for df in [qss, crps, empirical, mase]:

    df["model"] = (
        df["model"].apply(normalize_model)
    )


# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

required = {
    "QSS": (
        qss,
        ["model", "asset_class", "dataset", "qss_0.1"]
    ),
    "CRPS": (
        crps,
        ["model", "asset_class", "dataset", "mean_qcrps"]
    ),
    "Empirical CRPS": (
        empirical,
        ["model", "asset_class", "dataset", "mean_qcrps"]
    ),
    "MASE": (
        mase,
        ["model", "asset_class", "dataset", "mase_250"]
    ),
}

for label, (df, columns) in required.items():

    missing = [
        c for c in columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{label}: missing columns {missing}"
        )


# ============================================================
# FILTER ELIGIBLE MODELS
# ============================================================

qss = qss[
    qss["model"].isin(PROB_MODELS)
].copy()

crps = crps[
    crps["model"].isin(PROB_MODELS)
].copy()

mase_models = mase[
    mase["model"].isin(POINT_MODELS)
].copy()

zero = mase[
    mase["model"] == "Zero"
].copy()


# ============================================================
# CHECK MODEL COVERAGE
# ============================================================

check_models(
    qss,
    PROB_MODELS,
    "QSS"
)

check_models(
    crps,
    PROB_MODELS,
    "CRPS"
)

check_models(
    mase_models,
    POINT_MODELS,
    "MASE"
)

if len(zero) == 0:
    raise ValueError(
        "Zero-return baseline not found in MASE file."
    )

if not (
    empirical["model"] == "Empirical Quantiles"
).all():

    raise ValueError(
        "Unexpected empirical baseline model labels."
    )


# ============================================================
# CHECK DUPLICATES
# ============================================================

for label, df in [
    ("QSS", qss),
    ("CRPS", crps),
    ("MASE", mase_models),
]:

    check_unique(
        df,
        KEYS + ["model"],
        label
    )

check_unique(
    empirical,
    KEYS,
    "Empirical baseline"
)

check_unique(
    zero,
    KEYS,
    "Zero baseline"
)


# ============================================================
# USE QSS DATASETS AS REFERENCE
# ============================================================

datasets = (
    qss[
        ["asset_class", "dataset", "dataset_key"]
    ]
    .drop_duplicates(subset=KEYS)
    .copy()
)

if len(datasets) != EXPECTED_DATASETS:

    raise ValueError(
        f"Expected {EXPECTED_DATASETS} datasets, "
        f"found {len(datasets)}"
    )


# ============================================================
# VALIDATE COVERAGE
# ============================================================

for label, df in [
    ("CRPS", crps),
    ("MASE", mase_models),
    ("Empirical", empirical),
    ("Zero", zero),
]:

    check_coverage(
        datasets,
        df,
        label
    )

# Check that every dataset has every eligible model.

for label, df, expected in [
    ("QSS", qss, PROB_MODELS),
    ("CRPS", crps, PROB_MODELS),
    ("MASE", mase_models, POINT_MODELS),
]:

    counts = (
        df.groupby(KEYS)["model"]
        .nunique()
    )

    if (
        len(counts) != EXPECTED_DATASETS
        or not counts.eq(len(expected)).all()
    ):

        print(f"\nINCOMPLETE MODEL COVERAGE: {label}")

        display(counts)

        raise ValueError(
            f"{label}: incomplete model coverage"
        )


# ============================================================
# NUMERIC VALIDATION
# ============================================================

for df, column, label in [
    (qss, "qss_0.1", "QSS"),
    (crps, "mean_qcrps", "CRPS"),
    (empirical, "mean_qcrps", "Empirical CRPS"),
    (mase_models, "mase_250", "MASE"),
    (zero, "mase_250", "Zero MASE"),
]:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    if not np.isfinite(
        df[column].to_numpy()
    ).all():

        raise ValueError(
            f"{label}: non-finite values"
        )


# ============================================================
# BASELINES
# ============================================================

empirical_ref = (
    empirical[
        KEYS + ["mean_qcrps"]
    ]
    .rename(
        columns={
            "mean_qcrps": "empirical_crps"
        }
    )
)

zero_ref = (
    zero[
        KEYS + ["mase_250"]
    ]
    .rename(
        columns={
            "mase_250": "zero_mase"
        }
    )
)

if (
    empirical_ref["empirical_crps"] <= 0
).any():

    raise ValueError(
        "Empirical CRPS must be positive."
    )

if (
    zero_ref["zero_mase"] <= 0
).any():

    raise ValueError(
        "Zero MASE must be positive."
    )


# ============================================================
# ADD CRPSS
# ============================================================

crps = crps.merge(
    empirical_ref,
    on=KEYS,
    how="left",
    validate="many_to_one"
)

if crps["empirical_crps"].isna().any():
    raise ValueError(
        "Missing empirical CRPS after merge."
    )

crps["crpss"] = (
    1.0
    - crps["mean_qcrps"]
    / crps["empirical_crps"]
)


# ============================================================
# ADD MASESS
# ============================================================

# mase_models = mase_models.merge(
#     zero_ref,
#     on=KEYS,
#     how="left",
#     validate="many_to_one"
# )

# if mase_models["zero_mase"].isna().any():
#     raise ValueError(
#         "Missing zero MASE after merge."
#     )

# mase_models["masess"] = (
#     1.0
#     - mase_models["mase_250"]
#     / mase_models["zero_mase"]
# )


mase_models["zero_mase"] = pd.to_numeric(
    mase_models["zero_mase"],
    errors="coerce"
)

# Validate the existing baseline.
if (
    not np.isfinite(mase_models["zero_mase"]).all()
    or (mase_models["zero_mase"] <= 0).any()
):
    raise ValueError(
        "Invalid zero-return MASE baseline."
    )

# Calculate skill relative to zero return.
mase_models["masess"] = (
    1.0
    - mase_models["mase_250"]
    / mase_models["zero_mase"]
)

print("MASESS calculated successfully!")

display(
    mase_models[
        ["model", "mase_250", "zero_mase", "masess"]
    ].head()
)


# ============================================================
# BEST MODEL HELPER
# ============================================================

def best_by_dataset(
    df,
    score_col,
    prefix,
    higher_is_better,
    tolerance_col=None,
):

    """
    Select the best model independently for each dataset.

    Within-1% counts use the raw loss where available.

    For QSS, use an absolute tolerance of
    1% of the best score's magnitude.
    """

    output_rows = []

    for keys, group in df.groupby(
        KEYS,
        sort=False
    ):

        group = group.copy()

        # if higher_is_better:

        #     best_score = group[score_col].max()

        #     winners = group[
        #         np.isclose(
        #             group[score_col],
        #             best_score,
        #             rtol=1e-12,
        #             atol=1e-14
        #         )
        #     ]

        # else:

        #     best_score = group[score_col].min()

        #     winners = group[
        #         np.isclose(
        #             group[score_col],
        #             best_score,
        #             rtol=1e-12,
        #             atol=1e-14
        #         )
        #     ]

        # best_model = sorted(
        #     winners["model"].tolist()
        # )[0]

        if higher_is_better:
            best_idx = group[score_col].idxmax()
        else:
            best_idx = group[score_col].idxmin()
        
        best_score = group.loc[
            best_idx, score_col
        ]
        
        best_model = group.loc[
            best_idx, "model"
        ]

        # ----------------------------------------
        # WITHIN 1%
        # ----------------------------------------

        if tolerance_col is not None:

            # Use corresponding raw loss.
            # Lower is better.

            best_raw = group[
                tolerance_col
            ].min()

            threshold = (
                best_raw * (1 + WITHIN_PCT)
            )

            n_within = int(
                (
                    group[tolerance_col]
                    <= threshold
                ).sum()
            )

        elif higher_is_better:

            # QSS can be negative.
            # Therefore, avoid multiplying the
            # best score directly by 0.99.

            tolerance = (
                WITHIN_PCT * abs(best_score)
            )

            n_within = int(
                (
                    group[score_col]
                    >= best_score - tolerance
                ).sum()
            )

        else:

            threshold = (
                best_score * (1 + WITHIN_PCT)
            )

            n_within = int(
                (
                    group[score_col]
                    <= threshold
                ).sum()
            )

        output_rows.append({
            "asset_class": keys[0],
            "dataset_key": keys[1],
            f"best_{prefix}": best_score,
            f"best_{prefix}_model": best_model,
            f"{prefix}_within_1pct": n_within,
        })

    return pd.DataFrame(output_rows)


# ============================================================
# COMPUTE BEST VALUES
# ============================================================

best_qss = best_by_dataset(
    qss,
    score_col="qss_0.1",
    prefix="qss",
    higher_is_better=True,
)

best_crps = best_by_dataset(
    crps,
    score_col="mean_qcrps",
    prefix="crps",
    higher_is_better=False,
)

# best_crpss = best_by_dataset(
#     crps,
#     score_col="crpss",
#     prefix="crpss",
#     higher_is_better=True,
#     tolerance_col="mean_qcrps",
# )

best_crpss = best_by_dataset(
    crps,
    score_col="crpss",
    prefix="crpss",
    higher_is_better=True,
)

best_mase = best_by_dataset(
    mase_models,
    score_col="mase_250",
    prefix="mase",
    higher_is_better=False,
)

# best_masess = best_by_dataset(
#     mase_models,
#     score_col="masess",
#     prefix="masess",
#     higher_is_better=True,
#     tolerance_col="mase_250",
# )

best_masess = best_by_dataset(
    mase_models,
    score_col="masess",
    prefix="masess",
    higher_is_better=True,
)


# ============================================================
# COMBINE ALL COLUMNS
# ============================================================

result = datasets.copy()

components = [
    best_crps,
    best_qss,
    best_mase,
    best_masess,
    best_crpss,
    empirical_ref,
    zero_ref,
]

for component in components:

    result = result.merge(
        component,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )


# ============================================================
# FINAL COLUMN ORDER
# ============================================================

result = result[
    [
        "asset_class",
        "dataset",

        "best_crps",
        "best_crps_model",
        "crps_within_1pct",

        "best_qss",
        "best_qss_model",
        "qss_within_1pct",

        "best_mase",
        "best_mase_model",
        "mase_within_1pct",

        "best_masess",
        "best_masess_model",
        "masess_within_1pct",

        "best_crpss",
        "best_crpss_model",
        "crpss_within_1pct",

        "empirical_crps",
        "zero_mase",
    ]
]


# ============================================================
# FINAL VALIDATION
# ============================================================

if len(result) != EXPECTED_DATASETS:

    raise ValueError(
        f"Expected {EXPECTED_DATASETS} rows, "
        f"got {len(result)}"
    )

if result.isna().any().any():

    print(
        result.loc[
            result.isna().any(axis=1)
        ]
    )

    raise ValueError(
        "Missing values in final table."
    )


# MASE and MASESS should have identical winners.

if not (
    result["best_mase_model"]
    == result["best_masess_model"]
).all():

    raise ValueError(
        "MASE and MASESS winners differ."
    )


# CRPS and CRPSS should have identical winners.

if not (
    result["best_crps_model"]
    == result["best_crpss_model"]
).all():

    raise ValueError(
        "CRPS and CRPSS winners differ."
    )


# ============================================================
# SAVE
# ============================================================

output_path = (
    OUTPUT_DIR
    / "best_metrics_with_skill_weekly_with_timestral.csv"
)

result.to_csv(
    output_path,
    index=False
)

print(f"\nSaved: {output_path}")

print(
    f"Datasets: {len(result)}"
)

display(result)


# ============================================================
# SUMMARY: POSITIVE BEST-MODEL SKILL
# ============================================================

summary_rows = []

for metric in [
    "qss",
    "masess",
    "crpss",
]:

    values = result[f"best_{metric}"]

    summary_rows.append({
        "metric": metric.upper(),
        "n_datasets": len(values),
        "n_positive": int((values > 0).sum()),
        "n_zero": int((values == 0).sum()),
        "n_negative": int((values < 0).sum()),
        "pct_positive": (
            100 * (values > 0).mean()
        ),
        "mean_best_score": values.mean(),
        "median_best_score": values.median(),
    })

summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUTPUT_DIR
    / "best_skill_summary_weekly_with_timestral.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print("\nBEST-MODEL SKILL SUMMARY")

display(summary)

In [ ]:
# ============================================================
# EXTEND BEST-MODEL TABLE:
# RMSE, RMSESS, MSIS, MSISS
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# 1. LOAD MSIS RESULTS
# ============================================================

MSIS_PATH = (
    BASE_DIR
    / "msis_250_decomposition_weekly_with_timestral.csv"
)

msis = prepare_keys(
    pd.read_csv(MSIS_PATH)
)

msis["model"] = msis["model"].apply(normalize_model)

print("MSIS models:")
print(sorted(msis["model"].unique()))


# ============================================================
# 2. PREPARE RMSE AND RMSESS
# ============================================================

# The existing MASE results already contain:
# rmse, zero_rmse, mase_250, zero_mase

rmse_models = mase_models.copy()

for col in ["rmse", "zero_rmse"]:

    rmse_models[col] = pd.to_numeric(
        rmse_models[col],
        errors="coerce"
    )

    if not np.isfinite(
        rmse_models[col].to_numpy()
    ).all():
        raise ValueError(
            f"Non-finite values in {col}"
        )

if (rmse_models["zero_rmse"] <= 0).any():
    raise ValueError(
        "Zero-return RMSE must be positive."
    )

if (rmse_models["rmse"] < 0).any():
    raise ValueError(
        "Negative RMSE detected."
    )

# RMSE skill score relative to zero return

rmse_models["rmsess"] = (
    1.0
    - rmse_models["rmse"]
    / rmse_models["zero_rmse"]
)

# Dataset-level reference

zero_rmse_ref = (
    rmse_models[
        KEYS + ["zero_rmse"]
    ]
    .drop_duplicates()
)

check_unique(
    zero_rmse_ref,
    KEYS,
    "Zero RMSE reference"
)


# ============================================================
# 3. PREPARE MSIS AND MSISS
# ============================================================

required_msis_cols = [
    "model",
    "asset_class",
    "dataset",
    "msis_250",
    "empirical_msis",
]

missing_cols = [
    col for col in required_msis_cols
    if col not in msis.columns
]

if missing_cols:
    raise ValueError(
        f"MSIS missing columns: {missing_cols}"
    )

# Only probabilistic forecasting models.
# Exclude the empirical baseline from best-model selection.

msis_models = msis[
    msis["model"].isin(PROB_MODELS)
].copy()

check_models(
    msis_models,
    PROB_MODELS,
    "MSIS"
)

check_unique(
    msis_models,
    KEYS + ["model"],
    "MSIS"
)

check_coverage(
    datasets,
    msis_models,
    "MSIS"
)

# Confirm all eligible models are present
# for every dataset.

msis_counts = (
    msis_models
    .groupby(KEYS)["model"]
    .nunique()
)

if (
    len(msis_counts) != EXPECTED_DATASETS
    or not msis_counts.eq(len(PROB_MODELS)).all()
):
    raise ValueError(
        "Incomplete MSIS model coverage."
    )

# Convert scores to numeric

for col in ["msis_250", "empirical_msis"]:

    msis_models[col] = pd.to_numeric(
        msis_models[col],
        errors="coerce"
    )

    if not np.isfinite(
        msis_models[col].to_numpy()
    ).all():
        raise ValueError(
            f"Non-finite MSIS values in {col}"
        )

if (msis_models["empirical_msis"] <= 0).any():
    raise ValueError(
        "Empirical MSIS must be positive."
    )

if (msis_models["msis_250"] < 0).any():
    raise ValueError(
        "Negative MSIS detected."
    )

# Ensure the empirical baseline is consistent
# across models within each dataset.

baseline_counts = (
    msis_models
    .groupby(KEYS)["empirical_msis"]
    .nunique()
)

if not baseline_counts.eq(1).all():
    raise ValueError(
        "Inconsistent empirical MSIS references "
        "within one or more datasets."
    )

# Calculate MSIS skill score

msis_models["msiss"] = (
    1.0
    - msis_models["msis_250"]
    / msis_models["empirical_msis"]
)

# Dataset-level empirical MSIS reference

empirical_msis_ref = (
    msis_models[
        KEYS + ["empirical_msis"]
    ]
    .drop_duplicates()
)

check_unique(
    empirical_msis_ref,
    KEYS,
    "Empirical MSIS reference"
)


# ============================================================
# 4. BEST RMSE AND RMSESS
# ============================================================

best_rmse = best_by_dataset(
    rmse_models,
    score_col="rmse",
    prefix="rmse",
    higher_is_better=False,
)

best_rmsess = best_by_dataset(
    rmse_models,
    score_col="rmsess",
    prefix="rmsess",
    higher_is_better=True,
)


# ============================================================
# 5. BEST MSIS AND MSISS
# ============================================================

best_msis = best_by_dataset(
    msis_models,
    score_col="msis_250",
    prefix="msis",
    higher_is_better=False,
)

best_msiss = best_by_dataset(
    msis_models,
    score_col="msiss",
    prefix="msiss",
    higher_is_better=True,
)


# ============================================================
# 6. COMBINE WITH EXISTING RESULT
# ============================================================

# Begin with the previously generated table.
# Remove any extra columns from rerunning this extension
# to avoid pandas creating _x and _y suffixes.

new_components = [
    best_rmse,
    best_rmsess,
    best_msis,
    best_msiss,
    zero_rmse_ref,
    empirical_msis_ref,
]

new_columns = sorted({
    col
    for component in new_components
    for col in component.columns
    if col not in KEYS
})

# ============================================================
# 6. COMBINE WITH EXISTING RESULT — FIXED
# ============================================================

# Start from the previously generated table.
result_extended = result.copy()

# Reconstruct dataset_key using our existing
# dataset-name normalisation function.
result_extended["dataset_key"] = (
    result_extended["dataset"]
    .apply(normalize_dataset)
)

# Verify dataset identifiers are unique.
check_unique(
    result_extended,
    KEYS,
    "Existing best-model table"
)

# Remove columns that might already exist
# if this extension has been run before.
new_columns = sorted({
    col
    for component in new_components
    for col in component.columns
    if col not in KEYS
})

result_extended = result_extended.drop(
    columns=new_columns,
    errors="ignore"
)

# Merge the additional metrics.
for component in new_components:

    # Check that the component has both join keys.
    missing_keys = [
        key for key in KEYS
        if key not in component.columns
    ]

    if missing_keys:
        raise ValueError(
            f"Component missing join keys: {missing_keys}"
        )

    result_extended = result_extended.merge(
        component,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

print("All additional metrics merged successfully!")


# ============================================================
# 7. VERIFY BEST-MODEL CONSISTENCY
# ============================================================

# Raw loss and its corresponding skill score
# must select the same best model.

model_pairs = [
    ("rmse", "rmsess"),
    ("mase", "masess"),
    ("crps", "crpss"),
    ("msis", "msiss"),
]

for raw_metric, skill_metric in model_pairs:

    raw_col = f"best_{raw_metric}_model"
    skill_col = f"best_{skill_metric}_model"

    if not (
        result_extended[raw_col]
        == result_extended[skill_col]
    ).all():

        raise ValueError(
            f"{raw_metric.upper()} and "
            f"{skill_metric.upper()} "
            "best models do not match."
        )

print(
    "\nAll raw-loss and skill-score "
    "best-model identities match."
)


# ============================================================
# 8. FINAL COLUMN ORDER
# ============================================================

column_order = [
    "asset_class",
    "dataset",

    # RMSE
    "best_rmse",
    "best_rmse_model",
    "rmse_within_1pct",

    # RMSESS
    "best_rmsess",
    "best_rmsess_model",
    "rmsess_within_1pct",

    # MASE
    "best_mase",
    "best_mase_model",
    "mase_within_1pct",

    # MASESS
    "best_masess",
    "best_masess_model",
    "masess_within_1pct",

    # QSS
    "best_qss",
    "best_qss_model",
    "qss_within_1pct",

    # CRPS
    "best_crps",
    "best_crps_model",
    "crps_within_1pct",

    # CRPSS
    "best_crpss",
    "best_crpss_model",
    "crpss_within_1pct",

    # MSIS
    "best_msis",
    "best_msis_model",
    "msis_within_1pct",

    # MSISS
    "best_msiss",
    "best_msiss_model",
    "msiss_within_1pct",

    # Reference scores
    "zero_rmse",
    "zero_mase",
    "empirical_crps",
    "empirical_msis",
]

missing_output_cols = [
    col for col in column_order
    if col not in result_extended.columns
]

if missing_output_cols:
    raise ValueError(
        f"Missing output columns: {missing_output_cols}"
    )

result_extended = result_extended[
    column_order
]


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

if len(result_extended) != EXPECTED_DATASETS:
    raise ValueError(
        f"Expected {EXPECTED_DATASETS} datasets, "
        f"got {len(result_extended)}"
    )

if result_extended.isna().any().any():

    display(
        result_extended.loc[
            result_extended.isna().any(axis=1)
        ]
    )

    raise ValueError(
        "Missing values in extended table."
    )


# ============================================================
# 10. SAVE EXTENDED TABLE
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    OUTPUT_DIR
    / "best_metrics_all_weekly_with_timestral.csv"
)

result_extended.to_csv(
    output_path,
    index=False
)

print("\nSaved extended table:")
print(output_path)

display(result_extended)


# ============================================================
# 11. BEST-MODEL SKILL SUMMARY
# ============================================================

skill_metrics = [
    "rmsess",
    "masess",
    "qss",
    "crpss",
    "msiss",
]

summary_rows = []

for metric in skill_metrics:

    values = result_extended[
        f"best_{metric}"
    ]

    summary_rows.append({
        "metric": metric.upper(),
        "n_datasets": len(values),
        "n_positive": int((values > 0).sum()),
        "n_zero": int((values == 0).sum()),
        "n_negative": int((values < 0).sum()),
        "pct_positive": (
            100 * (values > 0).mean()
        ),
        "mean_best_score": float(
            values.mean()
        ),
        "median_best_score": float(
            values.median()
        ),
    })

summary_extended = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUTPUT_DIR
    / "best_skill_summary_all_weekly_with_timestral.csv"
)

summary_extended.to_csv(
    summary_path,
    index=False
)

print("\nBEST-MODEL SKILL SUMMARY")

display(summary_extended)

# Table

In [ ]:
# ------------------------------------------------------------
# 1. Load final best-model table
# ------------------------------------------------------------

best_messy = pd.read_csv(
    r"BForTFin_results/standardised/output_daily/best_model_tables/best_metrics_all_daily_with_timestral.csv"
)

best = (
    best_messy[
        [
            "asset_class",
            "dataset",
            "best_crps",
            "best_crps_model",
            "crps_within_1pct",
            "best_crpss",
            "best_crpss_model",
            "crpss_within_1pct",
            "empirical_crps",
            "best_qss",
            "best_qss_model",
            "qss_within_1pct",
            "best_mase",
            "best_mase_model",
            "mase_within_1pct",
            "best_masess",
            "best_masess_model",
            "masess_within_1pct",
            "zero_mase",
            "best_rmse",
            "best_rmse_model",
            "rmse_within_1pct",
            "best_rmsess",
            "best_rmsess_model",
            "rmsess_within_1pct",
            "best_msis",
            "best_msis_model",
            "msis_within_1pct",
            "best_msiss",
            "best_msiss_model",
            "msiss_within_1pct",
            "empirical_msis",
            "zero_rmse",
        ]
    ]
    .rename(
        columns={
            "asset_class": "Class",
            "dataset": "Dataset",
            "best_crps": "Best CRPS",
            "best_crps_model": "Best Model (CRPS)",
            "crps_within_1pct": "Models within 1% (CRPS)",
            "best_crpss": "Best CRPSS",
            "best_crpss_model": "Best Model (CRPSS)",
            "crpss_within_1pct": "Models within 1% (CRPSS)",
            "empirical_crps": "Empirical CRPS",
            "empirical_msis": "Empirical MSIS",
            "best_qss": "Best QSS",
            "best_qss_model": "Best Model (QSS)",
            "qss_within_1pct": "Models within 1% (QSS)",
            "best_mase": "Best MASE",
            "best_mase_model": "Best Model (MASE)",
            "mase_within_1pct": "Models within 1% (MASE)",
            "best_masess": "Best MASESS",
            "best_masess_model": "Best Model (MASESS)",
            "masess_within_1pct": "Models within 1% (MASESS)",
            "zero_mase": "Zero Return MASE",
            "best_rmse": "Best RMSE",
            "best_rmse_model": "Best Model (RMSE)",
            "rmse_within_1pct": "Models within 1% (RMSE)",
            "best_rmsess": "Best RMSESS",
            "best_rmsess_model": "Best Model (RMSESS)",
            "rmsess_within_1pct": "Models within 1% (RMSESS)",
            "best_msis": "Best MSIS",
            "best_msis_model": "Best Model (MSIS)",
            "msis_within_1pct": "Models within 1% (MSIS)",
            "best_msiss": "Best MSISS",
            "best_msiss_model": "Best Model (MSISS)",
            "msiss_within_1pct": "Models within 1% (MSISS)",
            "zero_rmse": "Zero Return RMSE",
        }
    )
)


# ------------------------------------------------------------
# 2. Create the original benchmark table as a dataframe
# ------------------------------------------------------------

benchmark_rows = [
    ["FX", "AUDUSD", 2739, 0.397, 0.004, 9.866],
    ["FX", "DXY", 2710, 0.405, 0.008, 6.783],
    ["FX", "EURUSD", 2739, 0.446, 0.009, 7.52],
    ["FX", "GBPUSD", 2739, 0.361, 0.005, 8.843],
    ["FX", "NZDUSD", 2739, 0.397, 0.004, 10.098],
    ["FX", "USDCAD", 2739, 0.396, 0.007, 6.996],
    ["FX", "USDCHF", 2739, 0.512, 0.003, 7.657],
    ["FX", "USDJPY", 2739, 0.398, 0.007, 8.52],

    ["EM FX", "USDAED", 3026, 0.361, 0.001, 0.081],
    ["EM FX", "USDBRL", 2738, 0.393, 0.009, 15.632],
    ["EM FX", "USDCNH", 2738, 0.382, 0.008, 4.601],
    ["EM FX", "USDIDR", 2673, 0.408, 0.005, 5.484],
    ["EM FX", "USDINR", 2727, 0.378, 0.006, 4.659],
    ["EM FX", "USDKRW", 2728, 0.384, 0.003, 8.33],
    ["EM FX", "USDMXN", 2739, 0.379, 0.005, 12.239],
    ["EM FX", "USDMYR", 2684, 0.478, 0.019, 5.295],
    ["EM FX", "USDQAR", 3349, 0.356, 0.000, 3.739],
    ["EM FX", "USDSAR", 3611, 0.372, 0.001, 0.253],
    ["EM FX", "USDSGD", 2728, 0.419, 0.006, 4.683],
    ["EM FX", "USDTHB", 2728, 0.438, 0.009, 5.778],
    ["EM FX", "USDVND", 2728, 0.462, 0.009, 1.976],
    ["EM FX", "USDZAR", 2739, 0.362, 0.004, 15.147],

    ["Rates", "Bund Futures", 2683, 0.36, 0.006, 6.626],
    ["Rates", "JGB Futures", 2790, 0.398, 0.003, 2.253],
    ["Rates", "Malaysia 5Y", 3085, 0.338, 0.012, 11.742],
    ["Rates", "Singapore 2Y", 3093, 0.426, 0.015, 43.956],
    ["Rates", "Singapore 5Y", 3111, 0.416, 0.013, 35.015],
    ["Rates", "UK Gilt", 2671, 0.35, 0.004, 7.772],
    ["Rates", "U.S. 10Y", 2733, 0.383, 0.006, 39.075],
    ["Rates", "U.S. 1Y", 2724, 0.401, 0.033, 49.554],
    ["Rates", "U.S. 2Y", 2727, 0.378, 0.021, 52.324],
    ["Rates", "U.S. 5Y", 2726, 0.382, 0.010, 48.582],

    ["REIT", "Dow Jones Equity", 2640, 0.369, 0.002, 18.353],
    ["REIT", "FTSE RAFI US 100", 2638, 0.37, 0.003, 18.885],
    ["REIT", "Hui Xian REIT", 2581, 0.354, 0.008, 19.727],
    ["REIT", "Wheeler REIT", 2419, 0.355, 0.005, 59.852],
    ["REIT", "Yuexiu REIT", 2581, 0.369, 0.004, 22.931],

    ["ETF", "EWM Malaysia", 2669, 0.452, 0.006, 17.068],
    ["ETF", "HUN Global Nat. Gas", 2111, 0.367, 0.009, 34.839],
    ["ETF", "REET Global REIT", 2666, 0.362, 0.004, 16.685],
    ["ETF", "VEU World ex. US", 2666, 0.384, 0.005, 15.839],
    ["ETF", "VWO EM", 2667, 0.399, 0.005, 18.661],

    ["MSCI", "Argentina", 2650, 0.38, 0.006, 38.388],
    ["MSCI", "Brazil", 2729, 0.663, 0.009, 37.839],
    ["MSCI", "China", 2704, 0.402, 0.007, 22.821],
    ["MSCI", "Colombia", 2729, 0.387, 0.006, 27.589],
    ["MSCI", "Denmark", 2628, 0.422, 0.011, 19.547],
    ["MSCI", "Egypt", 2097, 0.402, 0.012, 27.529],
    ["MSCI", "Eastern Europe", 2731, 0.363, 0.016, 25.629],
    ["MSCI", "France", 2688, 0.4, 0.004, 16.708],
    ["MSCI", "Germany", 2665, 0.42, 0.005, 17.361],
    ["MSCI", "India", 2591, 0.394, 0.006, 14.963],
    ["MSCI", "Indonesia", 2533, 0.413, 0.003, 18.868],
    ["MSCI", "Japan Net JPY", 2562, 0.432, 0.004, 18.184],
    ["MSCI", "Malaysia", 2567, 0.414, 0.002, 10.437],
    ["MSCI", "Philippines", 2556, 0.389, 0.003, 18.997],
    ["MSCI", "Singapore", 2702, 0.387, 0.009, 13.656],
    ["MSCI", "Sweden", 2653, 0.396, 0.004, 17.443],
    ["MSCI", "Thailand", 2551, 0.39, 0.006, 14.622],
    ["MSCI", "United Kingdom", 2656, 0.404, 0.003, 14.249],
    ["MSCI", "USA", 2642, 0.394, 0.003, 16.032],

    ["Equities", "BKGKZ", 2651, 0.375, 0.008, 28.639],
    ["Equities", "DJI", 2639, 0.391, 0.002, 15.195],
    ["Equities", "Euro Stoxx 50", 2691, 0.409, 0.004, 17.469],
    ["Equities", "NASDAQ", 2638, 0.381, 0.004, 19.885],
    ["Equities", "SPX", 2638, 0.385, 0.003, 15.881],
    ["Equities", "SGX", 2629, 0.419, 0.005, 17.442],

    ["Crypto", "Bitcoin", 3834, 0.382, 0.011, 53.223],
    ["Crypto", "BNB", 2791, 0.442, 0.021, 70.905],
    ["Crypto", "Ethereum", 2716, 0.399, 0.011, 68.2],
    ["Crypto", "Solana", 1807, 0.492, 0.022, 97.622],
    ["Crypto", "Tether", 2998, 0.675, 0.012, 7.274],
    ["Crypto", "XRP", 3812, 0.474, 0.011, 109.622],
]

benchmark = pd.DataFrame(
    benchmark_rows,
    columns=[
        "Class",
        "Dataset",
        "Samples",
        "Seasonality",
        "Trend",
        "Volatility",
    ]
)

# Normalise names for matching
import re
import pandas as pd
import unicodedata


def normalize_name(s):
    s = str(s).lower().strip()

    # remove accents / odd unicode
    s = unicodedata.normalize("NFKD", s)
    s = "".join(
        c for c in s
        if not unicodedata.combining(c)
    )

    # common irrelevant words in filenames
    remove_phrases = [
        "historical data",
        "historical",
        "index",
        "price",
    ]

    for phrase in remove_phrases:
        s = s.replace(phrase, "")

    # normalize a few common forms
    s = s.replace("united states", "us")
    s = s.replace("u.s.", "us")
    s = s.replace("s&p", "sp")

    # remove punctuation/spaces
    s = re.sub(r"[^a-z0-9]+", "", s)

    return s

ALIASES = {
    "BKGKZ": [
        "BNY Kazakhstan",
        "BNY Mellon Kazakhstan",
    ],

    "SPX": [
        "S&P 500",
        "S&P500",
        "SP500",
    ],

    "U.S. 1Y": [
        "United States 1Y Bond Yield",
    ],

    "U.S. 2Y": [
        "United States 2Y Bond Yield",
    ],

    "U.S. 5Y": [
        "United States 5Y Bond Yield",
    ],

    "U.S. 10Y": [
        "United States 10Y Bond Yield",
    ],

    "JGB Futures": [
        "Japan Gov. Bond Futures"
    ],

   "Hui Xian REIT" : [
       "Hui Xian Real Estate Stock History"
   ],

    "Wheeler REIT": [
        "Wheeler Real Estate Invest Pref Stock History"
    ],

    "Yuexiu REIT" : [
        "Yuexiu Real Estate Stock History"
    ],

    "EWM Malaysia" : [
        "EWM ETF Stock History"
    ],

    "HUN Global Nat. Gas" : [
        "HUN ETF Stock History"
    ],

    "REET Global REIT" : [
        "REET ETF Stock History"
    ],

    "VWO EM" : [
        "VWO ETF Stock History"
    ],

    "VEU World ex. US" : [
        "VEU ETF Stock Price History"
    ],

    "Eastern Europe" : [
        "MSCI EM Eastern Europe"
    ],

    "Malaysia" : [
        "MSCI Malaysia Historical Data_no_preprocess_lookback-250_horizon-20_slide-1"
    ],

    "Singapore" : [
        "MSCI Singapore Historical Data_no_preprocess_lookback-250_horizon-20_slide-1"
    ],

    "Euro Stoxx 50" : [
        "Euro Historical Data_no_preprocess_lookback-250_horizon-20_slide-1"
    ],
}

def match_dataset_name(
    benchmark_name,
    available_result_names
):
    """
    Returns the matching result dataset name.

    Priority:
    1. Explicit alias
    2. Exact normalized match
    3. Substring containment
    """

    # -----------------------------------------
    # 1. Explicit aliases
    # -----------------------------------------
    candidates = [benchmark_name]

    if benchmark_name in ALIASES:
        candidates.extend(
            ALIASES[benchmark_name]
        )

    candidate_keys = [
        normalize_name(x)
        for x in candidates
    ]

    # normalize all available names once
    result_keys = {
        result_name: normalize_name(result_name)
        for result_name in available_result_names
    }

    # -----------------------------------------
    # 2. Exact normalized match
    # -----------------------------------------
    for candidate_key in candidate_keys:

        exact = [
            original
            for original, key
            in result_keys.items()
            if key == candidate_key
        ]

        if len(exact) == 1:
            return exact[0]

    # -----------------------------------------
    # 3. Containment match
    # -----------------------------------------
    matches = []

    for candidate_key in candidate_keys:

        for original, result_key in result_keys.items():

            if (
                candidate_key in result_key
                or result_key in candidate_key
            ):
                matches.append(original)

    matches = list(
        dict.fromkeys(matches)
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        print(
            f"AMBIGUOUS: {benchmark_name} "
            f"-> {matches}"
        )
        return None

    print(
        f"NO MATCH: {benchmark_name}"
    )

    return None

# Merge
available_result_names = (
    best["Dataset"]
    .dropna()
    .unique()
    .tolist()
)

benchmark["matched_result_dataset"] = (
    benchmark["Dataset"].apply(
        lambda x: match_dataset_name(
            x,
            available_result_names
        )
    )
)

display(benchmark)

unmatched = benchmark[
    benchmark[
        "matched_result_dataset"
    ].isna()
]

display(unmatched)

chronos_summary = pd.read_csv(
    r"BForTFin_results/standardised/output_daily/chronos_standardized_train_test/chronos_train_test_summary_all_datasets.csv"
)

sample_counts = (
    chronos_summary[
        [
            "asset_class",
            "dataset",
            "n_total_windows",
            "n_test_windows",
        ]
    ]
    .rename(
        columns={
            "asset_class": "Class",
            "dataset": "Dataset",
            "n_total_windows": "Samples",
            "n_test_windows": "Test Samples",
        }
    )
)

benchmark = benchmark.drop(
    columns=["Samples"]
).merge(
    sample_counts,
    left_on="matched_result_dataset",
    right_on="Dataset",
    how="left",
    suffixes=("", "_results")
)

benchmark = benchmark[
    [
        "Class",
        "Dataset",
        "Test Samples",
        "matched_result_dataset"
    ]
]

display(benchmark)

merged = benchmark.merge(
    best,
    left_on="matched_result_dataset",
    right_on="Dataset",
    how="left",
    suffixes=("", "_results")
)
display(merged)

merged = merged.drop(
    columns=[
        "matched_result_dataset",
        "Dataset_results",
        "Class_results",
        "_dataset_key",
        "_class_key",
        "_dataset_key_results",
        "_class_key_results",
    ],
    errors="ignore"
)

merged = merged[
    [
        "Class",
        "Dataset",

        # new column
        # "Test Samples",

        # CRPS
        "Empirical CRPS",
        "Best CRPS",
        "Best Model (CRPS)",
        "Models within 1% (CRPS)",

        "Best CRPSS",
        "Best Model (CRPSS)",
        "Models within 1% (CRPSS)",
        "Best QSS",
        "Best Model (QSS)",
        "Models within 1% (QSS)",
        "Empirical MSIS",
        "Best MSIS",
        "Best Model (MSIS)",
        "Models within 1% (MSIS)",

        "Best MSISS",
        "Best Model (MSISS)",
        "Models within 1% (MSISS)",
        "Zero Return MASE",
        "Best MASE",
        "Best Model (MASE)",
        "Models within 1% (MASE)",
        "Best MASESS",
        "Best Model (MASESS)",
        "Models within 1% (MASESS)",
        "Zero Return RMSE",
        "Best RMSE",
        "Best Model (RMSE)",
        "Models within 1% (RMSE)",
        "Best RMSESS",
        "Best Model (RMSESS)",
        "Models within 1% (RMSESS)",
    ]
]

display(merged)

# Format values
merged["Best QSS"] = merged[
    "Best QSS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best CRPS"] = merged[
    "Best CRPS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best CRPSS"] = merged[
    "Best CRPSS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MASESS"] = merged[
    "Best MASESS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MASE"] = merged[
    "Best MASE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MSIS"] = merged[
    "Best MSIS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MSISS"] = merged[
    "Best MSISS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best RMSESS"] = merged[
    "Best RMSESS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best RMSE"] = merged[
    "Best RMSE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Empirical CRPS"] = merged[
    "Empirical CRPS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Zero Return MASE"] = merged[
    "Zero Return MASE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Empirical MSIS"] = merged[
    "Empirical MSIS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Zero Return RMSE"] = merged[
    "Zero Return RMSE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

# merged["Test Samples"] = merged[
#     "Test Samples"
# ].map(
#         lambda x: "--"
#         if pd.isna(x)
#         else f"{x:.0f}"
#     )

for col in [
    "Best Model (QSS)",
    "Best Model (MASE)",
    "Best Model (CRPSS)",
    "Best Model (MASESS)",
    "Best Model (CRPS)",
    "Best Model (RMSE)",
    "Best Model (MSISS)",
    "Best Model (RMSESS)",
    "Best Model (MSIS)",
    
]:  
    merged[col] = (
        merged[col]
        .fillna("--")
        .replace(
            {
                "GARCH(1,1)": "GARCH",
                "Chronos-T5": "Chronos",
            }
        )
    )

for col in [
    "Models within 1% (QSS)",
    "Models within 1% (MASE)",
    "Models within 1% (CRPSS)",
    "Models within 1% (MASESS)",
    "Models within 1% (CRPS)",
    "Models within 1% (RMSE)",
    "Models within 1% (MSISS)",
    "Models within 1% (RMSESS)",
    "Models within 1% (MSIS)",
]:
    merged[col] = merged[col].map(
        lambda x: "--"
        if pd.isna(x)
        else str(int(x))
    )

    display(merged)





body = merged.loc[:,"Class":"Models within 1% (MSISS)"].to_latex(
    index=False,
    header=False,
    escape=True,
)

print(body)




body = merged.loc[:,[
    "Class", 
    "Dataset",
    "Zero Return MASE",
    "Best MASE",
    "Best Model (MASE)",
    "Models within 1% (MASE)",
    "Best MASESS",
    "Best Model (MASESS)",
    "Models within 1% (MASESS)",
    "Zero Return RMSE",
    "Best RMSE",
    "Best Model (RMSE)",
    "Models within 1% (RMSE)",
    "Best RMSESS",
    "Best Model (RMSESS)",
    "Models within 1% (RMSESS)"
                    ]].to_latex(
    index=False,
    header=False,
    escape=True,
)

print(body)

### Weekly

In [ ]:
# ------------------------------------------------------------
# 1. Load final best-model table
# ------------------------------------------------------------

best_messy = pd.read_csv(
    r"BForTFin_results/standardised/output_weekly/best_model_tables/best_metrics_all_weekly_with_timestral.csv"
)

best = (
    best_messy[
        [
            "asset_class",
            "dataset",
            "best_crps",
            "best_crps_model",
            "crps_within_1pct",
            "best_crpss",
            "best_crpss_model",
            "crpss_within_1pct",
            "empirical_crps",
            "best_qss",
            "best_qss_model",
            "qss_within_1pct",
            "best_mase",
            "best_mase_model",
            "mase_within_1pct",
            "best_masess",
            "best_masess_model",
            "masess_within_1pct",
            "zero_mase",
            "best_rmse",
            "best_rmse_model",
            "rmse_within_1pct",
            "best_rmsess",
            "best_rmsess_model",
            "rmsess_within_1pct",
            "best_msis",
            "best_msis_model",
            "msis_within_1pct",
            "best_msiss",
            "best_msiss_model",
            "msiss_within_1pct",
            "empirical_msis",
            "zero_rmse",
        ]
    ]
    .rename(
        columns={
            "asset_class": "Class",
            "dataset": "Dataset",
            "best_crps": "Best CRPS",
            "best_crps_model": "Best Model (CRPS)",
            "crps_within_1pct": "Models within 1% (CRPS)",
            "best_crpss": "Best CRPSS",
            "best_crpss_model": "Best Model (CRPSS)",
            "crpss_within_1pct": "Models within 1% (CRPSS)",
            "empirical_crps": "Empirical CRPS",
            "empirical_msis": "Empirical MSIS",
            "best_qss": "Best QSS",
            "best_qss_model": "Best Model (QSS)",
            "qss_within_1pct": "Models within 1% (QSS)",
            "best_mase": "Best MASE",
            "best_mase_model": "Best Model (MASE)",
            "mase_within_1pct": "Models within 1% (MASE)",
            "best_masess": "Best MASESS",
            "best_masess_model": "Best Model (MASESS)",
            "masess_within_1pct": "Models within 1% (MASESS)",
            "zero_mase": "Zero Return MASE",
            "best_rmse": "Best RMSE",
            "best_rmse_model": "Best Model (RMSE)",
            "rmse_within_1pct": "Models within 1% (RMSE)",
            "best_rmsess": "Best RMSESS",
            "best_rmsess_model": "Best Model (RMSESS)",
            "rmsess_within_1pct": "Models within 1% (RMSESS)",
            "best_msis": "Best MSIS",
            "best_msis_model": "Best Model (MSIS)",
            "msis_within_1pct": "Models within 1% (MSIS)",
            "best_msiss": "Best MSISS",
            "best_msiss_model": "Best Model (MSISS)",
            "msiss_within_1pct": "Models within 1% (MSISS)",
            "zero_rmse": "Zero Return RMSE",
        }
    )
)


# ------------------------------------------------------------
# 2. Create the original benchmark table as a dataframe
# ------------------------------------------------------------

benchmark_rows = [
    ["FX", "AUDUSD", 2739, 0.397, 0.004, 9.866],
    ["FX", "DXY", 2710, 0.405, 0.008, 6.783],
    ["FX", "EURUSD", 2739, 0.446, 0.009, 7.52],
    ["FX", "GBPUSD", 2739, 0.361, 0.005, 8.843],
    ["FX", "NZDUSD", 2739, 0.397, 0.004, 10.098],
    ["FX", "USDCAD", 2739, 0.396, 0.007, 6.996],
    ["FX", "USDCHF", 2739, 0.512, 0.003, 7.657],
    ["FX", "USDJPY", 2739, 0.398, 0.007, 8.52],

    ["EM FX", "USDAED", 3026, 0.361, 0.001, 0.081],
    ["EM FX", "USDBRL", 2738, 0.393, 0.009, 15.632],
    ["EM FX", "USDCNH", 2738, 0.382, 0.008, 4.601],
    ["EM FX", "USDIDR", 2673, 0.408, 0.005, 5.484],
    ["EM FX", "USDINR", 2727, 0.378, 0.006, 4.659],
    ["EM FX", "USDKRW", 2728, 0.384, 0.003, 8.33],
    ["EM FX", "USDMXN", 2739, 0.379, 0.005, 12.239],
    ["EM FX", "USDMYR", 2684, 0.478, 0.019, 5.295],
    ["EM FX", "USDQAR", 3349, 0.356, 0.000, 3.739],
    ["EM FX", "USDSAR", 3611, 0.372, 0.001, 0.253],
    ["EM FX", "USDSGD", 2728, 0.419, 0.006, 4.683],
    ["EM FX", "USDTHB", 2728, 0.438, 0.009, 5.778],
    ["EM FX", "USDVND", 2728, 0.462, 0.009, 1.976],
    ["EM FX", "USDZAR", 2739, 0.362, 0.004, 15.147],

    ["Rates", "Bund Futures", 2683, 0.36, 0.006, 6.626],
    ["Rates", "JGB Futures", 2790, 0.398, 0.003, 2.253],
    ["Rates", "Malaysia 5Y", 3085, 0.338, 0.012, 11.742],
    ["Rates", "Singapore 2Y", 3093, 0.426, 0.015, 43.956],
    ["Rates", "Singapore 5Y", 3111, 0.416, 0.013, 35.015],
    ["Rates", "UK Gilt", 2671, 0.35, 0.004, 7.772],
    ["Rates", "U.S. 10Y", 2733, 0.383, 0.006, 39.075],
    ["Rates", "U.S. 1Y", 2724, 0.401, 0.033, 49.554],
    ["Rates", "U.S. 2Y", 2727, 0.378, 0.021, 52.324],
    ["Rates", "U.S. 5Y", 2726, 0.382, 0.010, 48.582],

    ["REIT", "Dow Jones Equity", 2640, 0.369, 0.002, 18.353],
    ["REIT", "FTSE RAFI US 100", 2638, 0.37, 0.003, 18.885],
    ["REIT", "Hui Xian REIT", 2581, 0.354, 0.008, 19.727],
    ["REIT", "Wheeler REIT", 2419, 0.355, 0.005, 59.852],
    ["REIT", "Yuexiu REIT", 2581, 0.369, 0.004, 22.931],

    ["ETF", "EWM Malaysia", 2669, 0.452, 0.006, 17.068],
    ["ETF", "HUN Global Nat. Gas", 2111, 0.367, 0.009, 34.839],
    ["ETF", "REET Global REIT", 2666, 0.362, 0.004, 16.685],
    ["ETF", "VEU World ex. US", 2666, 0.384, 0.005, 15.839],
    ["ETF", "VWO EM", 2667, 0.399, 0.005, 18.661],

    ["MSCI", "Argentina", 2650, 0.38, 0.006, 38.388],
    ["MSCI", "Brazil", 2729, 0.663, 0.009, 37.839],
    ["MSCI", "China", 2704, 0.402, 0.007, 22.821],
    ["MSCI", "Colombia", 2729, 0.387, 0.006, 27.589],
    ["MSCI", "Denmark", 2628, 0.422, 0.011, 19.547],
    ["MSCI", "Egypt", 2097, 0.402, 0.012, 27.529],
    ["MSCI", "Eastern Europe", 2731, 0.363, 0.016, 25.629],
    ["MSCI", "France", 2688, 0.4, 0.004, 16.708],
    ["MSCI", "Germany", 2665, 0.42, 0.005, 17.361],
    ["MSCI", "India", 2591, 0.394, 0.006, 14.963],
    ["MSCI", "Indonesia", 2533, 0.413, 0.003, 18.868],
    ["MSCI", "Japan Net JPY", 2562, 0.432, 0.004, 18.184],
    ["MSCI", "Malaysia", 2567, 0.414, 0.002, 10.437],
    ["MSCI", "Philippines", 2556, 0.389, 0.003, 18.997],
    ["MSCI", "Singapore", 2702, 0.387, 0.009, 13.656],
    ["MSCI", "Sweden", 2653, 0.396, 0.004, 17.443],
    ["MSCI", "Thailand", 2551, 0.39, 0.006, 14.622],
    ["MSCI", "United Kingdom", 2656, 0.404, 0.003, 14.249],
    ["MSCI", "USA", 2642, 0.394, 0.003, 16.032],

    ["Equities", "BKGKZ", 2651, 0.375, 0.008, 28.639],
    ["Equities", "DJI", 2639, 0.391, 0.002, 15.195],
    ["Equities", "Euro Stoxx 50", 2691, 0.409, 0.004, 17.469],
    ["Equities", "NASDAQ", 2638, 0.381, 0.004, 19.885],
    ["Equities", "SPX", 2638, 0.385, 0.003, 15.881],
    ["Equities", "SGX", 2629, 0.419, 0.005, 17.442],

    ["Crypto", "Bitcoin", 3834, 0.382, 0.011, 53.223],
    ["Crypto", "BNB", 2791, 0.442, 0.021, 70.905],
    ["Crypto", "Ethereum", 2716, 0.399, 0.011, 68.2],
    ["Crypto", "Solana", 1807, 0.492, 0.022, 97.622],
    ["Crypto", "Tether", 2998, 0.675, 0.012, 7.274],
    ["Crypto", "XRP", 3812, 0.474, 0.011, 109.622],
]

benchmark = pd.DataFrame(
    benchmark_rows,
    columns=[
        "Class",
        "Dataset",
        "Samples",
        "Seasonality",
        "Trend",
        "Volatility",
    ]
)

# Normalise names for matching
import re
import pandas as pd
import unicodedata


def normalize_name(s):
    s = str(s).lower().strip()

    # remove accents / odd unicode
    s = unicodedata.normalize("NFKD", s)
    s = "".join(
        c for c in s
        if not unicodedata.combining(c)
    )

    # common irrelevant words in filenames
    remove_phrases = [
        "historical data",
        "historical",
        "index",
        "price",
    ]

    for phrase in remove_phrases:
        s = s.replace(phrase, "")

    # normalize a few common forms
    s = s.replace("united states", "us")
    s = s.replace("u.s.", "us")
    s = s.replace("s&p", "sp")

    # remove punctuation/spaces
    s = re.sub(r"[^a-z0-9]+", "", s)

    return s

ALIASES = {
    "BKGKZ": [
        "BNY Kazakhstan",
        "BNY Mellon Kazakhstan",
    ],

    "SPX": [
        "S&P 500",
        "S&P500",
        "SP500",
    ],

    "U.S. 1Y": [
        "United States 1Y Bond Yield",
    ],

    "U.S. 2Y": [
        "United States 2Y Bond Yield",
    ],

    "U.S. 5Y": [
        "United States 5Y Bond Yield",
    ],

    "U.S. 10Y": [
        "United States 10Y Bond Yield",
    ],

    "JGB Futures": [
        "Japan Gov. Bond Futures"
    ],

   "Hui Xian REIT" : [
       "Hui Xian Real Estate Stock History"
   ],

    "Wheeler REIT": [
        "Wheeler Real Estate Invest Pref Stock History"
    ],

    "Yuexiu REIT" : [
        "Yuexiu Real Estate Stock History"
    ],

    "EWM Malaysia" : [
        "EWM ETF Stock History"
    ],

    "HUN Global Nat. Gas" : [
        "HUN ETF Stock History"
    ],

    "REET Global REIT" : [
        "REET ETF Stock History"
    ],

    "VWO EM" : [
        "VWO ETF Stock History"
    ],

    "VEU World ex. US" : [
        "VEU ETF Stock Price History"
    ],

    "Eastern Europe" : [
        "MSCI EM Eastern Europe"
    ],

    "Malaysia" : [
        "MSCI Malaysia Historical Data_no_preprocess_lookback-250_horizon-20_slide-1"
    ],

    "Singapore" : [
        "MSCI Singapore Historical Data_no_preprocess_lookback-250_horizon-20_slide-1"
    ],

    "Euro Stoxx 50" : [
        "Euro Historical Data_no_preprocess_lookback-250_horizon-20_slide-1"
    ],
}

def match_dataset_name(
    benchmark_name,
    available_result_names
):
    """
    Returns the matching result dataset name.

    Priority:
    1. Explicit alias
    2. Exact normalized match
    3. Substring containment
    """

    # -----------------------------------------
    # 1. Explicit aliases
    # -----------------------------------------
    candidates = [benchmark_name]

    if benchmark_name in ALIASES:
        candidates.extend(
            ALIASES[benchmark_name]
        )

    candidate_keys = [
        normalize_name(x)
        for x in candidates
    ]

    # normalize all available names once
    result_keys = {
        result_name: normalize_name(result_name)
        for result_name in available_result_names
    }

    # -----------------------------------------
    # 2. Exact normalized match
    # -----------------------------------------
    for candidate_key in candidate_keys:

        exact = [
            original
            for original, key
            in result_keys.items()
            if key == candidate_key
        ]

        if len(exact) == 1:
            return exact[0]

    # -----------------------------------------
    # 3. Containment match
    # -----------------------------------------
    matches = []

    for candidate_key in candidate_keys:

        for original, result_key in result_keys.items():

            if (
                candidate_key in result_key
                or result_key in candidate_key
            ):
                matches.append(original)

    matches = list(
        dict.fromkeys(matches)
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        print(
            f"AMBIGUOUS: {benchmark_name} "
            f"-> {matches}"
        )
        return None

    print(
        f"NO MATCH: {benchmark_name}"
    )

    return None

# Merge
available_result_names = (
    best["Dataset"]
    .dropna()
    .unique()
    .tolist()
)

benchmark["matched_result_dataset"] = (
    benchmark["Dataset"].apply(
        lambda x: match_dataset_name(
            x,
            available_result_names
        )
    )
)

display(benchmark)

unmatched = benchmark[
    benchmark[
        "matched_result_dataset"
    ].isna()
]

display(unmatched)

chronos_summary = pd.read_csv(
    r"BForTFin_results/standardised/output_weekly/chronos_standardized_train_test/chronos_train_test_summary_all_datasets.csv"
)

sample_counts = (
    chronos_summary[
        [
            "asset_class",
            "dataset",
            "n_total_windows",
            "n_test_windows",
        ]
    ]
    .rename(
        columns={
            "asset_class": "Class",
            "dataset": "Dataset",
            "n_total_windows": "Samples",
            "n_test_windows": "Test Samples",
        }
    )
)

benchmark = benchmark.drop(
    columns=["Samples"]
).merge(
    sample_counts,
    left_on="matched_result_dataset",
    right_on="Dataset",
    how="left",
    suffixes=("", "_results")
)

benchmark = benchmark[
    [
        "Class",
        "Dataset",
        "Test Samples",
        "matched_result_dataset"
    ]
]

display(benchmark)

merged = benchmark.merge(
    best,
    left_on="matched_result_dataset",
    right_on="Dataset",
    how="left",
    suffixes=("", "_results")
)
display(merged)

merged = merged.drop(
    columns=[
        "matched_result_dataset",
        "Dataset_results",
        "Class_results",
        "_dataset_key",
        "_class_key",
        "_dataset_key_results",
        "_class_key_results",
    ],
    errors="ignore"
)

merged = merged[
    [
        "Class",
        "Dataset",

        # new column
        # "Test Samples",

        # CRPS
        "Empirical CRPS",
        "Best CRPS",
        "Best Model (CRPS)",
        "Models within 1% (CRPS)",

        "Best CRPSS",
        "Best Model (CRPSS)",
        "Models within 1% (CRPSS)",
        "Best QSS",
        "Best Model (QSS)",
        "Models within 1% (QSS)",
        "Empirical MSIS",
        "Best MSIS",
        "Best Model (MSIS)",
        "Models within 1% (MSIS)",

        "Best MSISS",
        "Best Model (MSISS)",
        "Models within 1% (MSISS)",
        "Zero Return MASE",
        "Best MASE",
        "Best Model (MASE)",
        "Models within 1% (MASE)",
        "Best MASESS",
        "Best Model (MASESS)",
        "Models within 1% (MASESS)",
        "Zero Return RMSE",
        "Best RMSE",
        "Best Model (RMSE)",
        "Models within 1% (RMSE)",
        "Best RMSESS",
        "Best Model (RMSESS)",
        "Models within 1% (RMSESS)",
    ]
]

display(merged)

# Format values
merged["Best QSS"] = merged[
    "Best QSS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best CRPS"] = merged[
    "Best CRPS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best CRPSS"] = merged[
    "Best CRPSS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MASESS"] = merged[
    "Best MASESS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MASE"] = merged[
    "Best MASE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MSIS"] = merged[
    "Best MSIS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best MSISS"] = merged[
    "Best MSISS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best RMSESS"] = merged[
    "Best RMSESS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Best RMSE"] = merged[
    "Best RMSE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Empirical CRPS"] = merged[
    "Empirical CRPS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Zero Return MASE"] = merged[
    "Zero Return MASE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Empirical MSIS"] = merged[
    "Empirical MSIS"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

merged["Zero Return RMSE"] = merged[
    "Zero Return RMSE"
].map(
    lambda x: "--"
    if pd.isna(x)
    else f"{x:.6f}"
)

# merged["Test Samples"] = merged[
#     "Test Samples"
# ].map(
#         lambda x: "--"
#         if pd.isna(x)
#         else f"{x:.0f}"
#     )

for col in [
    "Best Model (QSS)",
    "Best Model (MASE)",
    "Best Model (CRPSS)",
    "Best Model (MASESS)",
    "Best Model (CRPS)",
    "Best Model (RMSE)",
    "Best Model (MSISS)",
    "Best Model (RMSESS)",
    "Best Model (MSIS)",
    
]:  
    merged[col] = (
        merged[col]
        .fillna("--")
        .replace(
            {
                "GARCH(1,1)": "GARCH",
                "Chronos-T5": "Chronos",
            }
        )
    )

for col in [
    "Models within 1% (QSS)",
    "Models within 1% (MASE)",
    "Models within 1% (CRPSS)",
    "Models within 1% (MASESS)",
    "Models within 1% (CRPS)",
    "Models within 1% (RMSE)",
    "Models within 1% (MSISS)",
    "Models within 1% (RMSESS)",
    "Models within 1% (MSIS)",
]:
    merged[col] = merged[col].map(
        lambda x: "--"
        if pd.isna(x)
        else str(int(x))
    )

    display(merged)





body = merged.loc[:,"Class":"Models within 1% (MSISS)"].to_latex(
    index=False,
    header=False,
    escape=True,
)

print(body)






body = merged.loc[:,[
    "Class", 
    "Dataset",
    "Zero Return MASE",
    "Best MASE",
    "Best Model (MASE)",
    "Models within 1% (MASE)",
    "Best MASESS",
    "Best Model (MASESS)",
    "Models within 1% (MASESS)",
    "Zero Return RMSE",
    "Best RMSE",
    "Best Model (RMSE)",
    "Models within 1% (RMSE)",
    "Best RMSESS",
    "Best Model (RMSESS)",
    "Models within 1% (RMSESS)"
                    ]].to_latex(
    index=False,
    header=False,
    escape=True,
)

print(body)

# Calibration analysis updated

Generate:

- quantile reliability for 0.1–0.9
- 80% interval coverage using q0.1 and q0.9 interval width
- downside exceedance at h=1,5,20
- daily and weekly summaries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

QUANTILE_LEVELS_EXPECTED = np.arange(
    0.1,
    1.0,
    0.1,
)

SELECTED_HORIZONS = [
    1,
    5,
    20,
]

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

for freq in ["daily", "weekly"]:
    # ------------------------------------------------------------
    # DIRS
    # ------------------------------------------------------------
    
    BASE_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_results"
        rf"\standardised\output_{freq}"
    )
    
    DATA_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_log_return"
        rf"\arima_split_datasets\output_{freq}"
    )
    
    MODEL_DIRS = {
        "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
        "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
        "Sundial": BASE_DIR / "sundial_standardized_train_test",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
        "Toto": BASE_DIR / "toto_standardized_train_test",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
        "Moirai": BASE_DIR / "moirai_standardized_train_test",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
        "Timestral": BASE_DIR / "timestral_standardized_train_test",
    }
    
    
    
    # Quantile-file suffix used by each model.
    MODEL_SUFFIXES = {
        "GARCH(1,1)": "_garch_1_1",
        "Student-t GARCH": "_garch-t-1-1-simulated",
        "LightGBM": "_lightgbm",
        "Sundial": "_sundial",
        "Lag-Llama": "_lag-llama",
        "Chronos-T5": "_chronos-t5-small",
        "Toto": "_toto",
        "TimesFM": "_timesfm-2.5-200m",
        "Timestral": "_timestral-209m_standardized_zero-shot",
        "Moirai": "_moirai-2.0-r-small",
        "Moirai-MoE": "_moirai-moe-1.0-r-base",
        "ARIMA": "_arima",
        "LSTM": "_lstm",
    }
    
    
    
    # ============================================================
    # LOAD TEST TARGETS
    # ============================================================
    
    def load_test_y(test_path):
    
        df = pd.read_csv(test_path)
    
        y_cols = [
            f"Y_{i}"
            for i in range(HORIZON)
        ]
    
        Y = df[y_cols].to_numpy(
            dtype=float
        )
    
        return Y
    
    
    # ============================================================
    # GARCH FILENAME HELPERS
    # ============================================================
    
    def clean_garch_dataset_name(dataset_name):
    
        return (
            dataset_name
            .lower()
            .replace(" historical data", "")
            .replace(
                "_no_preprocess_lookback-250_horizon-20_slide-1",
                "",
            )
            .replace(" ", "_")
            .replace(".", "")
        )
    
    
    def find_garch_quantile_file(
        model_root,
        asset_class,
        dataset_name,
    ):
    
        asset_dir = model_root / asset_class
    
        clean_name = clean_garch_dataset_name(
            dataset_name
        )
    
        prefix = (
            f"{asset_class.lower()}_"
            f"{clean_name}_"
            f"garch_1_1_train_test"
        )
    
        expected = (
            asset_dir
            / f"{prefix}_quantiles.npz"
        )
    
        if expected.exists():
            return expected
    
        matches = list(
            asset_dir.glob(
                f"*{clean_name}*"
                f"garch_1_1_train_test*"
                f"quantiles.npz"
            )
        )
    
        if len(matches) == 1:
            return matches[0]
    
        if len(matches) == 0:
            raise FileNotFoundError(
                f"No GARCH quantile file found for "
                f"{dataset_name} in {asset_class}"
            )
    
        raise RuntimeError(
            f"Multiple GARCH quantile files found for "
            f"{dataset_name}: {matches}"
        )
    
    
    # ============================================================
    # FIND MODEL QUANTILE FILE
    # ============================================================
    
    def find_quantile_file(
        model_name,
        asset_class,
        dataset_name,
    ):
    
        model_root = MODEL_DIRS[
            model_name
        ]
    
        if model_name == "GARCH(1,1)":
    
            return find_garch_quantile_file(
                model_root=model_root,
                asset_class=asset_class,
                dataset_name=dataset_name,
            )
    
        suffix = MODEL_SUFFIXES[
            model_name
        ]
    
        model_asset_dir = (
            model_root
            / asset_class
        )
    
        matches = list(
            model_asset_dir.glob(
                f"test_{dataset_name}"
                f"{suffix}_quantiles.npz"
            )
        )
    
        if len(matches) == 1:
            return matches[0]
    
        raise FileNotFoundError(
            f"Could not uniquely identify quantiles "
            f"for {model_name}, {asset_class}, "
            f"{dataset_name}. "
            f"Found {len(matches)} matches."
        )
    
    
    # ============================================================
    # LOAD QUANTILES
    # ============================================================
    
    def load_quantiles(path):
    
        with np.load(path) as q_file:
    
            quantiles = np.asarray(
                q_file["quantiles"],
                dtype=float,
            )
    
            levels = np.asarray(
                q_file["quantile_levels"],
                dtype=float,
            )
    
        return quantiles, levels
    
    
    # ============================================================
    # QUANTILE INDEX HELPER
    # ============================================================
    
    def get_quantile_index(
        levels,
        tau,
    ):
    
        idx = np.argmin(
            np.abs(levels - tau)
        )
    
        if not np.isclose(
            levels[idx],
            tau,
        ):
            raise ValueError(
                f"No quantile tau={tau} found. "
                f"Available levels: {levels}"
            )
    
        return idx
    
    
    # ============================================================
    # CHECK QUANTILE SHAPE / ORDER
    # ============================================================
    
    def validate_quantiles(
        Y,
        Q,
        levels,
    ):
    
        if Q.ndim != 3:
            raise ValueError(
                f"Expected quantiles with 3 dimensions "
                f"(origins, horizons, quantiles), "
                f"got shape {Q.shape}"
            )
    
        if Q.shape[0] != Y.shape[0]:
            raise ValueError(
                f"Origin mismatch: "
                f"Y={Y.shape}, Q={Q.shape}"
            )
    
        if Q.shape[1] != Y.shape[1]:
            raise ValueError(
                f"Horizon mismatch: "
                f"Y={Y.shape}, Q={Q.shape}"
            )
    
        if Q.shape[2] != len(levels):
            raise ValueError(
                f"Quantile dimension mismatch: "
                f"Q={Q.shape}, levels={len(levels)}"
            )
    
        # Check that reported levels increase
        if not np.all(
            np.diff(levels) > 0
        ):
            raise ValueError(
                f"Quantile levels not strictly increasing: "
                f"{levels}"
            )
    
    
    # ============================================================
    # 1. QUANTILE RELIABILITY
    # ============================================================
    
    def calculate_reliability(
        Y,
        Q,
        levels,
    ):
        """
        For each predicted quantile tau:
    
            observed frequency =
                mean(Y <= q_tau)
    
        Perfect calibration implies:
    
            observed frequency ~= tau
        """
    
        rows = []
    
        for j, tau in enumerate(levels):
    
            q_tau = Q[:, :, j]
    
            observed_frequency = float(
                np.mean(
                    Y <= q_tau
                )
            )
    
            calibration_error = (
                observed_frequency
                - float(tau)
            )
    
            rows.append({
                "quantile_level": float(tau),
                "observed_frequency": (
                    observed_frequency
                ),
                "calibration_error": (
                    calibration_error
                ),
                "absolute_calibration_error": (
                    abs(calibration_error)
                ),
            })
    
        return rows
    
    
    # ============================================================
    # 2. 80% INTERVAL COVERAGE + WIDTH
    # ============================================================
    
    def calculate_interval_80(
        Y,
        Q,
        levels,
    ):
        """
        Nominal 80% interval:
    
            [q_0.1, q_0.9]
    
        Perfect empirical coverage would be ~0.80.
        """
    
        q10_idx = get_quantile_index(
            levels,
            0.1,
        )
    
        q90_idx = get_quantile_index(
            levels,
            0.9,
        )
    
        lower = Q[:, :, q10_idx]
        upper = Q[:, :, q90_idx]
    
        coverage = float(
            np.mean(
                (Y >= lower)
                & (Y <= upper)
            )
        )
    
        width = float(
            np.mean(
                upper - lower
            )
        )
    
        return {
            "coverage_80": coverage,
            "coverage_error": (
                coverage - 0.80
            ),
            "absolute_coverage_error": (
                abs(coverage - 0.80)
            ),
            "mean_interval_width": width,
        }
    
    
    # ============================================================
    # 3. DOWNSIDE EXCEEDANCE AT h = 1, 5, 20
    # ============================================================
    
    def calculate_downside_exceedance(
        Y,
        Q,
        levels,
    ):
        """
        Frequency with which the realized return falls
        below the predicted 10% quantile.
    
        For a calibrated q_0.1 forecast this should be
        approximately 0.10.
        """
    
        q10_idx = get_quantile_index(
            levels,
            0.1,
        )
    
        q10 = Q[:, :, q10_idx]
    
        rows = []
    
        for h in SELECTED_HORIZONS:
    
            idx = h - 1
    
            observed_frequency = float(
                np.mean(
                    Y[:, idx]
                    <= q10[:, idx]
                )
            )
    
            rows.append({
                "horizon": h,
                "nominal_probability": 0.1,
                "observed_frequency": (
                    observed_frequency
                ),
                "calibration_error": (
                    observed_frequency
                    - 0.1
                ),
                "absolute_calibration_error": (
                    abs(
                        observed_frequency
                        - 0.1
                    )
                ),
            })
    
        return rows
    
    
    # ============================================================
    # RUN ALL DAILY DATASETS
    # ============================================================
    
    reliability_rows = []
    interval_rows = []
    downside_rows = []
    
    
    for model_name in MODEL_DIRS:
    
        print()
        print("=" * 70)
        print(f"MODEL: {model_name}")
        print("=" * 70)
    
        for asset_class, asset_name in ASSET_ORDER.items():
    
            test_asset_dir = (
                DATA_DIR
                / asset_class
            )
    
            test_files = sorted(
                test_asset_dir.glob(
                    "test_*.csv"
                )
            )
    
            for test_path in test_files:
    
                try:
    
                    dataset_name = (
                        test_path.stem
                        .removeprefix("test_")
                    )
    
                    # --------------------------------------------
                    # Actual test targets
                    # --------------------------------------------
    
                    Y = load_test_y(
                        test_path
                    )
    
                    # --------------------------------------------
                    # Saved model quantiles
                    # --------------------------------------------
    
                    quantile_path = (
                        find_quantile_file(
                            model_name=model_name,
                            asset_class=asset_class,
                            dataset_name=dataset_name,
                        )
                    )
    
                    Q, levels = (
                        load_quantiles(
                            quantile_path
                        )
                    )
    
                    validate_quantiles(
                        Y=Y,
                        Q=Q,
                        levels=levels,
                    )
    
                    # --------------------------------------------
                    # Reliability
                    # --------------------------------------------
    
                    reliability = (
                        calculate_reliability(
                            Y=Y,
                            Q=Q,
                            levels=levels,
                        )
                    )
    
                    for row in reliability:
    
                        reliability_rows.append({
                            "model": model_name,
                            "asset_class": asset_name,
                            "dataset": dataset_name,
                            **row,
                        })
    
                    # --------------------------------------------
                    # 80% interval
                    # --------------------------------------------
    
                    interval = (
                        calculate_interval_80(
                            Y=Y,
                            Q=Q,
                            levels=levels,
                        )
                    )
    
                    interval_rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        **interval,
                    })
    
                    # --------------------------------------------
                    # Downside exceedance
                    # --------------------------------------------
    
                    downside = (
                        calculate_downside_exceedance(
                            Y=Y,
                            Q=Q,
                            levels=levels,
                        )
                    )
    
                    for row in downside:
    
                        downside_rows.append({
                            "model": model_name,
                            "asset_class": asset_name,
                            "dataset": dataset_name,
                            **row,
                        })
    
                    print(
                        f"  {dataset_name}: done"
                    )
    
                except Exception as exc:
    
                    print(
                        f"  ERROR {test_path.name}: "
                        f"{exc}"
                    )
    
    
    # ============================================================
    # DATAFRAMES
    # ============================================================
    
    reliability_df = pd.DataFrame(
        reliability_rows
    )
    
    interval_df = pd.DataFrame(
        interval_rows
    )
    
    downside_df = pd.DataFrame(
        downside_rows
    )
    
    
    # ============================================================
    # MODEL-LEVEL SUMMARY
    #
    # Equal-series weighting:
    # each dataset contributes equally.
    # ============================================================
    
    reliability_summary = (
        reliability_df
        .groupby(
            [
                "model",
                "quantile_level",
            ],
            as_index=False,
        )
        .agg(
            mean_observed_frequency=(
                "observed_frequency",
                "mean",
            ),
            mean_calibration_error=(
                "calibration_error",
                "mean",
            ),
            mean_absolute_calibration_error=(
                "absolute_calibration_error",
                "mean",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
        )
    )
    
    
    interval_summary = (
        interval_df
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            mean_coverage_80=(
                "coverage_80",
                "mean",
            ),
            mean_absolute_coverage_error=(
                "absolute_coverage_error",
                "mean",
            ),
            median_coverage_80=(
                "coverage_80",
                "median",
            ),
            mean_interval_width=(
                "mean_interval_width",
                "mean",
            ),
            median_interval_width=(
                "mean_interval_width",
                "median",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
        )
    )
    
    
    downside_summary = (
        downside_df
        .groupby(
            [
                "model",
                "horizon",
            ],
            as_index=False,
        )
        .agg(
            mean_observed_frequency=(
                "observed_frequency",
                "mean",
            ),
            mean_calibration_error=(
                "calibration_error",
                "mean",
            ),
            mean_absolute_calibration_error=(
                "absolute_calibration_error",
                "mean",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
        )
    )
    
    
    # ============================================================
    # ASSET-CLASS SUMMARY
    # ============================================================
    
    reliability_asset_summary = (
        reliability_df
        .groupby(
            [
                "model",
                "asset_class",
                "quantile_level",
            ],
            as_index=False,
        )
        .agg(
            mean_observed_frequency=(
                "observed_frequency",
                "mean",
            ),
            mean_absolute_calibration_error=(
                "absolute_calibration_error",
                "mean",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
        )
    )
    
    
    interval_asset_summary = (
        interval_df
        .groupby(
            [
                "model",
                "asset_class",
            ],
            as_index=False,
        )
        .agg(
            mean_coverage_80=(
                "coverage_80",
                "mean",
            ),
            mean_absolute_coverage_error=(
                "absolute_coverage_error",
                "mean",
            ),
            mean_interval_width=(
                "mean_interval_width",
                "mean",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
        )
    )
    
    
    downside_asset_summary = (
        downside_df
        .groupby(
            [
                "model",
                "asset_class",
                "horizon",
            ],
            as_index=False,
        )
        .agg(
            mean_observed_frequency=(
                "observed_frequency",
                "mean",
            ),
            mean_absolute_calibration_error=(
                "absolute_calibration_error",
                "mean",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
        )
    )
    
    
    # ============================================================
    # SAVE
    # ============================================================
    
    reliability_df.to_csv(
        BASE_DIR
        / f"calibration_reliability_{freq}_with_timestral.csv",
        index=False,
    )
    
    interval_df.to_csv(
        BASE_DIR
        / f"calibration_interval80_{freq}_with_timestral.csv",
        index=False,
    )
    
    downside_df.to_csv(
        BASE_DIR
        / f"calibration_downside_horizons_{freq}_with_timestral.csv",
        index=False,
    )
    
    
    reliability_summary.to_csv(
        BASE_DIR
        / f"calibration_reliability_summary_{freq}_with_timestral.csv",
        index=False,
    )
    
    interval_summary.to_csv(
        BASE_DIR
        / f"calibration_interval80_summary_{freq}_with_timestral.csv",
        index=False,
    )
    
    downside_summary.to_csv(
        BASE_DIR
        / f"calibration_downside_summary_{freq}_with_timestral.csv",
        index=False,
    )
    
    
    reliability_asset_summary.to_csv(
        BASE_DIR
        / f"calibration_reliability_asset_class_{freq}_with_timestral.csv",
        index=False,
    )
    
    interval_asset_summary.to_csv(
        BASE_DIR
        / f"calibration_interval80_asset_class_{freq}_with_timestral.csv",
        index=False,
    )
    
    downside_asset_summary.to_csv(
        BASE_DIR
        / f"calibration_downside_asset_class_{freq}_with_timestral.csv",
        index=False,
    )
    
    
    print()
    print("=" * 70)
    print(f"SAVED {freq.upper()} CALIBRATION DIAGNOSTICS")
    print("=" * 70)

### Summary

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


# ============================================================
# BUILD ONE FREQUENCY SUMMARY
# ============================================================

def build_calibration_summary(
    base_dir,
    frequency,
):

    # --------------------------------------------------------
    # Load existing summary files
    # --------------------------------------------------------

    reliability = pd.read_csv(
        base_dir
        / f"calibration_reliability_summary_{frequency.lower()}_with_timestral.csv"
    )

    interval = pd.read_csv(
        base_dir
        / f"calibration_interval80_summary_{frequency.lower()}_with_timestral.csv"
    )

    downside = pd.read_csv(
        base_dir
        / f"calibration_downside_summary_{frequency.lower()}_with_timestral.csv"
    )


    # ========================================================
    # 1. OVERALL RELIABILITY ERROR
    #
    # reliability currently has one row per:
    # model × quantile level
    #
    # Average absolute calibration error across q=0.1,...,0.9
    # ========================================================

    reliability_overall = (
        reliability
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            mean_abs_reliability_error=(
                "mean_absolute_calibration_error",
                "mean",
            ),
            mean_signed_reliability_error=(
                "mean_calibration_error",
                "mean",
            ),
        )
    )


    # ========================================================
    # 2. INTERVAL RESULTS
    # ========================================================

    interval_keep = interval[
        [
            "model",
            "mean_coverage_80",
            "mean_absolute_coverage_error",
            "mean_interval_width",
        ]
    ].copy()


    # ========================================================
    # 3. DOWNSIDE q0.1 EXCEEDANCE
    #
    # Convert:
    #
    # model | horizon | observed freq
    #
    # into:
    #
    # model | q10_h1 | q10_h5 | q10_h20
    # ========================================================

    downside_keep = downside[
        downside["horizon"].isin(
            [1, 5, 20]
        )
    ].copy()

    downside_pivot = (
        downside_keep
        .pivot(
            index="model",
            columns="horizon",
            values="mean_observed_frequency",
        )
        .reset_index()
    )

    downside_pivot = downside_pivot.rename(
        columns={
            1: "q10_exceedance_h1",
            5: "q10_exceedance_h5",
            20: "q10_exceedance_h20",
        }
    )


    # ========================================================
    # MERGE EVERYTHING
    # ========================================================

    summary = (
        reliability_overall
        .merge(
            interval_keep,
            on="model",
            how="outer",
        )
        .merge(
            downside_pivot,
            on="model",
            how="outer",
        )
    )

    summary.insert(
        0,
        "frequency",
        frequency,
    )


    # ========================================================
    # ADD SIMPLE INTERPRETATION METRICS
    # ========================================================

    # Mean absolute q0.1 calibration error across h=1,5,20

    summary["mean_abs_q10_error_h1_5_20"] = (
        (
            (summary["q10_exceedance_h1"] - 0.10).abs()
            + (summary["q10_exceedance_h5"] - 0.10).abs()
            + (summary["q10_exceedance_h20"] - 0.10).abs()
        )
        / 3
    )


    # ========================================================
    # SORT
    #
    # Lowest reliability error first
    # ========================================================

    summary = summary.sort_values(
        "mean_abs_reliability_error",
        ascending=True,
    ).reset_index(drop=True)

    return summary


# ============================================================
# DAILY + WEEKLY
# ============================================================

all_summaries = []

for frequency, base_dir in FREQUENCIES.items():

    summary = build_calibration_summary(
        base_dir=base_dir,
        frequency=frequency,
    )

    # Save frequency-specific table

    output_path = (
        base_dir
        / f"calibration_compact_summary_{frequency.lower()}.csv"
    )

    summary.to_csv(
        output_path,
        index=False,
    )

    print()
    print("=" * 80)
    print(f"{frequency.upper()} CALIBRATION")
    print("=" * 80)

    print(
        summary.round(4).to_string(
            index=False
        )
    )

    print()
    print(
        f"Saved to: {output_path}"
    )

    all_summaries.append(
        summary
    )


# ============================================================
# COMBINED DAILY + WEEKLY TABLE
# ============================================================

combined = pd.concat(
    all_summaries,
    ignore_index=True,
)

combined_path = (
    ROOT
    / "calibration_compact_summary_daily_weekly_with_timestral.csv"
)

combined.to_csv(
    combined_path,
    index=False,
)

print()
print("=" * 80)
print("COMBINED TABLE SAVED")
print("=" * 80)
print(combined_path)

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


for frequency, base_dir in FREQUENCIES.items():

    df = pd.read_csv(
        base_dir
        / f"calibration_reliability_summary_{frequency.lower()}_with_timestral.csv"
    )

    df["model"] = df["model"].replace({
        "Chronos-T5": "Chronos",
        "GARCH(1,1)": "Gaussian GARCH",
    })

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    num_lines = 11
    colors = plt.cm.tab20(np.linspace(0, 1, num_lines))
    ax.set_prop_cycle(color=colors)

    # --------------------------------------------------------
    # Perfect calibration line
    # --------------------------------------------------------

    # ax.plot(
    #     [0.1, 0.9],
    #     [0.1, 0.9],
    #     linestyle="--",
    #     linewidth=1.5,
    #     label="Perfect calibration",
    # )


    # --------------------------------------------------------
    # One reliability curve per model
    # --------------------------------------------------------

    for model, group in df.groupby("model"):

        group = group.sort_values(
            "quantile_level"
        )

        ax.plot(
            group["quantile_level"],
            group["mean_observed_frequency"],
            marker="o",
            linewidth=1.9,
            label=model,
            alpha=0.6
        )

    ax.plot(
        [0.1, 0.9],
        [0.1, 0.9],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration",
        alpha=1,
        color="black",
    )
    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    ax.set_xlabel(
        "Nominal Quantile Level",
        fontsize=14,
    )

    ax.set_ylabel(
        "Observed Frequency Below Predicted Quantile",
        fontsize=14,
    )

    ax.set_title(
        f"{frequency} Quantile Reliability",
        fontsize=18,
    )

    ax.set_xlim(
        0.08,
        0.92,
    )

    ax.set_ylim(
        0.0,
        1.0,
    )

    ax.grid(
        alpha=0.4
    )

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
    )

    fig.tight_layout()

    output_path = (
        base_dir
        / f"calibration_reliability_plot_{frequency.lower()}_with_timestral.png"
    )

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        f"Saved to: {output_path}"
    )

### Asset class summary

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


# ============================================================
# BUILD ASSET-CLASS CALIBRATION TABLE
# ============================================================

def build_asset_class_calibration_table(
    base_dir,
    frequency,
):

    freq_lower = frequency.lower()

    # --------------------------------------------------------
    # Load asset-class summaries
    # --------------------------------------------------------

    reliability = pd.read_csv(
        base_dir
        / f"calibration_reliability_asset_class_{freq_lower}_with_timestral.csv"
    )

    interval = pd.read_csv(
        base_dir
        / f"calibration_interval80_asset_class_{freq_lower}_with_timestral.csv"
    )

    downside = pd.read_csv(
        base_dir
        / f"calibration_downside_asset_class_{freq_lower}_with_timestral.csv"
    )


    # ========================================================
    # 1. OVERALL RELIABILITY MAE BY MODEL × ASSET CLASS
    #
    # reliability currently has one row per:
    # model × asset_class × quantile_level
    #
    # Average over q = 0.1,...,0.9
    # ========================================================

    reliability_overall = (
        reliability
        .groupby(
            [
                "model",
                "asset_class",
            ],
            as_index=False,
        )
        .agg(
            reliability_mae=(
                "mean_absolute_calibration_error",
                "mean",
            ),
        )
    )


    # ========================================================
    # 2. 80% INTERVAL RESULTS
    # ========================================================

    interval_keep = interval[
        [
            "model",
            "asset_class",
            "mean_coverage_80",
            "mean_absolute_coverage_error",
            "mean_interval_width",
        ]
    ].copy()


    # ========================================================
    # 3. q0.1 EXCEEDANCE AT h = 1, 5, 20
    # ========================================================

    downside_keep = downside[
        downside["horizon"].isin(
            [1, 5, 20]
        )
    ].copy()

    downside_pivot = (
        downside_keep
        .pivot_table(
            index=[
                "model",
                "asset_class",
            ],
            columns="horizon",
            values="mean_observed_frequency",
            aggfunc="first",
        )
        .reset_index()
    )

    downside_pivot = downside_pivot.rename(
        columns={
            1: "q10_exceedance_h1",
            5: "q10_exceedance_h5",
            20: "q10_exceedance_h20",
        }
    )


    # ========================================================
    # 4. MERGE
    # ========================================================

    table = (
        reliability_overall
        .merge(
            interval_keep,
            on=[
                "model",
                "asset_class",
            ],
            how="outer",
        )
        .merge(
            downside_pivot,
            on=[
                "model",
                "asset_class",
            ],
            how="outer",
        )
    )

    table.insert(
        0,
        "frequency",
        frequency,
    )


    # ========================================================
    # 5. OPTIONAL:
    # MEAN ABSOLUTE q0.1 ERROR ACROSS h=1,5,20
    # ========================================================

    table["q10_mae_h1_5_20"] = (
        (
            (table["q10_exceedance_h1"] - 0.10).abs()
            + (table["q10_exceedance_h5"] - 0.10).abs()
            + (table["q10_exceedance_h20"] - 0.10).abs()
        )
        / 3
    )


    # ========================================================
    # 6. SORT
    # ========================================================

    asset_order = [
        "FX",
        "EM FX",
        "MSCI",
        "Rates",
        "REIT",
        "Equities",
        "ETF",
        "Crypto",
    ]

    table["asset_class"] = pd.Categorical(
        table["asset_class"],
        categories=asset_order,
        ordered=True,
    )

    table = table.sort_values(
        [
            "asset_class",
            "reliability_mae",
            "model",
        ]
    ).reset_index(drop=True)

    display_table = table[
        [
            "frequency",
            "model",
            "asset_class",
            "reliability_mae",
            "mean_coverage_80",
            "q10_exceedance_h1",
            "q10_exceedance_h5",
            "q10_exceedance_h20",
        ]
    ].copy()
    
    display_table = display_table.rename(
        columns={
            "frequency": "Frequency",
            "model": "Model",
            "asset_class": "Asset Class",
            "reliability_mae": "Reliability MAE",
            "mean_coverage_80": "80% Coverage",
            "q10_exceedance_h1": "q0.1 Exceedance h=1",
            "q10_exceedance_h5": "q0.1 Exceedance h=5",
            "q10_exceedance_h20": "q0.1 Exceedance h=20",
        }
    )
    
    print(
        display_table
        .round(3)
        .to_string(index=False)
    )


    return table


# ============================================================
# RUN DAILY + WEEKLY
# ============================================================

all_tables = []

for frequency, base_dir in FREQUENCIES.items():

    table = build_asset_class_calibration_table(
        base_dir=base_dir,
        frequency=frequency,
    )

    output_path = (
        base_dir
        / f"calibration_asset_class_table_{frequency.lower()}_with_timestral.csv"
    )

    table.to_csv(
        output_path,
        index=False,
    )

    print()
    print("=" * 100)
    print(
        f"{frequency.upper()} ASSET-CLASS CALIBRATION"
    )
    print("=" * 100)

    print(
        table.round(4).to_string(
            index=False
        )
    )

    print()
    print(f"Saved to: {output_path}")

    all_tables.append(
        table
    )


# ============================================================
# COMBINED DAILY + WEEKLY
# ============================================================

combined = pd.concat(
    all_tables,
    ignore_index=True,
)

combined_path = (
    ROOT
    / "calibration_asset_class_table_daily_weekly_with_timestral.csv"
)

combined.to_csv(
    combined_path,
    index=False,
)

print()
print("=" * 100)
print("COMBINED ASSET-CLASS TABLE SAVED")
print("=" * 100)
print(combined_path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

INPUT_PATH = (
    ROOT
    / "calibration_asset_class_table_daily_weekly_with_timestral.csv"
)

OUTPUT_PATH = (
    ROOT
    / "calibration_asset_class_analysis_with_timestral.csv"
)


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(
    INPUT_PATH
)


# ============================================================
# DERIVED ERRORS
# ============================================================

# 80% interval coverage:
# smaller error from 0.80 = better
df["coverage80_error"] = (
    df["mean_coverage_80"]
    - 0.80
).abs()


# q0.1 exceedance:
# smaller error from 0.10 = better

df["q10_error_h1"] = (
    df["q10_exceedance_h1"]
    - 0.10
).abs()

df["q10_error_h5"] = (
    df["q10_exceedance_h5"]
    - 0.10
).abs()

df["q10_error_h20"] = (
    df["q10_exceedance_h20"]
    - 0.10
).abs()


df["q10_mean_error"] = (
    df[
        [
            "q10_error_h1",
            "q10_error_h5",
            "q10_error_h20",
        ]
    ]
    .mean(axis=1)
)


# ============================================================
# RANK WITHIN EACH FREQUENCY × ASSET CLASS
# ============================================================

group_cols = [
    "frequency",
    "asset_class",
]


# Lower = better for all of these
df["rank_reliability"] = (
    df
    .groupby(group_cols)[
        "reliability_mae"
    ]
    .rank(
        method="average",
        ascending=True,
    )
)


df["rank_coverage80"] = (
    df
    .groupby(group_cols)[
        "coverage80_error"
    ]
    .rank(
        method="average",
        ascending=True,
    )
)


df["rank_q10"] = (
    df
    .groupby(group_cols)[
        "q10_mean_error"
    ]
    .rank(
        method="average",
        ascending=True,
    )
)


# ============================================================
# OVERALL CALIBRATION RANK
#
# Equal weight to:
# - reliability
# - 80% interval coverage
# - q0.1 exceedance
# ============================================================

df["mean_calibration_rank"] = (
    df[
        [
            "rank_reliability",
            "rank_coverage80",
            "rank_q10",
        ]
    ]
    .mean(axis=1)
)


# ============================================================
# BEST MODEL PER ASSET CLASS
# ============================================================

best_reliability = (
    df.loc[
        df.groupby(group_cols)[
            "reliability_mae"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "reliability_mae",
        ]
    ]
    .rename(
        columns={
            "model": "best_reliability_model",
            "reliability_mae": "best_reliability_mae",
        }
    )
)


best_coverage = (
    df.loc[
        df.groupby(group_cols)[
            "coverage80_error"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "mean_coverage_80",
            "coverage80_error",
        ]
    ]
    .rename(
        columns={
            "model": "best_coverage_model",
            "mean_coverage_80": "best_coverage80",
            "coverage80_error": "best_coverage80_error",
        }
    )
)


best_q10 = (
    df.loc[
        df.groupby(group_cols)[
            "q10_mean_error"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "q10_mean_error",
        ]
    ]
    .rename(
        columns={
            "model": "best_q10_model",
            "q10_mean_error": "best_q10_mean_error",
        }
    )
)


best_overall = (
    df.loc[
        df.groupby(group_cols)[
            "mean_calibration_rank"
        ].idxmin()
    ][
        [
            "frequency",
            "asset_class",
            "model",
            "mean_calibration_rank",
        ]
    ]
    .rename(
        columns={
            "model": "best_overall_calibration_model",
        }
    )
)


# ============================================================
# COMBINE BEST-MODEL SUMMARY
# ============================================================

best_summary = (
    best_reliability
    .merge(
        best_coverage,
        on=group_cols,
    )
    .merge(
        best_q10,
        on=group_cols,
    )
    .merge(
        best_overall,
        on=group_cols,
    )
)


# ============================================================
# MODEL CONSISTENCY ACROSS ASSET CLASSES
#
# Average rank across all asset classes
# separately for Daily / Weekly
# ============================================================

model_consistency = (
    df
    .groupby(
        [
            "frequency",
            "model",
        ],
        as_index=False,
    )
    .agg(
        mean_reliability_rank=(
            "rank_reliability",
            "mean",
        ),
        mean_coverage_rank=(
            "rank_coverage80",
            "mean",
        ),
        mean_q10_rank=(
            "rank_q10",
            "mean",
        ),
        mean_overall_calibration_rank=(
            "mean_calibration_rank",
            "mean",
        ),
        mean_reliability_mae=(
            "reliability_mae",
            "mean",
        ),
        mean_coverage80_error=(
            "coverage80_error",
            "mean",
        ),
        mean_q10_error=(
            "q10_mean_error",
            "mean",
        ),
    )
)


model_consistency = (
    model_consistency
    .sort_values(
        [
            "frequency",
            "mean_overall_calibration_rank",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# COUNT "WINS" ACROSS ASSET CLASSES
# ============================================================

overall_wins = (
    best_overall
    .groupby(
        [
            "frequency",
            "best_overall_calibration_model",
        ]
    )
    .size()
    .reset_index(
        name="n_asset_class_wins"
    )
    .rename(
        columns={
            "best_overall_calibration_model": "model"
        }
    )
)


reliability_wins = (
    best_reliability
    .groupby(
        [
            "frequency",
            "best_reliability_model",
        ]
    )
    .size()
    .reset_index(
        name="n_reliability_wins"
    )
    .rename(
        columns={
            "best_reliability_model": "model"
        }
    )
)


# ============================================================
# SAVE DETAILED RANKED TABLE
# ============================================================

ranked_output = df[
    [
        "frequency",
        "asset_class",
        "model",
        "reliability_mae",
        "mean_coverage_80",
        "coverage80_error",
        "q10_exceedance_h1",
        "q10_exceedance_h5",
        "q10_exceedance_h20",
        "q10_mean_error",
        "rank_reliability",
        "rank_coverage80",
        "rank_q10",
        "mean_calibration_rank",
    ]
].copy()


ranked_output = ranked_output.sort_values(
    [
        "frequency",
        "asset_class",
        "mean_calibration_rank",
    ]
)


ranked_output.to_csv(
    OUTPUT_PATH,
    index=False,
)


# ============================================================
# SAVE OTHER SUMMARIES
# ============================================================

best_summary.to_csv(
    ROOT
    / "calibration_best_model_by_asset_class_with_timestral.csv",
    index=False,
)

model_consistency.to_csv(
    ROOT
    / "calibration_model_consistency_by_frequency_with_timestral.csv",
    index=False,
)

overall_wins.to_csv(
    ROOT
    / "calibration_asset_class_wins_with_timestral.csv",
    index=False,
)

reliability_wins.to_csv(
    ROOT
    / "calibration_reliability_wins_with_timestral.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 100)
print("BEST MODELS BY ASSET CLASS")
print("=" * 100)

print(
    best_summary
    .round(4)
    .to_string(index=False)
)


print()
print("=" * 100)
print("MODEL CONSISTENCY ACROSS ASSET CLASSES")
print("=" * 100)

print(
    model_consistency
    .round(4)
    .to_string(index=False)
)


print()
print("=" * 100)
print("OVERALL CALIBRATION WINS")
print("=" * 100)

print(
    overall_wins.to_string(
        index=False
    )
)

# QSS analysis updated

For each model, calculate:

- mean QSS
- median QSS
- number / percentage of datasets with QSS>0
- an equal-asset-class-weighted mean QSS because the asset classes contain different numbers of series

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


# ============================================================
# BUILD QSS SUMMARY
# ============================================================

def build_qss_summary(
    base_dir,
    frequency,
):

    freq_lower = frequency.lower()

    qss = pd.read_csv(
        base_dir
        / "qss_summary_all_models_with_timestral.csv"
    )


    # ========================================================
    # DATASET-LEVEL INDICATORS
    # ========================================================

    qss["positive_qss"] = (
        qss["qss_0.1"] > 0
    )

    qss["negative_qss"] = (
        qss["qss_0.1"] < 0
    )

    qss["zero_qss"] = (
        qss["qss_0.1"] == 0
    )


    # ========================================================
    # ORDINARY EQUAL-SERIES SUMMARY
    #
    # Every dataset gets equal weight.
    # ========================================================

    series_summary = (
        qss
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            mean_qss=(
                "qss_0.1",
                "mean",
            ),
            median_qss=(
                "qss_0.1",
                "median",
            ),
            min_qss=(
                "qss_0.1",
                "min",
            ),
            max_qss=(
                "qss_0.1",
                "max",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
            n_positive_qss=(
                "positive_qss",
                "sum",
            ),
            n_negative_qss=(
                "negative_qss",
                "sum",
            ),
        )
    )


    series_summary["pct_positive_qss"] = (
        100
        * series_summary["n_positive_qss"]
        / series_summary["n_datasets"]
    )

    series_summary["pct_negative_qss"] = (
        100
        * series_summary["n_negative_qss"]
        / series_summary["n_datasets"]
    )


    # ========================================================
    # ASSET-CLASS MEANS
    #
    # First calculate one mean QSS per:
    # model × asset class.
    # ========================================================

    asset_means = (
        qss
        .groupby(
            [
                "model",
                "asset_class",
            ],
            as_index=False,
        )
        .agg(
            asset_mean_qss=(
                "qss_0.1",
                "mean",
            ),
            asset_median_qss=(
                "qss_0.1",
                "median",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
            n_positive_qss=(
                "positive_qss",
                "sum",
            ),
        )
    )


    asset_means["pct_positive_qss"] = (
        100
        * asset_means["n_positive_qss"]
        / asset_means["n_datasets"]
    )


    # ========================================================
    # EQUAL-ASSET-CLASS SUMMARY
    #
    # Each of the 8 asset classes contributes equally,
    # regardless of how many datasets it contains.
    # ========================================================

    equal_asset_summary = (
        asset_means
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            equal_asset_mean_qss=(
                "asset_mean_qss",
                "mean",
            ),
            equal_asset_median_qss=(
                "asset_median_qss",
                "mean",
            ),
            mean_asset_pct_positive=(
                "pct_positive_qss",
                "mean",
            ),
            n_asset_classes=(
                "asset_class",
                "count",
            ),
        )
    )


    # ========================================================
    # MERGE
    # ========================================================

    summary = series_summary.merge(
        equal_asset_summary,
        on="model",
        how="left",
        validate="one_to_one",
    )

    summary.insert(
        0,
        "frequency",
        frequency,
    )


    # ========================================================
    # SORT
    #
    # Higher QSS = better
    # ========================================================

    summary = (
        summary
        .sort_values(
            "mean_qss",
            ascending=False,
        )
        .reset_index(drop=True)
    )


    return (
        summary,
        asset_means,
    )


# ============================================================
# RUN DAILY + WEEKLY
# ============================================================

all_summaries = []

for frequency, base_dir in FREQUENCIES.items():

    summary, asset_means = (
        build_qss_summary(
            base_dir=base_dir,
            frequency=frequency,
        )
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    summary_path = (
        base_dir
        / f"qss_robustness_summary_{frequency.lower()}_with_timestral.csv"
    )

    asset_path = (
        base_dir
        / f"qss_asset_class_summary_{frequency.lower()}_with_timestral.csv"
    )


    summary.to_csv(
        summary_path,
        index=False,
    )

    asset_means.to_csv(
        asset_path,
        index=False,
    )


    # --------------------------------------------------------
    # Print compact version
    # --------------------------------------------------------

    print()
    print("=" * 110)
    print(
        f"{frequency.upper()} QSS ROBUSTNESS"
    )
    print("=" * 110)

    display_cols = [
        "model",
        "mean_qss",
        "median_qss",
        "n_positive_qss",
        "pct_positive_qss",
        "equal_asset_mean_qss",
        "mean_asset_pct_positive",
    ]

    print(
        summary[
            display_cols
        ]
        .round(4)
        .to_string(index=False)
    )


    print()
    print(
        f"Saved to: {summary_path}"
    )

    all_summaries.append(
        summary
    )


# ============================================================
# COMBINED DAILY + WEEKLY
# ============================================================

combined = pd.concat(
    all_summaries,
    ignore_index=True,
)

combined.to_csv(
    ROOT
    / "qss_robustness_summary_daily_weekly_with_timestral.csv",
    index=False,
)

# Bootstrap updated

### QSS moving block bootstrap

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

N_BOOT = 2000
BLOCK_LENGTHS = [10, 20, 40]
SEED = 42

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


for freq in ["daily", "weekly"]:

    BASE_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_results"
        rf"\standardised\output_{freq}"
    )
    
    DATA_DIR = Path(
        r"."
        r"\BForTFinProbability\BForTFin_log_return"
        rf"\arima_split_datasets\output_{freq}"
    )
    
    
    MODEL_DIRS = {
        "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
        "Student-t GARCH": BASE_DIR / "garch_t_1_1_standardized_train_test",
        "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
        "Sundial": BASE_DIR / "sundial_standardized_train_test",
        "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
        "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
        "Toto": BASE_DIR / "toto_standardized_train_test",
        "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
        "Moirai": BASE_DIR / "moirai_standardized_train_test",
        "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
        "Timestral": BASE_DIR / "timestral_standardized_train_test",
    }
    
    
    
    # Quantile-file suffix used by each model.
    MODEL_SUFFIXES = {
        "GARCH(1,1)": "_garch_1_1",
        "Student-t GARCH": "_garch-t-1-1-simulated",
        "LightGBM": "_lightgbm",
        "Sundial": "_sundial",
        "Lag-Llama": "_lag-llama",
        "Chronos-T5": "_chronos-t5-small",
        "Toto": "_toto",
        "TimesFM": "_timesfm-2.5-200m",
        "Timestral": "_timestral-209m_standardized_zero-shot",
        "Moirai": "_moirai-2.0-r-small",
        "Moirai-MoE": "_moirai-moe-1.0-r-base",
        "ARIMA": "_arima",
        "LSTM": "_lstm",
    }
    
    # ============================================================
    # BASIC HELPERS
    # ============================================================
    
    def pinball_loss(y, q, tau=TAU):
    
        error = y - q
    
        return np.maximum(
            tau * error,
            (tau - 1.0) * error,
        )
    
    
    def load_test_xy(test_path):
    
        df = pd.read_csv(test_path)
    
        x_cols = [
            f"X_{i}"
            for i in range(LOOKBACK)
        ]
    
        y_cols = [
            f"Y_{i}"
            for i in range(HORIZON)
        ]
    
        X = df[x_cols].to_numpy(dtype=float)
        Y = df[y_cols].to_numpy(dtype=float)
    
        return X, Y
    
    
    def get_q10_index(levels):
    
        idx = np.argmin(
            np.abs(levels - TAU)
        )
    
        if not np.isclose(
            levels[idx],
            TAU,
        ):
            raise ValueError(
                f"No q{TAU} found: {levels}"
            )
    
        return idx
    
    
    # ============================================================
    # GARCH FILE HELPERS
    # ============================================================
    
    def clean_garch_dataset_name(dataset_name):
    
        return (
            dataset_name
            .lower()
            .replace(" historical data", "")
            .replace(
                "_no_preprocess_lookback-250_horizon-20_slide-1",
                "",
            )
            .replace(" ", "_")
            .replace(".", "")
        )
    
    
    def find_garch_quantile_file(
        model_root,
        asset_class,
        dataset_name,
    ):
    
        asset_dir = (
            model_root
            / asset_class
        )
    
        clean_name = clean_garch_dataset_name(
            dataset_name
        )
    
        prefix = (
            f"{asset_class.lower()}_"
            f"{clean_name}_"
            f"garch_1_1_train_test"
        )
    
        expected = (
            asset_dir
            / f"{prefix}_quantiles.npz"
        )
    
        if expected.exists():
            return expected
    
        matches = list(
            asset_dir.glob(
                f"*{clean_name}*"
                f"garch_1_1_train_test*"
                f"quantiles.npz"
            )
        )
    
        if len(matches) == 1:
            return matches[0]
    
        raise FileNotFoundError(
            f"Could not identify GARCH quantiles for "
            f"{dataset_name}; found {len(matches)}."
        )
    
    
    def find_quantile_file(
        model_name,
        asset_class,
        dataset_name,
    ):
    
        model_root = MODEL_DIRS[
            model_name
        ]
    
        if model_name == "GARCH(1,1)":
    
            return find_garch_quantile_file(
                model_root,
                asset_class,
                dataset_name,
            )
    
        suffix = MODEL_SUFFIXES[
            model_name
        ]
    
        matches = list(
            (
                model_root
                / asset_class
            ).glob(
                f"test_{dataset_name}"
                f"{suffix}_quantiles.npz"
            )
        )
    
        if len(matches) != 1:
            raise FileNotFoundError(
                f"{model_name}, {dataset_name}: "
                f"found {len(matches)} quantile files."
            )
    
        return matches[0]
    
    
    # ============================================================
    # MOVING-BLOCK BOOTSTRAP
    # ============================================================
    
    def moving_block_bootstrap_mean_ci(
        values,
        block_length,
        n_boot=N_BOOT,
        seed=SEED,
    ):
        """
        Bootstrap confidence interval for the mean of a
        serially dependent sequence using overlapping
        moving blocks.
    
        values:
            one paired loss difference per forecast origin
    
        Positive mean:
            model loss < empirical-reference loss
        """
    
        values = np.asarray(
            values,
            dtype=float,
        )
    
        n = len(values)
    
        if block_length > n:
            raise ValueError(
                f"Block length {block_length} > n={n}"
            )
    
        rng = np.random.default_rng(
            seed
        )
    
        # All possible contiguous block starts
        max_start = (
            n - block_length
        )
    
        n_blocks_needed = int(
            np.ceil(
                n / block_length
            )
        )
    
        boot_means = np.empty(
            n_boot,
            dtype=float,
        )
    
        for b in range(n_boot):
    
            starts = rng.integers(
                0,
                max_start + 1,
                size=n_blocks_needed,
            )
    
            sample = np.concatenate([
                values[
                    s:s + block_length
                ]
                for s in starts
            ])
    
            # Trim to original length
            sample = sample[:n]
    
            boot_means[b] = np.mean(
                sample
            )
    
        estimate = float(
            np.mean(values)
        )
    
        ci_lower = float(
            np.quantile(
                boot_means,
                0.025,
            )
        )
    
        ci_upper = float(
            np.quantile(
                boot_means,
                0.975,
            )
        )
    
        return {
            "mean_loss_difference": estimate,
            "ci_lower_95": ci_lower,
            "ci_upper_95": ci_upper,
            "significantly_positive": (
                ci_lower > 0
            ),
            "significantly_negative": (
                ci_upper < 0
            ),
        }
    
    
    # ============================================================
    # RUN
    # ============================================================
    
    rows = []
    
    
    for model_name in MODEL_DIRS:
    
        print()
        print("=" * 70)
        print(f"MODEL: {model_name}")
        print("=" * 70)
    
        for asset_class, asset_name in ASSET_ORDER.items():
    
            test_dir = (
                DATA_DIR
                / asset_class
            )
    
            test_files = sorted(
                test_dir.glob(
                    "test_*.csv"
                )
            )
    
            for test_path in test_files:
    
                try:
    
                    dataset_name = (
                        test_path.stem
                        .removeprefix("test_")
                    )
    
                    X, Y = load_test_xy(
                        test_path
                    )
    
                    # --------------------------------------------
                    # Empirical q0.1 reference
                    # --------------------------------------------
    
                    q_ref = np.quantile(
                        X,
                        TAU,
                        axis=1,
                    )
    
                    q_ref = q_ref[:, None]
    
                    reference_loss = pinball_loss(
                        Y,
                        q_ref,
                        TAU,
                    )
    
                    # --------------------------------------------
                    # Model q0.1
                    # --------------------------------------------
    
                    quantile_path = (
                        find_quantile_file(
                            model_name,
                            asset_class,
                            dataset_name,
                        )
                    )
    
                    with np.load(
                        quantile_path
                    ) as q_file:
    
                        Q = np.asarray(
                            q_file["quantiles"],
                            dtype=float,
                        )
    
                        levels = np.asarray(
                            q_file[
                                "quantile_levels"
                            ],
                            dtype=float,
                        )
    
                    q10_idx = get_q10_index(
                        levels
                    )
    
                    model_q10 = Q[
                        :,
                        :,
                        q10_idx,
                    ]
    
                    if model_q10.shape != Y.shape:
                        raise ValueError(
                            f"{model_name} {dataset_name}: "
                            f"q10 {model_q10.shape}, "
                            f"Y {Y.shape}"
                        )
    
                    model_loss = pinball_loss(
                        Y,
                        model_q10,
                        TAU,
                    )
    
                    # --------------------------------------------
                    # One paired loss difference per origin
                    #
                    # Average the 20 horizons within each origin.
                    #
                    # Overall mean of this vector is exactly the
                    # aggregate mean loss difference.
                    # --------------------------------------------
    
                    loss_difference_by_origin = (
                        np.mean(
                            reference_loss
                            - model_loss,
                            axis=1,
                        )
                    )
    
                    # --------------------------------------------
                    # Bootstrap several block lengths
                    # --------------------------------------------
    
                    for block_length in BLOCK_LENGTHS:
    
                        result = (
                            moving_block_bootstrap_mean_ci(
                                loss_difference_by_origin,
                                block_length=block_length,
                            )
                        )
    
                        rows.append({
                            "model": model_name,
                            "asset_class": asset_name,
                            "dataset": dataset_name,
                            "block_length": block_length,
                            "n_origins": len(
                                loss_difference_by_origin
                            ),
                            **result,
                        })
    
                    print(
                        f"  {dataset_name}: done"
                    )
    
                except Exception as exc:
    
                    print(
                        f"  ERROR {test_path.name}: "
                        f"{exc}"
                    )
    
    
    # ============================================================
    # SAVE DATASET-LEVEL RESULTS
    # ============================================================
    
    results = pd.DataFrame(
        rows
    )
    
    results.to_csv(
        BASE_DIR
        / f"qss_loss_difference_block_bootstrap_{freq}_with_timestral.csv",
        index=False,
    )
    
    
    # ============================================================
    # SUMMARY FOR PRIMARY BLOCK LENGTH = 20
    # ============================================================
    
    primary = results[
        results["block_length"] == 20
    ].copy()
    
    
    model_summary = (
        primary
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            mean_loss_difference=(
                "mean_loss_difference",
                "mean",
            ),
            median_loss_difference=(
                "mean_loss_difference",
                "median",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
            n_ci_positive=(
                "significantly_positive",
                "sum",
            ),
            n_ci_negative=(
                "significantly_negative",
                "sum",
            ),
        )
    )
    
    
    model_summary["pct_ci_positive"] = (
        100
        * model_summary["n_ci_positive"]
        / model_summary["n_datasets"]
    )
    
    model_summary["pct_ci_negative"] = (
        100
        * model_summary["n_ci_negative"]
        / model_summary["n_datasets"]
    )
    
    
    model_summary = model_summary.sort_values(
        "mean_loss_difference",
        ascending=False,
    ).reset_index(drop=True)
    
    
    model_summary.to_csv(
        BASE_DIR
        / f"qss_loss_difference_bootstrap_summary_{freq}_with_timestral.csv",
        index=False,
    )
    
    
    # ============================================================
    # BLOCK-LENGTH SENSITIVITY
    # ============================================================
    
    sensitivity = (
        results
        .groupby(
            [
                "model",
                "block_length",
            ],
            as_index=False,
        )
        .agg(
            mean_loss_difference=(
                "mean_loss_difference",
                "mean",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
            n_ci_positive=(
                "significantly_positive",
                "sum",
            ),
            n_ci_negative=(
                "significantly_negative",
                "sum",
            ),
        )
    )
    
    
    sensitivity["pct_ci_positive"] = (
        100
        * sensitivity["n_ci_positive"]
        / sensitivity["n_datasets"]
    )
    
    sensitivity["pct_ci_negative"] = (
        100
        * sensitivity["n_ci_negative"]
        / sensitivity["n_datasets"]
    )
    
    
    sensitivity.to_csv(
        BASE_DIR
        / f"qss_loss_difference_block_sensitivity_{freq}_with_timestral.csv",
        index=False,
    )
    
    
    # ============================================================
    # PRINT
    # ============================================================
    
    print()
    print("=" * 100)
    print(f"BLOCK BOOTSTRAP LOSS-DIFFERENCE SUMMARY {freq.upper()}")
    print("=" * 100)
    
    print(
        model_summary
        .round(6)
        .to_string(index=False)
    )
    
    print()
    print("=" * 100)
    print("BLOCK-LENGTH SENSITIVITY")
    print("=" * 100)
    
    print(
        sensitivity
        .round(4)
        .to_string(index=False)
    )

### QSS calendar bootstrap

All 72 daily datasets share 87 forecast-origin dates. This initial analysis uses those 87 dates.

- All 72 datasets are retained.
- Every dataset contributes observations from the same 87 dates.
- The same calendar-date indices are resampled for every dataset.
- Each dataset receives equal weight in the primary analysis.

Also computes an alternative estimate where each of the eight asset classes receives equal weight.

Blocks of length 20 mean 20 consecutive positions on our shared observation-date grid, not necessarily 20 consecutive calendar days.

The primary CSV contains each model's:

- Mean paired loss difference.
- Lower and upper 95% confidence bounds.
- Whether the interval excludes zero.
- Weighting scheme.
- Number of datasets and shared dates.

A positive loss difference indicates that the model performs better than the empirical reference.

In [ ]:
%run "notebooks/daily_synchronised_calendar_bootstrap_with_timestral.py"

# Ranking robustness updated

Calculate rankings separately for RMSE, MASE, CRPS, QSS, MSIS. Within every dataset, rank the models. 

Then calculate:

- the ordinary overall mean rank across all datasets
- leave-one-asset-class-out mean ranks, eight times
- equal-asset-class-weighted mean ranks
- the top model for each omission
- Spearman correlation between each leave-one-class-out ranking and the full ranking, which gives a compact numerical measure of ranking stability.

Lower is better for RMSE, MASE, CRPS, MSIS, higher is better for QSS.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr


for freq in ["daily", "weekly"]:
    # ============================================================
    # PATHS
    # ============================================================
    
    ROOT = Path(
        r"."
        rf"\BForTFin_results\standardised\output_{freq}"
    )
    
    FILES = {
        "RMSE": ROOT / "rmse_summary_all_models_with_timestral.csv",
        "qCRPS": ROOT / "all_models_crps_by_dataset_with_timestral.csv",
        "QSS": ROOT / "qss_summary_all_models_with_timestral.csv",
        "MASE": ROOT / f"point_models_context_mase_{freq}_with_timestral.csv",
        "MSIS": ROOT / f"msis_250_by_dataset_{freq}_with_timestral.csv",
    }
    
    OUT_DIR = ROOT / "robustness_rankings"
    OUT_DIR.mkdir(exist_ok=True)
    
    
    # ============================================================
    # HELPERS
    # ============================================================
    
    def clean_dataset_name(x):
        """
        Standardise dataset names across the metric summary files.
    
        Examples:
          AUD_USD Historical Data_no_preprocess_lookback-250_horizon-20_slide-1
            -> AUD_USD Historical Data
    
          AUD_USD Historical Data
            -> AUD_USD Historical Data
        """
        x = str(x).strip()
    
        suffixes = [
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            "_lookback-250_horizon-20_slide-1",
            "_no_preprocess",
        ]
    
        for suffix in suffixes:
            if x.endswith(suffix):
                x = x[:-len(suffix)]
    
        return x.strip()
    
    
    def clean_model_name(x):
        """
        Minimal standardisation only.
        Add aliases here ONLY if inspection shows that the same model
        is written differently across files.
        """
        x = str(x).strip()
    
        aliases = {
            # Examples -- uncomment/change only if actually needed:
            # "GARCH": "GARCH(1,1)",
            # "Chronos": "Chronos-T5",
            # "Moirai MoE": "Moirai-MoE",
        }
    
        return aliases.get(x, x)
    
    
    def standardise_metric_file(path, metric):
        df = pd.read_csv(path)
    
        # Column containing the actual metric value
        value_columns = {
            "RMSE": "rmse",
            "qCRPS": "mean_qcrps",
            "QSS": "qss_0.1",
            "MASE": "mase_250",
            "MSIS": "msis_250",
        }
    
        value_col = value_columns[metric]
    
        required = {"model", "asset_class", "dataset", value_col}
        missing = required - set(df.columns)
    
        if missing:
            raise ValueError(
                f"{metric}: missing columns {missing}\n"
                f"Available columns: {list(df.columns)}"
            )
    
        out = df[
            ["model", "asset_class", "dataset", value_col]
        ].copy()
    
        out["model"] = out["model"].map(clean_model_name)
        out["asset_class"] = out["asset_class"].astype(str).str.strip()
        out["dataset"] = out["dataset"].map(clean_dataset_name)
    
        out = out.rename(columns={value_col: "value"})
        out["metric"] = metric
    
        out["value"] = pd.to_numeric(out["value"], errors="coerce")
    
        return out
    
    
    # ============================================================
    # LOAD ALL METRICS
    # ============================================================
    
    metric_dfs = {
        metric: standardise_metric_file(path, metric)
        for metric, path in FILES.items()
    }
    
    all_metrics = pd.concat(
        metric_dfs.values(),
        ignore_index=True
    )
    
    print("Rows by metric:")
    print(all_metrics.groupby("metric").size())
    
    print("\nUnique datasets by metric:")
    print(all_metrics.groupby("metric")["dataset"].nunique())
    
    print("\nUnique models by metric:")
    print(all_metrics.groupby("metric")["model"].nunique())
    
    print("\nAsset classes:")
    print(
        all_metrics[["asset_class"]]
        .drop_duplicates()
        .sort_values("asset_class")
        .to_string(index=False)
    )
    
    
    # ============================================================
    # OPTIONAL DUPLICATE CHECK
    # ============================================================
    
    dupes = all_metrics[
        all_metrics.duplicated(
            subset=["metric", "asset_class", "dataset", "model"],
            keep=False
        )
    ].sort_values(
        ["metric", "asset_class", "dataset", "model"]
    )
    
    if len(dupes):
        print("\nWARNING: duplicate model-dataset rows found:")
        print(dupes.to_string(index=False))
    else:
        print("\nNo duplicate model-dataset rows found.")
    
    
    # ============================================================
    # RANKING FUNCTION
    # ============================================================
    
    def add_dataset_ranks(df, metric):
        """
        Rank models WITHIN each dataset.
    
        RMSE, MASE, qCRPS, MSIS: lower is better.
        QSS: higher is better.
    
        method='average' handles exact ties fairly.
        """
        out = df.copy()
    
        higher_is_better = metric == "QSS"
    
        out["rank"] = out.groupby(
            ["asset_class", "dataset"]
        )["value"].rank(
            ascending=not higher_is_better,
            method="average"
        )
    
        return out
    
    
    # ============================================================
    # COMPUTE ROBUSTNESS RESULTS FOR ONE METRIC
    # ============================================================
    
    def robustness_rankings(df, metric):
    
        df = df.dropna(subset=["value"]).copy()
        ranked = add_dataset_ranks(df, metric)
    
        # --------------------------------------------------------
        # 1. Ordinary overall ranking:
        #    each dataset gets equal weight
        # --------------------------------------------------------
    
        overall = (
            ranked.groupby("model", as_index=False)
            .agg(
                mean_rank=("rank", "mean"),
                n_datasets=("dataset", "nunique"),
            )
            .sort_values("mean_rank")
            .reset_index(drop=True)
        )
    
        overall["overall_position"] = np.arange(1, len(overall) + 1)
        overall["metric"] = metric
    
    
        # --------------------------------------------------------
        # 2. Mean rank within each asset class
        # --------------------------------------------------------
    
        by_asset = (
            ranked.groupby(
                ["asset_class", "model"],
                as_index=False
            )
            .agg(
                mean_rank=("rank", "mean"),
                n_datasets=("dataset", "nunique")
            )
        )
    
    
        # --------------------------------------------------------
        # 3. Equal-asset-class-weighted ranking
        #
        # First average datasets within asset class,
        # THEN average the 8 asset classes.
        # --------------------------------------------------------
    
        equal_asset = (
            by_asset.groupby("model", as_index=False)
            .agg(
                equal_asset_mean_rank=("mean_rank", "mean"),
                n_asset_classes=("asset_class", "nunique")
            )
            .sort_values("equal_asset_mean_rank")
            .reset_index(drop=True)
        )
    
        equal_asset["equal_asset_position"] = (
            np.arange(1, len(equal_asset) + 1)
        )
    
        equal_asset["metric"] = metric
    
    
        # --------------------------------------------------------
        # 4. Leave-one-asset-class-out ranking
        # --------------------------------------------------------
    
        loo_tables = []
        stability_rows = []
    
        asset_classes = sorted(ranked["asset_class"].unique())
    
        # Full ranking lookup for correlation
        full_lookup = overall.set_index("model")["mean_rank"]
    
        for excluded in asset_classes:
    
            sub = ranked[
                ranked["asset_class"] != excluded
            ].copy()
    
            loo = (
                sub.groupby("model", as_index=False)
                .agg(
                    mean_rank=("rank", "mean"),
                    n_datasets=("dataset", "nunique"),
                    n_asset_classes=("asset_class", "nunique"),
                )
                .sort_values("mean_rank")
                .reset_index(drop=True)
            )
    
            loo["position"] = np.arange(1, len(loo) + 1)
            loo["excluded_asset_class"] = excluded
            loo["metric"] = metric
    
            loo_tables.append(loo)
    
    
            # ----------------------------------------------------
            # Spearman correlation with full ranking
            # ----------------------------------------------------
    
            common_models = sorted(
                set(full_lookup.index)
                & set(loo["model"])
            )
    
            full_ranks = (
                full_lookup
                .loc[common_models]
                .rank(method="average")
            )
    
            loo_lookup = loo.set_index("model")["mean_rank"]
    
            loo_ranks = (
                loo_lookup
                .loc[common_models]
                .rank(method="average")
            )
    
            rho, p = spearmanr(full_ranks, loo_ranks)
    
            top_model = loo.iloc[0]["model"]
            top_mean_rank = loo.iloc[0]["mean_rank"]
    
            stability_rows.append({
                "metric": metric,
                "excluded_asset_class": excluded,
                "spearman_rho_vs_full": rho,
                "spearman_pvalue": p,
                "top_model": top_model,
                "top_mean_rank": top_mean_rank,
                "n_models": len(common_models),
            })
    
    
        loo_all = pd.concat(
            loo_tables,
            ignore_index=True
        )
    
        stability = pd.DataFrame(stability_rows)
    
    
        # --------------------------------------------------------
        # Wide leave-one-out table -- easy to read in Excel
        # --------------------------------------------------------
    
        loo_wide = (
            loo_all.pivot(
                index="model",
                columns="excluded_asset_class",
                values="mean_rank"
            )
        )
    
        loo_wide.insert(
            0,
            "Full",
            full_lookup
        )
    
        loo_wide = (
            loo_wide
            .sort_values("Full")
            .reset_index()
        )
    
    
        return {
            "ranked_dataset_rows": ranked,
            "overall": overall,
            "by_asset": by_asset,
            "equal_asset": equal_asset,
            "loo_long": loo_all,
            "loo_wide": loo_wide,
            "stability": stability,
        }
    
    
    # ============================================================
    # RUN ALL FOUR METRICS
    # ============================================================
    
    results = {}
    
    for metric, df in metric_dfs.items():
    
        print("\n" + "=" * 70)
        print(metric)
        print("=" * 70)
    
        res = robustness_rankings(df, metric)
        results[metric] = res
    
        print("\nOverall ranking:")
        print(
            res["overall"][
                ["overall_position", "model", "mean_rank", "n_datasets"]
            ].to_string(index=False)
        )
    
        print(f"\n{freq.capitalize()} equal-asset-class ranking:")
        print(
            res["equal_asset"][
                [
                    "equal_asset_position",
                    "model",
                    "equal_asset_mean_rank",
                    "n_asset_classes"
                ]
            ].to_string(index=False)
        )
    
        print(f"\n{freq.capitalize()} leave-one-asset-class-out stability:")
        print(
            res["stability"][
                [
                    "excluded_asset_class",
                    "spearman_rho_vs_full",
                    "top_model",
                    "top_mean_rank"
                ]
            ].to_string(index=False)
        )
    
    
    # ============================================================
    # SAVE EVERYTHING
    # ============================================================
    
    for metric, res in results.items():
    
        safe_metric = (
            metric.lower()
            .replace("qcrps", "qcrps")
        )
    
        res["ranked_dataset_rows"].to_csv(
            OUT_DIR / f"{safe_metric}_dataset_ranks_with_timestral.csv",
            index=False
        )
    
        res["overall"].to_csv(
            OUT_DIR / f"{safe_metric}_overall_ranking_with_timestral.csv",
            index=False
        )
    
        res["by_asset"].to_csv(
            OUT_DIR / f"{safe_metric}_rank_by_asset_class_with_timestral.csv",
            index=False
        )
    
        res["equal_asset"].to_csv(
            OUT_DIR / f"{safe_metric}_equal_asset_class_ranking_with_timestral.csv",
            index=False
        )
    
        res["loo_long"].to_csv(
            OUT_DIR / f"{safe_metric}_leave_one_asset_class_out_long_with_timestral.csv",
            index=False
        )
    
        res["loo_wide"].to_csv(
            OUT_DIR / f"{safe_metric}_leave_one_asset_class_out_wide_with_timestral.csv",
            index=False
        )
    
        res["stability"].to_csv(
            OUT_DIR / f"{safe_metric}_ranking_stability_with_timestral.csv",
            index=False
        )
    
    print(f"\nSaved {freq} robustness results to:\n{OUT_DIR}")